# **Amazon Wipes Price Analysis**

## **Problem Definition**
Optimized Wipes pricing on Amazon.com.
### **The Context:**

 - Why is this problem important to solve?
 1. There is an increasing demand for flushable wipes, adult wipes, hand wipes, feminine wipes, and baby wipes on Amazon and the projected sales is above $0.7 billion.
 2. The price is hard to set based on so many factors, such as brand equity, claims, ingredients, functions/benefits/attributes, packsize and competitors' promotion and pricing etc.  
 3. There are strong business/increasing revenues opportunities if the pricing for the innovation wipes are reasonable.
 4. Consumer trust and brand position. If the price is set up right for our wipes in different tiers and categories, we can gain postion our wipes to the right usage groups and gain more consumer trust, higher rating, repeat purchases and healthy revenue growth in the long term.

### **The objective:**

 - What is the intended goal?
 The intended goal is to help the business to set up the optimized pricing for different wipe categories on Amazon.com that drives volume and value growth.

### **The key questions:**

- What are the key questions that need to be answered?
1. Pricing strategy. What kind of pricing range should the business set based on the functions/benefits/attributes/claims of the wipes.
2. Targeted marketing: Which group of potential customers each wipes category should target based on the usage occation/claims/benefits/packaging of the wipes.
3. How can the wipes business maximize the profits? What are the price elasticity for baby and non-baby positioned wipes?
4. What are the key factors influencing the price of the wipes?
5. How can we predict the price accurately for our innovation wipes?
6. Which model is most suitable for wipes price predicting?
7. How are our packaging, online contents, and pricing comparing with major competitor brands? What are the opportunities to improve?
8. How can we handle missing or inconsistent data we get from Profitero or throug Amazon web scraping? i.e Missing ASINs in Profitero and sales $ and inconsistent ASINS month over month, outlier Average price based on the division of Total Sales by total units of the ASIN etc.


### **The problem formulation**:

- What is it that we are trying to solve using data science?
1. Data collection and cleasing.
2. Data transformation and data visualization , such as using log instead of the original feature number if there are many outliers, using histogram or boxplot to present the data.
3. Exploratory Data Analysis: exploring the data using both univariate and bivariate analysis to identify the trends, missing values, and other key insights.
4. Model building: Train machine learning models to predict wipes prices. If the wipes total projected sales $ and units are not available, use ratings and rating counts to predict.
5. Model evaluation: Evaluate models using metrics like R-sqaured, RMSE, and MAE.
6. Business strategy: Use the model to create differential pricing strategies for different targeted user groups, usage occasions and packsize and improve profitability.

### **Total Amazon‑scraped unique ASINs with rating(s): 17,004, of which 2,979 match Profitero API ASINs. The cleaned, selected Amazon subset includes 5,160 out of 6,102 body‑wipe ASINs (~85%) with at least one rating. ** ###

In [ ]:
# 1. Build a correlation between total market $ vs ratings * total rating counts. i.e if the total wipes market is $0.7B, and the ratings * total rating counts % of Pampers Wipes to the sumproduct of ratings * total rating counts is 5%, then we would estimate Pamper wipes sales to be 5% * $0.7 B = $0.035B.
# Use Hello Bello Total sales units and ratings * total rating counts as an example.

### **Data Dictionary**

**The data used in this price analysis are drawn from a cleaned Amazon web‑scraped dataset, representing more than 84% of all collected listings across multiple body‑wipe categories.(Excluding bundles)**

**User** : Consumer segment. If the ProductTitle refers exclusively to babies or young children (for example, includes terms such as “baby,” “child,” “infant,” “newborn” or close synonyms), classify User as “Baby.” If it refers exclusively to women (for example, includes “feminine,” “women,” “woman,” “lady” or synonyms), classify as “Feminine.” If it refers exclusively to men (for example, includes “men,” “man” or synonyms), classify as “Men.” If the ProductTitle is not user‑specific or mentions multiple user groups, classify the User as “Non‑Baby.”

**XL** : Extra‑large format indicator. Mark XL as “Yes” when the product is positioned as extra‑large, including but not limited to descriptors such as “Extra Large,” “X‑Large,” “XL,” “Jumbo,” “Bath Wipes,” “Max,” “Oversized,” “All Over,” or “Head‑to‑Toe” in the product labeling or title.

**Flushable** : Flushability indicator. Mark Flushable as “Yes” only when the ProductTitle explicitly includes the term “flushable” or a clear equivalent claim.

**CLEANED BRAND** : Standardized parent brand. Enter the verified parent brand name based on the ProductTitle or the brand listed on Amazon.com. Use “UNCODED” when the brand is not present in the ProductTitle and has not yet been verified on Amazon.com.

**ProductTitle** : The full product title associated with the ASIN on Amazon.com.

**ASIN** : The unique Amazon Standard Identification Number corresponding to the product.

**Count** : Number of wipes per individual pack. For individually wrapped wipes, Count equals the number of individual wipe packages.

**Pack** : Number of packs included under a single ASIN (for example, a 3‑pack or 10‑pack).

**RatingCount** : Total number of customer ratings associated with the ASIN.

**CurrentPrice** : The current selling price for the ASIN at the time of web scraping. When this price is unavailable, substitute a price from a comparable pack size of the same product or, if necessary, from another major retailer (for example, Walmart.com, eBay.com, or the brand’s own website). Document any such substitutions in your notes.

**Calculated \$/Count  or \$/Bundle** : Unit price per wipe (or per bundle, if more appropriate). Calculate as CurrentPrice ÷ TotalCount, where TotalCount = Count × Pack.

### **Loading libraries**

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

# To ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Remove the limit from the number of displayed columns and rows. It helps to see the entire dataframe while printing it
pd.set_option("display.max_columns", None)

In [ ]:
from google import colab
colab.drive.mount('/content/drive')

### **Let us load the data**

In [ ]:
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

### **Understand the data by observing a few rows**

In [ ]:
# View first 5 rows
data.head(5)

In [ ]:
# View last 5 rows Hint: Use tail() method
data.tail(5)

In [ ]:
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

# Display the head of the DataFrame with the new column in percentage format
display_df = data[['Rating', 'RatingCount', 'Rating_x_RatingCount', 'Rating_Count Weight']].head().copy()
display_df['Rating_Count Weight'] = display_df['Rating_Count Weight'].apply(lambda x: f"{x:.6%}")
display(display_df)

**Observations and Insights: There are 14 columns and 7252 rows in the used_cars dataset, and some columns have null values.**

### **Let us check the data types and and missing values of each column**

In [ ]:
# Check the datatypes of each column. Hint: Use info() method
data.info()

In [ ]:
# Check total number of missing values of each column. Hint: Use isnull() method
data.isnull().sum()

In [ ]:
data.columns

## **Exploratory Data Analysis**




### **Let us now explore the summary statistics of numerical variables**

In [ ]:
# Explore basic summary statistics of numeric variables. Hint: Use describe() method.
data.describe()

**Observations and Insights:**
1. There are about 5,133 non-null values in the columns.
2. The Avg Count/pack is 49 counts of wipes per pack.
3. The Avg Total Count per unit of product if 259 counts of wipes.
4. The Avg rating of Amazon Wipe Products is 4.51.
5. The Avg rating counts of Amazon Wipe Products is about 3,784.
6. The Avg Price/ Unit is $27.42 (excluding bundles).

7. The calculated \$/count is \$0.34/count of wipe.


### **Check for duplicate 'CLEANED BRAND' entries considering case and whitespace**

In [ ]:
# Create a cleaned version of the 'CLEANED BRAND' column for comparison
data['CLEANED BRAND_cleaned'] = data['CLEANED BRAND'].str.strip().str.lower()

# Find unique cleaned brand names that correspond to multiple original brand names
duplicate_cleaned_brands = data.groupby('CLEANED BRAND_cleaned')['CLEANED BRAND'].nunique()
duplicate_cleaned_brands = duplicate_cleaned_brands[duplicate_cleaned_brands > 1].index

if not duplicate_cleaned_brands.empty:
    print("Original 'CLEANED BRAND' entries that map to the same cleaned brand (potential duplicates):")
    for cleaned_brand in duplicate_cleaned_brands:
        original_brands = data[data['CLEANED BRAND_cleaned'] == cleaned_brand]['CLEANED BRAND'].unique()
        print(f"  Cleaned Brand: '{cleaned_brand}' -> Original Brands: {list(original_brands)}")
else:
    print("No duplicate 'CLEANED BRAND' entries found based on case or whitespace differences.")

# Drop the temporary cleaned column
data = data.drop(columns=['CLEANED BRAND_cleaned'])

### **Standardize 'CLEANED BRAND' to address inconsistencies**
To ensure consistency and accurate analysis, I will standardize the `CLEANED BRAND` column by removing leading/trailing whitespace and converting all text to uppercase. This will group together entries that are currently considered distinct due to minor formatting differences.


In [ ]:


# Standardize 'CLEANED BRAND' column by stripping whitespace and converting to uppercase
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Display unique values and their counts to verify cleaning
print("Unique 'CLEANED BRAND' entries after standardization:")
display(data['CLEANED BRAND'].value_counts().sort_index())

### **Let us also explore the summary statistics of all categorical variables and the number of unique observations in each category**

In [ ]:
# Explore basic summary statistics of categorical variables. Hint: Use the argument include = ['object']
data.describe(include = ['object'])

In [ ]:
hello_bello_data = data[data['CLEANED BRAND'] == 'HELLO BELLO']
display(hello_bello_data.head())

In [ ]:
# Get unique 'TotalCount' values from hello_bello_data
hello_bello_total_counts = hello_bello_data['TotalCount'].unique()

# Filter the main 'data' DataFrame to find products with matching TotalCount
products_with_matching_total_count = data[data['TotalCount'].isin(hello_bello_total_counts)]

# Display the head of the filtered DataFrame
display(products_with_matching_total_count.head())

In [ ]:
from google.colab import files

# Save the DataFrame to a CSV file
csv_filename = 'products_with_matching_total_count.csv'
products_with_matching_total_count.to_csv(csv_filename, index=False)

# Provide a download link for the CSV file
files.download(csv_filename)
print(f"'{csv_filename}' has been downloaded.")

In [ ]:
from google.colab import files

# Split the DataFrame into 'Baby' users and 'Other' users
baby_users_df = products_with_matching_total_count[products_with_matching_total_count['User'] == 'Baby']
other_users_df = products_with_matching_total_count[products_with_matching_total_count['User'] != 'Baby']

# Define filenames
baby_users_filename = 'products_baby_users.csv'
other_users_filename = 'products_other_users.csv'

# Save the DataFrames to CSV files
baby_users_df.to_csv(baby_users_filename, index=False)
other_users_df.to_csv(other_users_filename, index=False)

print(f"'{baby_users_filename}' and '{other_users_filename}' have been created.")

# Optionally, provide download links for these files
# files.download(baby_users_filename)
# files.download(other_users_filename)


In [ ]:
from google.colab import files

# Provide a download link for 'products_baby_users.csv'
files.download('products_baby_users.csv')
print("'products_baby_users.csv' is ready for download.")

# Provide a download link for 'products_other_users.csv'
files.download('products_other_users.csv')
print("'products_other_users.csv' is ready for download.")

In [ ]:
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
# import numpy as np

# # Ensure 'Gissel's L52W $' has no NaN values for plotting
# baby_users_df['Gissel\'s L52W $'] = baby_users_df['Gissel\'s L52W $'].fillna(0)

# if not baby_users_df.empty:
#     # Extract the data for the 3D plot
#     x = baby_users_df['TotalCount']
#     y = baby_users_df['CurrentPrice']
#     z = baby_users_df['Gissel\'s L52W $']
#     brands = baby_users_df['CLEANED BRAND']

#     # Create the 3D plot
#     fig = plt.figure(figsize=(12, 10))
#     ax = fig.add_subplot(111, projection='3d')

#     # Plot the scatter points.
#     scatter = ax.scatter(x, y, z, c=z, cmap='viridis', s=50)

#     # Add brand names as labels to each dot
#     for i in range(len(baby_users_df)):
#         ax.text(x.iloc[i], y.iloc[i], z.iloc[i], brands.iloc[i], color='black', fontsize=8, alpha=0.7)

#     # Set labels and title
#     ax.set_xlabel('Total Count')
#     ax.set_ylabel('Current Price ($)')
#     ax.set_zlabel('Gissel\'s L52W $')
#     ax.set_title('3D Scatter Plot for All Brands (Baby Users) - TotalCount, CurrentPrice, Gissel\'s L52W $')

#     # Add a color bar
#     cbar = fig.colorbar(scatter, shrink=0.5, aspect=5)
#     cbar.set_label('Gissel\'s L52W $')

#     plt.show()
# else:
#     print("No data in products_baby_users to plot.")

### **Individual 3D Scatter Plots for Each Unique `TotalCount` (matching 'Hello Bello' TotalCounts)**

These plots visualize the relationship between `TotalCount`, `CurrentPrice`, and `Gissel's L52W $` for all products(including All Users positions) that share the same `TotalCount` as 'Hello Bello' products. Each data point is labeled with its corresponding 'CLEANED BRAND'.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
products_with_matching_total_count['Gissel\'s L52W $'] = products_with_matching_total_count['Gissel\'s L52W $'].fillna(0)

if not hello_bello_total_counts.size == 0:
    for total_count_value in hello_bello_total_counts:
        # Filter data for the current TotalCount
        subset_data = products_with_matching_total_count[
            products_with_matching_total_count['TotalCount'] == total_count_value
        ]

        if not subset_data.empty:
            # Group by CLEANED BRAND and calculate mean for plotting
            plot_data = subset_data.groupby('CLEANED BRAND').agg({
                'CurrentPrice': 'mean',
                'Gissel\'s L52W $': 'mean'
            }).reset_index()

            if not plot_data.empty:
                fig, ax = plt.subplots(figsize=(14, 8)) # Adjusted figure size for bar chart
                sns.barplot(x='CLEANED BRAND', y='CurrentPrice', data=plot_data, ax=ax, palette='viridis')

                # Add Gissel's L52W $ on top of each bar
                for index, row in plot_data.iterrows():
                    ax.text(index, row['CurrentPrice'], f"${row['Gissel\'s L52W $']:.2f}",
                            color='black', ha="center", va='bottom', fontsize=9)

                # Calculate mean and median CurrentPrice for the current plot_data
                mean_current_price = plot_data['CurrentPrice'].mean()
                median_current_price = plot_data['CurrentPrice'].median()

                # Add green line for average Current Price
                ax.axhline(mean_current_price, color='green', linestyle='--', label=f'Mean Current Price: ${mean_current_price:.2f}')
                # Add black line for median Current Price
                ax.axhline(median_current_price, color='black', linestyle='-', label=f'Median Current Price: ${median_current_price:.2f}')

                ax.set_xlabel('CLEANED BRAND')
                ax.set_ylabel('Average Current Price ($)')
                ax.set_title(f'Current Price and Gissel\'s L52W $ by Brand for TotalCount = {total_count_value}')
                plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
                plt.legend() # Show legend for mean and median lines
                plt.tight_layout()
                plt.show()
            else:
                print(f"No aggregated data for TotalCount = {total_count_value}.")
        else:
            print(f"No data for TotalCount = {total_count_value} in products_with_matching_total_count.")
else:
    print("No unique 'TotalCount' values found in hello_bello_data to plot.")

** Hello Bello TotalCount by Baby & Non-baby Competitor Products**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 'data' is loaded and CLEANED BRAND is standardized
# This assumes 'data' has been loaded in a preceding cell. If not, uncomment the line below:
# data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Define baby_wipe_data and other_wipe_data
baby_wipe_data = data[data['User'] == 'Baby']
other_wipe_data = data[data['User'] != 'Baby']


# Assuming 'data', 'baby_wipe_data', and 'other_wipe_data' are already defined from previous cells.
# They represent 'All Users', 'Baby Users', and 'Non-Baby Users' respectively.

# Define the segments and their names
user_segments = {
    "All Users": data,
    "Baby Users": baby_wipe_data,
    "Non-Baby Users": other_wipe_data
}

print("Performing requested aggregations and visualizations:")

# Ensure hello_bello_total_counts is available. If not, re-create it.
if 'hello_bello_data' not in locals():
    # Assuming 'data' is loaded and CLEANED BRAND is standardized
    data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
    hello_bello_data = data[data['CLEANED BRAND'] == 'HELLO BELLO']
hello_bello_total_counts = hello_bello_data['TotalCount'].unique()

for segment_name, segment_df in user_segments.items():
    print(f"\n--- Processing for {segment_name} ---")

    if segment_df.empty:
        print(f"No data available for {segment_name}.")
        continue

    # 1. Aggregate by Brand: Calculate average CurrentPrice and Average Price/Count
    brand_agg_df = segment_df.groupby('CLEANED BRAND').agg(
        Average_CurrentPrice=('CurrentPrice', 'mean'),
        Average_Price_Per_Count=('Calculated Price/Count or Price/Bundle', 'mean')
    ).reset_index()

    print(f"\nAverage CurrentPrice and Price/Count by Brand for {segment_name} (first 5 rows):")
    display(brand_agg_df.head()) # Displaying head to keep output concise

    # 2. Aggregate by Product: Calculate average CurrentPrice and Average Price/Count
    # Since ProductTitle uniquely identifies a product, this effectively lists each product with its metrics
    product_agg_df = segment_df.groupby('ProductTitle').agg(
        CLEANED_BRAND=('CLEANED BRAND', 'first'), # Include brand for context
        Average_CurrentPrice=('CurrentPrice', 'mean'),
        Average_Price_Per_Count=('Calculated Price/Count or Price/Bundle', 'mean')
    ).reset_index()

    # Reorder columns for better readability
    product_agg_df = product_agg_df[['CLEANED_BRAND', 'ProductTitle', 'Average_CurrentPrice', 'Average_Price_Per_Count']]

    print(f"\nAverage CurrentPrice and Price/Count by Product for {segment_name} (first 5 rows):")
    display(product_agg_df.head()) # Displaying head to keep output concise

    # --- Add plotting logic for TotalCount matching Hello Bello ---
    print(f"\nGenerating visualizations for {segment_name} based on TotalCount values matching Hello Bello...")

    # Filter segment_df for TotalCount values present in hello_bello_total_counts
    filtered_for_hb_total_counts = segment_df[segment_df['TotalCount'].isin(hello_bello_total_counts)].copy()

    # Fill NaN 'Gissel\'s L52W $' with 0 for plotting
    filtered_for_hb_total_counts["Gissel's L52W $"] = filtered_for_hb_total_counts["Gissel's L52W $"].fillna(0)

    if not filtered_for_hb_total_counts.empty and hello_bello_total_counts.size > 0:
        for total_count_value in sorted(hello_bello_total_counts):
            subset_data_for_plot = filtered_for_hb_total_counts[
                filtered_for_hb_total_counts['TotalCount'] == total_count_value
            ].copy()

            if not subset_data_for_plot.empty:
                # Group by CLEANED BRAND and calculate mean for plotting
                plot_data = subset_data_for_plot.groupby('CLEANED BRAND').agg({
                    'CurrentPrice': 'mean',
                    "Gissel's L52W $": 'mean'
                }).reset_index()

                if not plot_data.empty:
                    fig, ax = plt.subplots(figsize=(14, 8))
                    sns.barplot(x='CLEANED BRAND', y='CurrentPrice', data=plot_data, ax=ax, palette='viridis')

                    # Add Gissel's L52W $ on top of each bar
                    for index, row in plot_data.iterrows():
                        ax.text(index, row['CurrentPrice'], f"${row['Gissel\'s L52W $']:.2f}",
                                color='black', ha="center", va='bottom', fontsize=9)

                    # Calculate mean and median CurrentPrice for the current plot_data
                    mean_current_price = plot_data['CurrentPrice'].mean()
                    median_current_price = plot_data['CurrentPrice'].median()

                    # Add green line for average Current Price
                    ax.axhline(mean_current_price, color='green', linestyle='--', label=f'Mean Current Price: ${mean_current_price:.2f}')
                    # Add black line for median Current Price
                    ax.axhline(median_current_price, color='black', linestyle='-', label=f'Median Current Price: ${median_current_price:.2f}')

                    ax.set_xlabel('CLEANED BRAND')
                    ax.set_ylabel('Average Current Price ($)')
                    ax.set_title(f'Average Current Price and Gissel\'s L52W $ by Brand for TotalCount = {total_count_value} ({segment_name})')
                    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
                    plt.legend() # Show legend for mean and median lines
                    plt.tight_layout()
                    plt.show()
                else:
                    print(f"No aggregated brand data for TotalCount = {total_count_value} ({segment_name}).")
            else:
                print(f"No data for TotalCount = {total_count_value} in {segment_name}.")
    else:
        print(f"No matching TotalCount values with Hello Bello found in {segment_name} to plot.")

print("\nFinished generating brand and product level average price metrics and visualizations.")

### **Bar Diagrams for `products_baby_users` by Rating and Current Price (matching 'Hello Bello' TotalCounts), showing `Gissel's L52W $`**

These plots visualize the relationship between `Rating` and `CurrentPrice` for products in `products_baby_users` that share the same `TotalCount` values as 'Hello Bello' products. Each bar displays the average `CurrentPrice` for a given `Rating`, with the average `Gissel's L52W $` annotated on top of the bar.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting in baby_users_df
baby_users_df['Gissel\'s L52W $'] = baby_users_df['Gissel\'s L52W $'].fillna(0)

# Get unique 'TotalCount' values from baby_users_df
unique_baby_users_total_counts = baby_users_df['TotalCount'].unique()

if not unique_baby_users_total_counts.size == 0:
    for total_count_value in unique_baby_users_total_counts:
        # Filter data for the current TotalCount
        subset_data = baby_users_df[
            baby_users_df['TotalCount'] == total_count_value
        ]

        if not subset_data.empty:
            # Group by Rating and calculate mean for plotting
            plot_data = subset_data.groupby('Rating').agg({
                'CurrentPrice': 'mean',
                'Gissel\'s L52W $': 'mean'
            }).reset_index()

            if not plot_data.empty:
                fig, ax = plt.subplots(figsize=(12, 7)) # Adjusted figure size for bar chart
                sns.barplot(x='Rating', y='CurrentPrice', data=plot_data, ax=ax, palette='viridis')

                # Add Gissel's L52W $ on top of each bar
                for index, row in plot_data.iterrows():
                    # Display both CurrentPrice (bar height) and Gissel's L52W $ on top of the bar
                    ax.text(row['Rating'], row['CurrentPrice'], f"Avg Price: ${row['CurrentPrice']:.2f}\nAvg Sales: ${row['Gissel\'s L52W $']:.2f}",
                            color='black', ha="center", va='bottom', fontsize=9)

                ax.set_xlabel('Rating')
                ax.set_ylabel('Average Current Price ($)')
                ax.set_title(f'Average Current Price and Gissel\'s L52W $ by Rating for TotalCount = {total_count_value} (Baby Users)')
                plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability if many ratings
                plt.tight_layout()
                plt.show()
            else:
                print(f"No aggregated data for TotalCount = {total_count_value} (Baby Users).")
        else:
            print(f"No data for TotalCount = {total_count_value} in products_baby_users.")
else:
    print("No unique 'TotalCount' values found in products_baby_users to plot.")

### **Bar Diagrams for `products_non_baby_users` by Rating and Current Price, showing `Gissel's L52W $`**

These plots visualize the relationship between `Rating` and `CurrentPrice` for products in `products_non_baby_users` that share the same `TotalCount` values as 'Hello Bello' products. Each bar displays the average `CurrentPrice` for a given `Rating`, with the average `Gissel's L52W $` annotated on top of the bar.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Load products_other_users.csv (assuming it was saved previously)
other_users_df = pd.read_csv('products_other_users.csv')

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
other_users_df['Gissel\'s L52W $'] = other_users_df['Gissel\'s L52W $'].fillna(0)

# Get unique 'TotalCount' values from other_users_df
unique_other_users_total_counts = other_users_df['TotalCount'].unique()

if not unique_other_users_total_counts.size == 0:
    for total_count_value in unique_other_users_total_counts:
        # Filter data for the current TotalCount
        subset_data = other_users_df[
            other_users_df['TotalCount'] == total_count_value
        ]

        if not subset_data.empty:
            # Group by Rating and calculate mean for plotting
            plot_data = subset_data.groupby('Rating').agg({
                'CurrentPrice': 'mean',
                'Gissel\'s L52W $': 'mean'
            }).reset_index()

            if not plot_data.empty:
                fig, ax = plt.subplots(figsize=(12, 7)) # Adjusted figure size for bar chart
                sns.barplot(x='Rating', y='CurrentPrice', data=plot_data, ax=ax, palette='viridis')

                # Add Gissel's L52W $ on top of each bar
                for index, row in plot_data.iterrows():
                    # Calculate a small vertical offset to place the text slightly above the bar
                    # Use a percentage of the current Y-axis limit for robustness
                    y_offset = ax.get_ylim()[1] * 0.005  # Reduced to 0.5% of the max y-axis value as offset
                    ax.text(row['Rating'], row['CurrentPrice'] + y_offset, f"${row['Gissel\'s L52W $']:.2f}",
                            color='black', ha="center", va='bottom', fontsize=9)

                ax.set_xlabel('Rating')
                ax.set_ylabel('Average Current Price ($)')
                ax.set_title(f"Average Current Price and Gissel's L52W $ by Rating for TotalCount = {total_count_value} (Non-Baby Users)")
                plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability if many ratings
                plt.tight_layout()
                plt.show()
            else:
                print(f"No aggregated data for TotalCount = {total_count_value} (Non-Baby Users).")
        else:
            print(f"No data for TotalCount = {total_count_value} in products_other_users.")
else:
    print("No unique 'TotalCount' values found in products_other_users to plot.")

** Insights: **
From the bar diagrams for both baby and non-baby positioned wipes, we can see that there is not a strong positive relationship between Rating and Average product unit price. The higher the product rating, doesn't necessarily contributes to higher unit price.

**Number of unique observations in each category**

In [ ]:
cat_cols = data.select_dtypes(include = ['object']).columns

for column in cat_cols:

    print("For column:", column)

    print("Number of unique values:", data[column].nunique())

    print('-'*50)

## **Univariate Analysis**
Univariate analysis is used to explore each variable in a data set, separately. It looks at the range of values, as well as the central tendency of the values. It can be done for both numerical and categorical variables.

## **1. Univariate Analysis - Numerical Data**
Histograms and box plots help to visualize and describe numerical data. We use box plot and histogram to analyse the numerical columns.

In [ ]:
# Function to plot a boxplot and a histogram along the same scale

def histogram_boxplot(data, feature, figsize = (12, 7), kde = False, bins = None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows = 2,      # Number of rows of the subplot grid = 2
        sharex = True,  # x-axis will be shared among all subplots
        gridspec_kw = {"height_ratios": (0.25, 0.75)},
        figsize = figsize,
    )                   # Creating the 2 subplots
    sns.boxplot(
        data = data, x = feature, ax = ax_box2, showmeans = True, color = "violet"
    )                   # Boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2, bins = bins, palette = "winter"
    ) if bins else sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2
    )                   # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color = "green", linestyle = "--"
    )                   # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color = "black", linestyle = "-"
    )                   # Add median to the histogram

**Let us plot histogram and box-plot for the feature 'Count' to understand the distribution and outliers, if any.**

In [ ]:
# Plot histogram and box-plot for 'Kilometers_Driven'
histogram_boxplot(data, 'Count')

**Think About It**: Count is highly right-skewed. Can we use Log transformation of the feature to reduce/remove the skewness? Why can't we keep skewed data?

In [ ]:
# Log transformation of the feature 'Count'
sns.distplot(np.log(data["Count"]), axlabel = "Log(Count)");

**Observations and Insights: After applying the logarithmic scale, the distribution for Count doesn't follow a normal distribution with some outliers appear on the tail.**

In [ ]:
# Create three dataframes:
# 1. 'data' (all users - already loaded)
# 2. 'baby_wipe_data' (only 'Baby' users)
# 3. 'other_wipe_data' (all users except 'Baby')

baby_wipe_data = data[data['User'] == 'Baby']
other_wipe_data = data[data['User'] != 'Baby']




# Plot histogram and box-plot for 'TotalCount' for all three data parts
print("Shape of overall data:", data.shape)
print("\nDistribution of TotalCount for ALL Users:")
histogram_boxplot(data, 'TotalCount')
print("--------------------------------------------------------------------------------------------------------")

print("Shape of baby_wipe_data:", baby_wipe_data.shape)
print("\nDistribution of TotalCount for BABY Users:")
histogram_boxplot(baby_wipe_data, 'TotalCount')
print("--------------------------------------------------------------------------------------------------------")

print("Shape of other_wipe_data:", other_wipe_data.shape)
print("\nDistribution of TotalCount for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'TotalCount')
print("--------------------------------------------------------------------------------------------------------")

In [ ]:
# Log transformation of the feature 'TotalCount'

print("\nLog Distribution of TotalCount for ALL Users:")
sns.histplot(np.log(data["TotalCount"]), kde=True);
plt.xlabel("Log(TotalCount)")
plt.show()

print("\nLog Distribution of TotalCount for BABY Users:")
sns.histplot(np.log(baby_wipe_data["TotalCount"]), kde=True);
plt.xlabel("Log(TotalCount)")
plt.show()

print("\nLog Distribution of TotalCount for NON-BABY Users:")
sns.histplot(np.log(other_wipe_data["TotalCount"]), kde=True);
plt.xlabel("Log(TotalCount)")
plt.show()

**Observations and Insights: After applying the logarithmic scale, the distribution for TotalCount driven follows a semi-normal distribution for all Users and non-baby users.**

In [ ]:
# Log transformation of the feature 'TotalCount'
data['TotalCount_log'] = np.log(data['TotalCount'])

In [ ]:
# We can Add a transformed TotalCount feature in data
data.head(5)

In [ ]:
# Ensure 'TotalCount_log' column exists in baby_wipe_data and other_wipe_data
baby_wipe_data['TotalCount_log'] = np.log(baby_wipe_data['TotalCount'])
other_wipe_data['TotalCount_log'] = np.log(other_wipe_data['TotalCount'])

print("\nDistribution of Log(TotalCount) for ALL Users:")
histogram_boxplot(data, 'TotalCount_log')
plt.show()

print("\nDistribution of Log(TotalCount) for BABY Users:")
histogram_boxplot(baby_wipe_data, 'TotalCount_log')
plt.show()

print("\nDistribution of Log(TotalCount) for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'TotalCount_log')
plt.show()

For All users and non-baby wipes, there are more TotalCount outliers.
1. **For all users**:
(1) The distribution **right-skwed**, have many “moderate” users and a trailing tail of heavy users. The **mean**(green dotted line) **is higher than median**(solid dark line) suggests** a subset of very high-count wipe products**.  
2. **For baby users**:
(1) The distribution is **left-skewed** (negatively skewed); the **mean is less than the median**: the tail extends more toward lower values, and there are some relatively low‑usage BABY users pulling the mean down.The box spans the middle 50% of BABY users; this is the core usage band for this segment.
(2) The median log value is around the middle of the box, suggesting the **distribution across that central range is fairly balanced**.
(3) Most BABY users are around the median or a bit higher on log(TotalCount)
(4) BABY users are shifted to higher log values overall (their box is more right‑shifted on the x‑axis), implying **higher typical TotalCount** than the full user base.
(5) There are very few users at the very low log counts (left tail), suggesting **that if someone is in the BABY segment, they tend not to be light users**.
3. **For non-baby users** :
(1) The **mean and median are nearly identical**, the NON‑BABY distribution is close to symmetric around the central usage level, with lighter left-skew than what you saw in the BABY segment.
(2) The box is centered at a lower log value than in your BABY plots, meaning **typical NON‑BABY usage is lower than BABY usage** on the log scale.

***BABY vs NON‑BABY interpretation***
Putting all three together (ALL, BABY, NON‑BABY):
1.   **BABY users: higher central log(TotalCount) and fewer very low‑usage users → more engaged segment on average.**
2.  **NON‑BABY users: lower central log(TotalCount) and a more balanced shape, with many moderate users and some low‑usage users pulling the left tail.**

***For targeting:***
1.   **BABY segment is where you’d expect heavier repeat behavior (e.g., higher purchase or usage frequency).**
2.   **NON‑BABY users are more “average” and may need different messaging or incentives to move up the usage curve.**





In [ ]:
# Plot histogram and box-plot for 'Rating' for All Users
print("\nDistribution of Rating for ALL Users:")
histogram_boxplot(data, 'Rating')

# Plot histogram and box-plot for 'Rating' for BABY Users
print("\nDistribution of Rating for BABY Users:")
histogram_boxplot(baby_wipe_data, 'Rating')

# Plot histogram and box-plot for 'Rating' for NON-BABY Users
print("\nDistribution of Rating for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'Rating')

**For all users, baby-positioned, or non-baby positioned, the Average rating for all Amazon wipes is around 4.5; the median is around 4.6**

In [ ]:
print("\nDistribution of RatingCount for ALL Users:")
histogram_boxplot(data, 'RatingCount')

print("\nDistribution of RatingCount for BABY Users:")
histogram_boxplot(baby_wipe_data, 'RatingCount')

print("\nDistribution of RatingCount for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'RatingCount')


# print("\nDistribution of CurrentPrice for ALL Users:")
# histogram_boxplot(data, 'CurrentPrice')

# print("\nDistribution of CurrentPrice for BABY Users:")
# histogram_boxplot(baby_wipe_data, 'CurrentPrice')

# print("\nDistribution of CurrentPrice for NON-BABY Users:")
# histogram_boxplot(other_wipe_data, 'CurrentPrice')


# print("\nDistribution of Rating_x_RatingCount for ALL Users:")
# histogram_boxplot(data, 'Rating_x_RatingCount')

# print("\nDistribution of Rating_x_RatingCount for BABY Users:")
# histogram_boxplot(baby_wipe_data, 'Rating_x_RatingCount')

# print("\nDistribution of Rating_x_RatingCount for NON-BABY Users:")
# histogram_boxplot(other_wipe_data, 'Rating_x_RatingCount')


# print("\nDistribution of Rating_Count Weight for ALL Users:")
# histogram_boxplot(data, 'Rating_Count Weight')

# print("\nDistribution of Rating_Count Weight for BABY Users:")
# histogram_boxplot(baby_wipe_data, 'Rating_Count Weight')

# print("\nDistribution of Rating_Count Weight for NON-BABY Users:")
# histogram_boxplot(other_wipe_data, 'Rating_Count Weight')

In [ ]:
print("\nDistribution of Rating_x_RatingCount for ALL Users:")
histogram_boxplot(data, 'Rating_x_RatingCount')

print("\nDistribution of Rating_x_RatingCount for BABY Users:")
histogram_boxplot(baby_wipe_data, 'Rating_x_RatingCount')

print("\nDistribution of Rating_x_RatingCount for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'Rating_x_RatingCount')




In [ ]:
data['Rating_x_RatingCount_log'] = np.log1p(data['Rating_x_RatingCount'])
baby_wipe_data['Rating_x_RatingCount_log'] = np.log1p(baby_wipe_data['Rating_x_RatingCount'])
other_wipe_data['Rating_x_RatingCount_log'] = np.log1p(other_wipe_data['Rating_x_RatingCount'])

print("\nDistribution of Log(Rating_x_RatingCount) for ALL Users:")
histogram_boxplot(data, 'Rating_x_RatingCount_log')

print("\nDistribution of Log(Rating_x_RatingCount) for BABY Users:")
histogram_boxplot(baby_wipe_data, 'Rating_x_RatingCount_log')

print("\nDistribution of Log(Rating_x_RatingCount) for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'Rating_x_RatingCount_log')

### **Observations and Insights from Log-Transformed `Rating_x_RatingCount` Distributions:**

The `log1p` transformation has successfully normalized the `Rating_x_RatingCount` feature across all user segments (All Users, Baby Users, Non-Baby Users). Key observations are:

1.  **Normalization of Distributions**: For all three user segments, the distributions of `Rating_x_RatingCount_log` now appear more symmetrical and approximately bell-shaped, indicating a significant reduction in the original right-skewness. This transformation makes the data more suitable for various statistical modeling techniques.

2.  **Mean and Median Alignment**: After transformation, the mean and median values for `Rating_x_RatingCount_log` are much closer to each other in all segments, which confirms the reduced skewness and a more centered distribution.

3.  **Comparison Across Segments**:
    *   **All Users**: The overall distribution shows a normalized pattern, demonstrating that the `log1p` transformation is effective for the entire dataset.
    *   **Baby Users**: The `Rating_x_RatingCount_log` distribution for Baby Users appears similarly normalized. Comparing its central tendency to the 'Non-Baby Users' or 'All Users' might reveal if baby-related products tend to have higher or more concentrated combined ratings and review counts on average.
    *   **Non-Baby Users**: The distribution for Non-Baby Users is also normalized. It's important to compare its spread and peak with the Baby Users to understand if there are inherent differences in rating engagement between these two primary segments.

In summary, the `log1p` transformation is a valuable step in making `Rating_x_RatingCount` a more interpretable and robust feature for further analysis and modeling, allowing for more accurate comparisons and insights into user engagement.

In [ ]:

print("\nDistribution of Rating_Count Weight for ALL Users:")
histogram_boxplot(data, 'Rating_Count Weight')

print("\nDistribution of Rating_Count Weight for BABY Users:")
histogram_boxplot(baby_wipe_data, 'Rating_Count Weight')

print("\nDistribution of Rating_Count Weight for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'Rating_Count Weight')

In [ ]:
data['Rating_Count_Weight_log'] = np.log1p(data['Rating_Count Weight'])
baby_wipe_data['Rating_Count_Weight_log'] = np.log1p(baby_wipe_data['Rating_Count Weight'])
other_wipe_data['Rating_Count_Weight_log'] = np.log1p(other_wipe_data['Rating_Count Weight'])

print("\nDistribution of Log(Rating_Count Weight) for ALL Users:")
histogram_boxplot(data, 'Rating_Count_Weight_log')

print("\nDistribution of Log(Rating_Count Weight) for BABY Users:")
histogram_boxplot(baby_wipe_data, 'Rating_Count_Weight_log')

print("\nDistribution of Log(Rating_Count Weight) for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'Rating_Count_Weight_log')

### Observations and Insights from Log-Transformed `Rating_Count_Weight` Distributions:

After applying the `log1p` transformation to the `Rating_Count Weight` feature, the distributions across all user segments ('All Users', 'Baby Users', and 'Non-Baby Users') still exhibit significant right-skewness.

1.  **Persistent Skewness**: Unlike the `Rating_x_RatingCount_log` feature, which became more symmetrical after transformation, `Rating_Count_Weight_log` distributions remain heavily skewed to the right for all segments. This indicates that the vast majority of products have very low `Rating_Count Weight` values, even on a logarithmic scale, with a long tail extending to higher values.

2.  **Mean and Median Relationship**: In all three plots, the mean (green dashed line) is noticeably higher than the median (solid black line). This further confirms the right-skewed nature of the distributions, where a few data points with exceptionally high values pull the mean towards the right.

3.  **Comparison Across Segments**:
    *   **All Users**: The overall distribution highlights the extreme concentration of `Rating_Count Weight` at the lowest end, with a very sparse spread towards higher values.
    *   **Baby Users**: The distribution for 'Baby Users' mirrors the overall trend, showing that even within this segment, products generally have low `Rating_Count Weight`, suggesting that few baby products achieve a very high proportion of the total weighted ratings.
    *   **Non-Baby Users**: Similarly, 'Non-Baby Users' display a right-skewed distribution, indicating a similar pattern of uneven distribution of weighted ratings within this category.

In conclusion, while the `log1p` transformation compressed the range of `Rating_Count Weight`, it did not fully normalize its distribution. This suggests that the inherent nature of `Rating_Count Weight` (being a proportion that is typically very small for most items) leads to a highly concentrated distribution even after logarithmic scaling. Further analysis might require different transformations or specific handling for such highly skewed proportional data.

In [ ]:
data['Rating_Count_Weight_log'] = np.log1p(data['Rating_Count Weight'])
baby_wipe_data['Rating_Count_Weight_log'] = np.log1p(baby_wipe_data['Rating_Count Weight'])
other_wipe_data['Rating_Count_Weight_log'] = np.log1p(other_wipe_data['Rating_Count Weight'])

print("\nDistribution of Log(Rating_Count Weight) for ALL Users:")
histogram_boxplot(data, 'Rating_Count_Weight_log')

print("\nDistribution of Log(Rating_Count Weight) for BABY Users:")
histogram_boxplot(baby_wipe_data, 'Rating_Count_Weight_log')

print("\nDistribution of Log(Rating_Count Weight) for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'Rating_Count_Weight_log')

In [ ]:
print("\nDistribution of CurrentPrice for ALL Users:")
histogram_boxplot(data, 'CurrentPrice')

print("\nDistribution of CurrentPrice for BABY Users:")
histogram_boxplot(baby_wipe_data, 'CurrentPrice')

print("\nDistribution of CurrentPrice for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'CurrentPrice')


In [ ]:
data['CurrentPrice_log'] = np.log1p(data['CurrentPrice'])
baby_wipe_data['CurrentPrice_log'] = np.log1p(baby_wipe_data['CurrentPrice'])
other_wipe_data['CurrentPrice_log'] = np.log1p(other_wipe_data['CurrentPrice'])

print("\nDistribution of Log(CurrentPrice) for ALL Users:")
histogram_boxplot(data, 'CurrentPrice_log')

print("\nDistribution of Log(CurrentPrice) for BABY Users:")
histogram_boxplot(baby_wipe_data, 'CurrentPrice_log')

print("\nDistribution of Log(CurrentPrice) for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'CurrentPrice_log')

### **Observations and Insights from Log-Transformed `CurrentPrice` Distributions:**

The `log1p` transformation has significantly normalized the `CurrentPrice` feature across all user segments (All Users, Baby Users, Non-Baby Users). Key observations are:

1.  **Normalization of Distributions**: For all three user segments, the distributions of `CurrentPrice_log` now appear much more symmetrical and less skewed compared to the original `CurrentPrice` distributions. This transformation is crucial for many statistical models that assume normally distributed data.

2.  **Mean and Median Alignment**: After transformation, the mean (green dashed line) and median (solid black line) values for `CurrentPrice_log` are much closer to each other in all segments. This indicates that the central tendency of the data is well-represented and the influence of extreme high prices has been reduced.

3.  **Comparison Across Segments**:
    *   **All Users**: The overall distribution shows a relatively well-normalized pattern, with most products clustering around a central logarithmic price point.
    *   **Baby Users**: The `CurrentPrice_log` distribution for Baby Users also shows good normalization. Visually, **its peak and spread can be compared to other segments to infer if baby products tend to be priced higher or lower on a logarithmic scale.**
    *   **Non-Baby Users**: Similarly, the distribution for Non-Baby Users is normalized. Comparing its central tendency and spread with Baby Users can **highlight potential pricing differences or market segments**. For instance, if the peak for 'Non-Baby Users' is at a lower `CurrentPrice_log` value, it **might suggest a broader range of lower-priced products in this segment, or simply a different average price point**.

In summary, the `log1p` transformation has successfully addressed the skewness in `CurrentPrice`, making it a more suitable feature for robust analysis and predictive modeling. The normalized distributions allow for clearer interpretation of average prices and comparison of pricing strategies across different user segments on a more balanced scale.

In [ ]:
print("\nDistribution of Calculated Price/Count or Price/Bundle for ALL Users:")
histogram_boxplot(data, 'Calculated Price/Count or Price/Bundle')

print("\nDistribution of Calculated Price/Count or Price/Bundle for BABY Users:")
histogram_boxplot(baby_wipe_data, 'Calculated Price/Count or Price/Bundle')

print("\nDistribution of Calculated Price/Count or Price/Bundle for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'Calculated Price/Count or Price/Bundle')

In [ ]:
data['Calculated Price/Count or Price/Bundle_log'] = np.log1p(data['Calculated Price/Count or Price/Bundle'])
baby_wipe_data['Calculated Price/Count or Price/Bundle_log'] = np.log1p(baby_wipe_data['Calculated Price/Count or Price/Bundle'])
other_wipe_data['Calculated Price/Count or Price/Bundle_log'] = np.log1p(other_wipe_data['Calculated Price/Count or Price/Bundle'])

print("\nDistribution of Log(Calculated Price/Count or Price/Bundle) for ALL Users:")
histogram_boxplot(data, 'Calculated Price/Count or Price/Bundle_log')

print("\nDistribution of Log(Calculated Price/Count or Price/Bundle) for BABY Users:")
histogram_boxplot(baby_wipe_data, 'Calculated Price/Count or Price/Bundle_log')

print("\nDistribution of Log(Calculated Price/Count or Price/Bundle) for NON-BABY Users:")
histogram_boxplot(other_wipe_data, 'Calculated Price/Count or Price/Bundle_log')

### **Observations and Insights from Log-Transformed `Calculated Price/Count or Price/Bundle` Distributions:**

The `log1p` transformation has significantly normalized the `Calculated Price/Count or Price/Bundle` feature across all user segments (All Users, Baby Users, Non-Baby Users). Key observations are:

1.  **Normalization of Distributions**: For all three user segments, the distributions of `Calculated Price/Count or Price/Bundle_log` now appear much more symmetrical and less skewed compared to the original feature. This transformation is beneficial for statistical analysis and modeling, as it brings the data closer to a normal distribution.

2.  **Mean and Median Alignment**: After the transformation, the mean (green dashed line) and median (solid black line) values for `Calculated Price/Count or Price/Bundle_log` are considerably closer in all segments. This indicates a more centered distribution and a reduction in the impact of extreme values.

3.  **Comparison Across Segments**:
    *   **All Users**: The overall distribution shows a well-normalized pattern, suggesting that **most products fall within a central range of logarithmic price-per-count/bundle**.
    *   **Baby Users**: The `Calculated Price/Count or Price/Bundle_log` distribution for Baby Users is also well-normalized. Comparing its central tendency and spread with other segments can reveal if **baby products tend to have a higher or lower price-per-count/bundle on a logarithmic scale**.
    *   **Non-Baby Users**: Similarly, the distribution for Non-Baby Users is normalized. By comparing its mean and median with those of Baby Users, we can identify potential differences in pricing strategies or average unit prices between these segments. For example,**a lower mean/median for 'Non-Baby Users' might indicate a broader range of lower-priced products or a different average unit price in this category.**

In conclusion, the `log1p` transformation successfully addressed the skewness in `Calculated Price/Count or Price/Bundle`, making it a more robust and interpretable feature for further analysis and predictive modeling. The normalized distributions allow for more accurate comparisons of unit pricing across different user segments on a balanced scale.

In [ ]:
import matplotlib.ticker as mticker

# Filter data for non-null 'Gissel's L52W $' values for all datasets
data_filtered_all = data[data['Gissel\'s L52W $'].notna()].copy()
baby_wipe_data_filtered = baby_wipe_data[baby_wipe_data['Gissel\'s L52W $'].notna()].copy()
other_wipe_data_filtered = other_wipe_data[other_wipe_data['Gissel\'s L52W $'].notna()].copy()

# Define a common y-axis increment of $1 million (1,000,000)
y_increment = 1000000 # 1 million

# Function to set y-axis ticks in $1 million increments
def set_y_axis_ticks(ax, df_filtered, y_label):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    # Calculate start and end ticks to encompass the data range with the given increment
    y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    # Format y-tick labels to show in millions, with zero decimal places
    y_labels = [f'${t/1000000:.0f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)
    # Removed: ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fM'))

# Create the scatter plot for ALL Users
fig_all, ax_all = plt.subplots(figsize=(12, 10)) # Increased height
sns.scatterplot(data=data_filtered_all, x='Calculated Price/Count or Price/Bundle', y='Gissel\'s L52W $', ax=ax_all)
ax_all.set_title('Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $ (All Users)')
ax_all.set_xlabel('Calculated Price/Count or Price/Bundle')
ax_all.set_ylabel('Gissel\'s L52W $')
min_x_all = data_filtered_all['Calculated Price/Count or Price/Bundle'].min()
max_x_all = data_filtered_all['Calculated Price/Count or Price/Bundle'].max()
x_ticks_all = np.arange(np.floor(min_x_all / 0.1) * 0.1, np.ceil(max_x_all / 0.1) * 0.1 + 0.1, 0.1)
ax_all.set_xticks(x_ticks_all)
ax_all.set_xticklabels([f'{t:.1f}' for t in x_ticks_all], rotation=45, ha='right')
set_y_axis_ticks(ax_all, data_filtered_all, 'Gissel\'s L52W $')
ax_all.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Create the scatter plot for BABY Users
fig_baby, ax_baby = plt.subplots(figsize=(12, 10)) # Increased height
sns.scatterplot(data=baby_wipe_data_filtered, x='Calculated Price/Count or Price/Bundle', y='Gissel\'s L52W $', ax=ax_baby)
ax_baby.set_title('Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $ (Baby Users)')
ax_baby.set_xlabel('Calculated Price/Count or Price/Bundle')
ax_baby.set_ylabel('Gissel\'s L52W $')
min_x_baby = baby_wipe_data_filtered['Calculated Price/Count or Price/Bundle'].min()
max_x_baby = baby_wipe_data_filtered['Calculated Price/Count or Price/Bundle'].max()
x_ticks_baby = np.arange(np.floor(min_x_baby / 0.1) * 0.1, np.ceil(max_x_baby / 0.1) * 0.1 + 0.1, 0.1)
ax_baby.set_xticks(x_ticks_baby)
ax_baby.set_xticklabels([f'{t:.1f}' for t in x_ticks_baby], rotation=45, ha='right')
set_y_axis_ticks(ax_baby, baby_wipe_data_filtered, 'Gissel\'s L52W $')
ax_baby.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Create the scatter plot for NON-BABY Users
fig_non_baby, ax_non_baby = plt.subplots(figsize=(12, 10)) # Increased height
sns.scatterplot(data=other_wipe_data_filtered, x='Calculated Price/Count or Price/Bundle', y='Gissel\'s L52W $', ax=ax_non_baby)
ax_non_baby.set_title('Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $ (Non-Baby Users)')
ax_non_baby.set_xlabel('Calculated Price/Count or Price/Bundle')
ax_non_baby.set_ylabel('Gissel\'s L52W $')
min_x_non_baby = other_wipe_data_filtered['Calculated Price/Count or Price/Bundle'].min()
max_x_non_baby = other_wipe_data_filtered['Calculated Price/Count or Price/Bundle'].max()
x_ticks_non_baby = np.arange(np.floor(min_x_non_baby / 0.1) * 0.1, np.ceil(max_x_non_baby / 0.1) * 0.1 + 0.1, 0.1)
ax_non_baby.set_xticks(x_ticks_non_baby)
ax_non_baby.set_xticklabels([f'{t:.1f}' for t in x_ticks_non_baby], rotation=45, ha='right')
set_y_axis_ticks(ax_non_baby, other_wipe_data_filtered, 'Gissel\'s L52W $')
ax_non_baby.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Top 10% Rating_Count Weight Brand's L52W's Sales (Include the brands that have/don't have sales \$ in Profitero (Gissel's L52W $)**

In [ ]:
# Top 10% Rating_Count Weight Brand's L52W's Sales (Include the brands that have/don't have sales $ in Profitero (Gissel's L52W $)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common y-axis increment of $1 million (1,000,000)
y_increment = 1000000 # 1 million

# Function to set y-axis ticks in $1 million increments
def set_y_axis_ticks(ax, df_filtered, y_label):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    # Calculate start and end ticks to encompass the data range with the given increment
    y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    # Format y-tick labels to show in millions, with zero decimal places
    y_labels = [f'${t/1000000:.0f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=0.1):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Calculate top 10% brands by Rating_Count Weight for All Users
top_brands_all_count = max(1, int(len(data['CLEANED BRAND'].unique()) * 0.1))
top_brands_all = data.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_all_count).index

# Calculate top 10% brands by Rating_Count Weight for Baby Users
top_brands_baby_count = max(1, int(len(baby_wipe_data['CLEANED BRAND'].unique()) * 0.1))
top_brands_baby = baby_wipe_data.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_baby_count).index

# Calculate top 10% brands by Rating_Count Weight for Non-Baby Users
top_brands_non_baby_count = max(1, int(len(other_wipe_data['CLEANED BRAND'].unique()) * 0.1))
top_brands_non_baby = other_wipe_data.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_non_baby_count).index

# Prepare data for plotting and table: Filter original dataframes to include only the selected top brands
data_for_top_brands_plotting = data[data['CLEANED BRAND'].isin(top_brands_all)].copy()
baby_data_for_top_brands_plotting = baby_wipe_data[baby_wipe_data['CLEANED BRAND'].isin(top_brands_baby)].copy()
non_baby_data_for_top_brands_plotting = other_wipe_data[other_wipe_data['CLEANED BRAND'].isin(top_brands_non_baby)].copy()

# --- AGGREGATE FOR PLOTTING --- (one point per brand per user segment)
aggregated_plotting_df = pd.DataFrame()

# For 'All Users' segment
if not data_for_top_brands_plotting.empty:
    agg_all = data_for_top_brands_plotting.groupby('CLEANED BRAND').agg(
        {'Calculated Price/Count or Price/Bundle': 'mean',
         "Gissel's L52W $": 'sum',
         'Rating_Count Weight': 'sum' # Keep this for consistency, though not directly plotted on axes
        }
    ).reset_index()
    agg_all['User_Segment_for_Plot'] = 'All Users' # Add a user type for clear distinction
    aggregated_plotting_df = pd.concat([aggregated_plotting_df, agg_all])

# For 'Baby Users' segment
if not baby_data_for_top_brands_plotting.empty:
    agg_baby = baby_data_for_top_brands_plotting.groupby('CLEANED BRAND').agg(
        {'Calculated Price/Count or Price/Bundle': 'mean',
         "Gissel's L52W $": 'sum',
         'Rating_Count Weight': 'sum'
        }
    ).reset_index()
    agg_baby['User_Segment_for_Plot'] = 'Baby Users'
    aggregated_plotting_df = pd.concat([aggregated_plotting_df, agg_baby])

# For 'Non-Baby Users' segment
if not non_baby_data_for_top_brands_plotting.empty:
    agg_non_baby = non_baby_data_for_top_brands_plotting.groupby('CLEANED BRAND').agg(
        {'Calculated Price/Count or Price/Bundle': 'mean',
         "Gissel's L52W $": 'sum',
         'Rating_Count Weight': 'sum'
        }
    ).reset_index()
    agg_non_baby['User_Segment_for_Plot'] = 'Non-Baby Users'
    aggregated_plotting_df = pd.concat([aggregated_plotting_df, agg_non_baby])


# Filter aggregated data for non-null 'Gissel's L52W $' specifically for plotting
aggregated_plotting_df_filtered = aggregated_plotting_df[aggregated_plotting_df["Gissel's L52W $"].notna()]

# Plot for ALL Users - Top 10% Brands
plot_data_all_agg = aggregated_plotting_df_filtered[aggregated_plotting_df_filtered['User_Segment_for_Plot'] == 'All Users'].copy()
if not plot_data_all_agg.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(12, 10))
    # Determine hue_order based on Gissel's L52W $ from the aggregated data
    brand_sales_all_order = plot_data_all_agg.sort_values(by="Gissel's L52W $", ascending=False)['CLEANED BRAND']
    sns.scatterplot(data=plot_data_all_agg, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='CLEANED BRAND', hue_order=brand_sales_all_order, ax=ax_all_top, s=100) # Increased marker size
    ax_all_top.set_title(f'Top {top_brands_all_count} Brands (All Users) by Rating_Count Weight\n Mean Calculated Price/Count or Price/Bundle vs. Total Gissel\'s L52W $ (Aggregated)')
    ax_all_top.set_xlabel('Mean Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Total Gissel\'s L52W $')
    set_x_axis_ticks(ax_all_top, plot_data_all_agg, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_all_top, plot_data_all_agg, "Gissel's L52W $")
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND')
    plt.tight_layout()
    plt.show()
else:
    print(f"No aggregated data with 'Gissel\"s L52W $' for top {top_brands_all_count} brands (All Users) to plot.")


# Plot for BABY Users - Top 10% Brands
plot_data_baby_agg = aggregated_plotting_df_filtered[aggregated_plotting_df_filtered['User_Segment_for_Plot'] == 'Baby Users'].copy()
if not plot_data_baby_agg.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(12, 10))
    # Determine hue_order based on Gissel's L52W $ from the aggregated data
    brand_sales_baby_order = plot_data_baby_agg.sort_values(by="Gissel's L52W $", ascending=False)['CLEANED BRAND']
    sns.scatterplot(data=plot_data_baby_agg, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='CLEANED BRAND', hue_order=brand_sales_baby_order, ax=ax_baby_top, s=100) # Increased marker size
    ax_baby_top.set_title(f'Top {top_brands_baby_count} Brands (Baby Users) by Rating_Count Weight\n Mean Calculated Price/Count or Price/Bundle vs. Total Gissel\'s L52W $ (Aggregated)')
    ax_baby_top.set_xlabel('Mean Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Total Gissel\'s L52W $')
    set_x_axis_ticks(ax_baby_top, plot_data_baby_agg, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_baby_top, plot_data_baby_agg, "Gissel's L52W $")
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND')
    plt.tight_layout()
    plt.show()
else:
    print(f"No aggregated data with 'Gissel\"s L52W $' for top {top_brands_baby_count} brands (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Brands
plot_data_non_baby_agg = aggregated_plotting_df_filtered[aggregated_plotting_df_filtered['User_Segment_for_Plot'] == 'Non-Baby Users'].copy()
if not plot_data_non_baby_agg.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(12, 10))
    # Determine hue_order based on Gissel's L52W $ from the aggregated data
    brand_sales_non_baby_order = plot_data_non_baby_agg.sort_values(by="Gissel's L52W $", ascending=False)['CLEANED BRAND']
    sns.scatterplot(data=plot_data_non_baby_agg, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='CLEANED BRAND', hue_order=brand_sales_non_baby_order, ax=ax_non_baby_top, s=100) # Increased marker size
    ax_non_baby_top.set_title(f'Top {top_brands_non_baby_count} Brands (Non-Baby Users) by Rating_Count Weight\n Mean Calculated Price/Count or Price/Bundle vs. Total Gissel\'s L52W $ (Aggregated)')
    ax_non_baby_top.set_xlabel('Mean Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Total Gissel\'s L52W $')
    set_x_axis_ticks(ax_non_baby_top, plot_data_non_baby_agg, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_non_baby_top, plot_data_non_baby_agg, "Gissel's L52W $")
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND')
    plt.tight_layout()
    plt.show()
else:
    print(f"No aggregated data with 'Gissel\"s L52W $' for top {top_brands_non_baby_count} brands (Non-Baby Users) to plot.")


# --- New code for table and CSV download ---

# Re-combine the original filtered-by-top-brand dataframes for the final output_df table
# These retain the original 'User' column for correct grouping in output_df
combined_top_brands_data_for_table = pd.concat([data_for_top_brands_plotting, baby_data_for_top_brands_plotting, non_baby_data_for_top_brands_plotting])

output_df = combined_top_brands_data_for_table.groupby(['CLEANED BRAND', 'User']).agg(
    {'Calculated Price/Count or Price/Bundle': 'mean',
     "Gissel's L52W $": 'sum', # sum will ignore NaN by default
     'Rating_Count Weight': 'sum'
    }
).reset_index()

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df = output_df.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Brands Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_brands_scatter_plot_data_consolidated.csv'
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

**Top 10% Rating_Count Weight Brand's L52W's Sales (Only include the brands that have sales \$ in Profitero (Gissel's L52W $)**

In [ ]:
# Top 10% Rating_Count Weight Brand's L52W's Sales (Only include the brands that have "Gissel's L52W $"  )
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common y-axis increment of $1 million (1,000,000)
y_increment = 1000000 # 1 million

# Function to set y-axis ticks in $1 million increments
def set_y_axis_ticks(ax, df_filtered, y_label):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    # Calculate start and end ticks to encompass the data range with the given increment
    y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    # Format y-tick labels to show in millions, with zero decimal places
    y_labels = [f'${t/1000000:.0f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=0.1):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Filter data for non-null 'Gissel's L52W $' values for all datasets
data_filtered_all = data[data["Gissel's L52W $"].notna()].copy()
baby_wipe_data_filtered = baby_wipe_data[baby_wipe_data["Gissel's L52W $"].notna()].copy()
other_wipe_data_filtered = other_wipe_data[other_wipe_data["Gissel's L52W $"].notna()].copy()

# Calculate top 10% brands by Rating_Count Weight for All Users
top_brands_all_count = max(1, int(len(data_filtered_all['CLEANED BRAND'].unique()) * 0.1))
top_brands_all = data_filtered_all.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_all_count).index
data_top_brands = data_filtered_all[data_filtered_all['CLEANED BRAND'].isin(top_brands_all)].copy()

# Calculate top 10% brands by Rating_Count Weight for Baby Users
top_brands_baby_count = max(1, int(len(baby_wipe_data_filtered['CLEANED BRAND'].unique()) * 0.1))
top_brands_baby = baby_wipe_data_filtered.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_baby_count).index
baby_data_top_brands = baby_wipe_data_filtered[baby_wipe_data_filtered['CLEANED BRAND'].isin(top_brands_baby)].copy()

# Calculate top 10% brands by Rating_Count Weight for Non-Baby Users
top_brands_non_baby_count = max(1, int(len(other_wipe_data_filtered['CLEANED BRAND'].unique()) * 0.1))
top_brands_non_baby = other_wipe_data_filtered.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_non_baby_count).index
non_baby_data_top_brands = other_wipe_data_filtered[other_wipe_data_filtered['CLEANED BRAND'].isin(top_brands_non_baby)].copy()

# Get hue order by 'Gissel's L52W $' sum for each group
brand_sales_all_order = data_top_brands.groupby('CLEANED BRAND')["Gissel's L52W $"].sum().sort_values(ascending=False).index
brand_sales_baby_order = baby_data_top_brands.groupby('CLEANED BRAND')["Gissel's L52W $"].sum().sort_values(ascending=False).index
brand_sales_non_baby_order = non_baby_data_top_brands.groupby('CLEANED BRAND')["Gissel's L52W $"].sum().sort_values(ascending=False).index

# Plot for ALL Users - Top 10% Brands
if not data_top_brands.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=data_top_brands, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='CLEANED BRAND', hue_order=brand_sales_all_order, ax=ax_all_top)
    ax_all_top.set_title(f'Top {top_brands_all_count} Brands (All Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_all_top, data_top_brands, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_all_top, data_top_brands, "Gissel's L52W $")
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No data for top 10% brands (All Users) to plot.")


# Plot for BABY Users - Top 10% Brands
if not baby_data_top_brands.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=baby_data_top_brands, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='CLEANED BRAND', hue_order=brand_sales_baby_order, ax=ax_baby_top)
    ax_baby_top.set_title(f'Top {top_brands_baby_count} Brands (Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_baby_top, baby_data_top_brands, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_baby_top, baby_data_top_brands, "Gissel's L52W $")
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No data for top 10% brands (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Brands
if not non_baby_data_top_brands.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=non_baby_data_top_brands, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='CLEANED BRAND', hue_order=brand_sales_non_baby_order, ax=ax_non_baby_top)
    ax_non_baby_top.set_title(f'Top {top_brands_non_baby_count} Brands (Non-Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_non_baby_top, non_baby_data_top_brands, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_non_baby_top, non_baby_data_top_brands, "Gissel's L52W $")
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No data for top 10% brands (Non-Baby Users) to plot.")


# --- New code for table and CSV download --- Modify for Consolidation

# Combine the top brands dataframes for all user types
# Consolidate by grouping by 'CLEANED BRAND' and 'User' and aggregating sales and price.
combined_top_brands_data = pd.concat([data_top_brands, baby_data_top_brands, non_baby_data_top_brands])

output_df = combined_top_brands_data.groupby(['CLEANED BRAND', 'User']).agg(
    {'Calculated Price/Count or Price/Bundle': 'mean',
     "Gissel's L52W $": 'sum',
     'Rating_Count Weight': 'sum' # Add Rating_Count Weight to the aggregation
    }
).reset_index()

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Brands Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_brands_scatter_plot_data_consolidated.csv'
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

### **Observations and Insights from the Scatter Plot: 'Calculated Price/Count or Price/Bundle' vs. 'Gissel's L52W $'**

1.  **Concentration at Lower Values**: **The majority of products are clustered at the lower end of both axes. This indicates that most items have a relatively low 'Calculated Price/Count** or Price/Bundle' (mostly below \$1) and **also generate lower "Gissel's L52W $" (primarily below \$500,000**).

2.  **Lack of Strong Linear Correlation**: There isn't a clear, strong linear relationship between the two variables. While there's a general tendency for higher "Gissel's L52W $" values to occur at slightly higher "Calculated Price/Count or Price/Bundle" values, the correlation is not very tight.

3.  **High Sales at Moderate Unit Prices**: Several products achieve very high "Gissel's L52W $" (e.g., in the millions) with **moderate "Calculated Price/Count or Price/Bundle"** (e.g., between 0.1 and 1.0). **This suggests that optimal sales performance in dollars might not necessarily come from the cheapest or most expensive unit prices.**

4.  **Limited Sales at High Unit Prices**: Products with extremely high 'Calculated Price/Count or Price/Bundle' (e.g., above 10) generally show very low 'Gissel's L52W $'. **This could indicate that very high unit prices might significantly limit overall sales volume and thus total revenue.**

**Top 10% Rating_Count Weight Products' L52W's Sales (Only include the products that have "Gissel's L52W $"  )**

In [ ]:
# Top 10% Rating_Count Weight Products' L52W's Sales (Only include the products that have "Gissel's L52W $")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common y-axis increment of $1 million (1,000,000)
y_increment = 1000000 # 1 million

# Function to set y-axis ticks in $1 million increments
def set_y_axis_ticks(ax, df_filtered, y_label):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    # Calculate start and end ticks to encompass the data range with the given increment
    y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    # Format y-tick labels to show in millions, with zero decimal places
    y_labels = [f'${t/1000000:.0f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=0.1):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Filter data for non-null 'Gissel's L52W $' values for all datasets
data_filtered_all = data[data["Gissel's L52W $"].notna()].copy()
baby_wipe_data_filtered = baby_wipe_data[baby_wipe_data["Gissel's L52W $"].notna()].copy()
other_wipe_data_filtered = other_wipe_data[other_wipe_data["Gissel's L52W $"].notna()].copy()

# Calculate top 10% products by Rating_Count Weight for All Users
top_products_all_count = max(1, int(len(data_filtered_all) * 0.1))
data_top_products = data_filtered_all.nlargest(top_products_all_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Baby Users
top_products_baby_count = max(1, int(len(baby_wipe_data_filtered) * 0.1))
baby_data_top_products = baby_wipe_data_filtered.nlargest(top_products_baby_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Non-Baby Users
top_products_non_baby_count = max(1, int(len(other_wipe_data_filtered) * 0.1))
non_baby_data_top_products = other_wipe_data_filtered.nlargest(top_products_non_baby_count, 'Rating_Count Weight').copy()


# Get hue order by 'Rating_Count Weight' sum for each group (for the selected top products)
# No longer needed as ProductTitle is used for hue and is typically unique
# brand_sales_all_order = data_top_products.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index
# brand_sales_baby_order = baby_data_top_products.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index
# brand_sales_non_baby_order = non_baby_data_top_products.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index

# Plot for ALL Users - Top 10% Products
if not data_top_products.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_all_top) # Changed hue to ProductTitle
    ax_all_top.set_title(f'Top {top_products_all_count} Products (All Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_all_top, data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_all_top, data_top_products, "Gissel's L52W $")
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_all_count} products (All Users) to plot.")


# Plot for BABY Users - Top 10% Products
if not baby_data_top_products.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=baby_data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_baby_top) # Changed hue to ProductTitle
    ax_baby_top.set_title(f'Top {top_products_baby_count} Products (Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_baby_top, baby_data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_baby_top, baby_data_top_products, "Gissel's L52W $")
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_baby_count} products (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Products
if not non_baby_data_top_products.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=non_baby_data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_non_baby_top) # Changed hue to ProductTitle
    ax_non_baby_top.set_title(f'Top {top_products_non_baby_count} Products (Non-Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_non_baby_top, non_baby_data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_non_baby_top, non_baby_data_top_products, "Gissel's L52W $")
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_non_baby_count} products (Non-Baby Users) to plot.")


# --- New code for table and CSV download --- Modify for Consolidation

# Combine the top products dataframes for all user types
# Consolidate by grouping by 'CLEANED BRAND' and 'User' and aggregating sales and price.
combined_top_products_data = pd.concat([data_top_products, baby_data_top_products, non_baby_data_top_products])

output_df = combined_top_products_data.groupby(['CLEANED BRAND', 'User']).agg(
    {'Calculated Price/Count or Price/Bundle': 'mean',
     "Gissel's L52W $": 'sum',
     'Rating_Count Weight': 'sum' # Add Rating_Count Weight to the aggregation
    }
).reset_index()

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df = output_df.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Products Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_products_scatter_plot_data_consolidated.csv' # Changed filename to reflect products
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

**# Top 50% Products TotalCount Distribution **

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Re-defining histogram_boxplot here to ensure it's available within this cell's context,
# even though it's defined elsewhere. This is safer if the notebook execution order
# isn't strictly linear or if the agent only focuses on the target cell.
def histogram_boxplot(data, feature, figsize = (12, 7), kde = False, bins = None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows = 2,      # Number of rows of the subplot grid = 2
        sharex = True,  # x-axis will be shared among all subplots
        gridspec_kw = {"height_ratios": (0.25, 0.75)},
        figsize = figsize,
    )                   # Creating the 2 subplots
    sns.boxplot(
        data = data, x = feature, ax = ax_box2, showmeans = True, color = "violet"
    )                   # Boxplot will be created and a star will indicate the mean value of the column

    # Histogram plotting
    sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2, bins = bins, palette = "viridis"
    ) if bins else sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2, palette = "viridis"
    )                   # For histogram

    # Add percentage labels to the histogram bars
    total_count = len(data[feature])
    if total_count > 0: # Avoid division by zero if DataFrame is empty
        for p in ax_hist2.patches:
            height = p.get_height()
            if height > 0: # Only label non-empty bars
                percentage = (height / total_count) * 100
                ax_hist2.annotate(f'{percentage:.1f}%', # Display percentage with one decimal place
                                 (p.get_x() + p.get_width() / 2., height),
                                 ha='center', va='bottom', fontsize=8, color='black',
                                 xytext=(0, 5), textcoords='offset points') # Offset text slightly above bar

    ax_hist2.axvline(
        data[feature].mean(), color = "green", linestyle = "--"
    )                   # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color = "black", linestyle = "-"
    )                   # Add median to the histogram

    # Add data labels for mean and median
    mean_val = data[feature].mean()
    median_val = data[feature].median()

    # Annotate mean
    ax_hist2.text(mean_val + (ax_hist2.get_xlim()[1] - ax_hist2.get_xlim()[0]) * 0.01,
                  ax_hist2.get_ylim()[1] * 0.9,
                  f'Mean: {mean_val:.2f}', color='green', ha='left', va='center')
    # Annotate median
    ax_hist2.text(median_val + (ax_hist2.get_xlim()[1] - ax_hist2.get_xlim()[0]) * 0.01,
                  ax_hist2.get_ylim()[1] * 0.8,
                  f'Median: {median_val:.2f}', color='black', ha='left', va='center')

    # Explicitly set x-axis labels for both subplots
    ax_hist2.set_xlabel(feature)
    ax_box2.set_xlabel(feature)

    # Removed specific tick logic for CurrentPrice as this plot is for TotalCount
    f2.suptitle(f'Distribution of {feature}', fontsize=16)
    plt.show()

# Ensure data and derived dataframes have 'Rating_Count Weight' column
# This recalculation ensures consistency, as previous steps might have altered 'data' or its derivatives.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define user segments
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

print("Generating TotalCount distribution for top 50% Rating_Count Weight products:")

for segment_name, segment_df in user_segments.items():
    if segment_df.empty:
        print(f"No data available for {segment_name}.")
        continue

    # Calculate the 50th percentile for 'Rating_Count Weight' in the current segment
    percentile_50_threshold = segment_df['Rating_Count Weight'].quantile(0.50)

    # Filter the DataFrame to include only products in the top 50% by 'Rating_Count Weight'
    top_50_percent_products = segment_df[
        segment_df['Rating_Count Weight'] >= percentile_50_threshold
    ].copy()

    if not top_50_percent_products.empty:
        # Exclude the top 2% of 'TotalCount' values (outliers)
        total_count_98th_percentile = top_50_percent_products['TotalCount'].quantile(0.98)
        filtered_products_no_outliers = top_50_percent_products[
            top_50_percent_products['TotalCount'] <= total_count_98th_percentile
        ].copy()

        if not filtered_products_no_outliers.empty:
            print(f"\n--- TotalCount Distribution for Top 50% Rating_Count Weight Products in {segment_name} (excluding top 2% TotalCount outliers) ---")
            histogram_boxplot(filtered_products_no_outliers, 'TotalCount')
        else:
            print(f"No products remaining after filtering top 50% Rating_Count Weight and excluding top 2% TotalCount outliers for {segment_name}.")
    else:
        print(f"No products found in the top 50% Rating_Count Weight for {segment_name}.")

print("\nFinished generating TotalCount distributions for top 50% Rating_Count Weight products.")

In [ ]:
# Top 10% Rating_Count Weight Products' L52W's Sales (Only include the products that have "Gissel's L52W $")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common y-axis increment of $1 million (1,000,000)
y_increment = 1000000 # 1 million

# Function to set y-axis ticks in $1 million increments
def set_y_axis_ticks(ax, df_filtered, y_label):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    # Calculate start and end ticks to encompass the data range with the given increment
    y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    # Format y-tick labels to show in millions, with zero decimal places
    y_labels = [f'${t/1000000:.0f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=0.1):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Filter data for non-null 'Gissel's L52W $' values for all datasets
data_filtered_all = data[data["Gissel's L52W $"].notna()].copy()
baby_wipe_data_filtered = baby_wipe_data[baby_wipe_data["Gissel's L52W $"].notna()].copy()
other_wipe_data_filtered = other_wipe_data[other_wipe_data["Gissel's L52W $"].notna()].copy()

# Calculate top 10% products by Rating_Count Weight for All Users
top_products_all_count = max(1, int(len(data_filtered_all) * 0.1))
data_top_products = data_filtered_all.nlargest(top_products_all_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Baby Users
top_products_baby_count = max(1, int(len(baby_wipe_data_filtered) * 0.1))
baby_data_top_products = baby_wipe_data_filtered.nlargest(top_products_baby_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Non-Baby Users
top_products_non_baby_count = max(1, int(len(other_wipe_data_filtered) * 0.1))
non_baby_data_top_products = other_wipe_data_filtered.nlargest(top_products_non_baby_count, 'Rating_Count Weight').copy()


# Get hue order by 'Rating_Count Weight' sum for each group (for the selected top products)
# No longer needed as ProductTitle is used for hue and is typically unique
# brand_sales_all_order = data_top_products.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index
# brand_sales_baby_order = baby_data_top_products.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index
# brand_sales_non_baby_order = non_baby_data_top_products.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index

# Plot for ALL Users - Top 10% Products
if not data_top_products.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_all_top) # Changed hue to ProductTitle
    ax_all_top.set_title(f'Top {top_products_all_count} Products (All Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_all_top, data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_all_top, data_top_products, "Gissel's L52W $")
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_all_count} products (All Users) to plot.")


# Plot for BABY Users - Top 10% Products
if not baby_data_top_products.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=baby_data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_baby_top) # Changed hue to ProductTitle
    ax_baby_top.set_title(f'Top {top_products_baby_count} Products (Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_baby_top, baby_data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_baby_top, baby_data_top_products, "Gissel's L52W $")
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_baby_count} products (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Products
if not non_baby_data_top_products.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=non_baby_data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_non_baby_top) # Changed hue to ProductTitle
    ax_non_baby_top.set_title(f'Top {top_products_non_baby_count} Products (Non-Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_non_baby_top, non_baby_data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_non_baby_top, non_baby_data_top_products, "Gissel's L52W $")
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_non_baby_count} products (Non-Baby Users) to plot.")


# --- New code for table and CSV download --- Modify for Consolidation

# Combine the top products dataframes for all user types
# Consolidate by grouping by 'CLEANED BRAND' and 'User' and aggregating sales and price.
combined_top_products_data = pd.concat([data_top_products, baby_data_top_products, non_baby_data_top_products])

output_df = combined_top_products_data.groupby(['CLEANED BRAND', 'User']).agg(
    {'Calculated Price/Count or Price/Bundle': 'mean',
     "Gissel's L52W $": 'sum',
     'Rating_Count Weight': 'sum' # Add Rating_Count Weight to the aggregation
    }
).reset_index()

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df = output_df.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Products Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_products_scatter_plot_data_consolidated.csv' # Changed filename to reflect products
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

**Top 10% Rating_Count Weight Products' L52W's Sales (Include the products that have/don't have "Gissel's L52W $"  )**

In [ ]:
# Top 10% Rating_Count Weight Products' L52W's Sales (Include the products that have/don't have "Gissel's L52W $" )
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common y-axis increment of $1 million (1,000,000)
y_increment = 1000000 # 1 million

# Function to set y-axis ticks in $1 million increments
def set_y_axis_ticks(ax, df_filtered, y_label):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    # If all values are 0, min_y and max_y will be 0. np.arange(0, y_increment, y_increment) will generate [0].
    # To ensure at least two ticks (0 and 1M) if all values are 0, we can adjust max_y for arange.
    if max_y == 0:
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)

    # Format y-tick labels to show in millions, with zero decimal places
    y_labels = [f'${t/1000000:.0f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=0.1):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Prepare data for all products, including those with null 'Gissel's L52W $'
data_all_products = data.copy()
baby_wipe_data_all_products = baby_wipe_data.copy()
other_wipe_data_all_products = other_wipe_data.copy()

# Fill NaN 'Gissel's L52W $' with 0 for plotting and aggregation to represent no sales data
data_all_products["Gissel's L52W $"] = data_all_products["Gissel's L52W $"].fillna(0)
baby_wipe_data_all_products["Gissel's L52W $"] = baby_wipe_data_all_products["Gissel's L52W $"].fillna(0)
other_wipe_data_all_products["Gissel's L52W $"] = other_wipe_data_all_products["Gissel's L52W $"].fillna(0)


# Calculate top 10% products by Rating_Count Weight for All Users
top_products_all_count = max(1, int(len(data_all_products) * 0.1))
data_top_products = data_all_products.nlargest(top_products_all_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Baby Users
top_products_baby_count = max(1, int(len(baby_wipe_data_all_products) * 0.1))
baby_data_top_products = baby_wipe_data_all_products.nlargest(top_products_baby_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Non-Baby Users
top_products_non_baby_count = max(1, int(len(other_wipe_data_all_products) * 0.1))
non_baby_data_top_products = other_wipe_data_all_products.nlargest(top_products_non_baby_count, 'Rating_Count Weight').copy()


# Plot for ALL Users - Top 10% Products
if not data_top_products.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_all_top) # Changed hue to ProductTitle
    ax_all_top.set_title(f'Top {top_products_all_count} Products (All Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_all_top, data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_all_top, data_top_products, "Gissel's L52W $")
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_all_count} products (All Users) to plot.")


# Plot for BABY Users - Top 10% Products
if not baby_data_top_products.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=baby_data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_baby_top) # Changed hue to ProductTitle
    ax_baby_top.set_title(f'Top {top_products_baby_count} Products (Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_baby_top, baby_data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_baby_top, baby_data_top_products, "Gissel's L52W $")
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_baby_count} products (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Products
if not non_baby_data_top_products.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(12, 10))
    sns.scatterplot(data=non_baby_data_top_products, x='Calculated Price/Count or Price/Bundle', y="Gissel's L52W $", hue='ProductTitle', ax=ax_non_baby_top) # Changed hue to ProductTitle
    ax_non_baby_top.set_title(f'Top {top_products_non_baby_count} Products (Non-Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Gissel\'s L52W $')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Gissel\'s L52W $')
    set_x_axis_ticks(ax_non_baby_top, non_baby_data_top_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks(ax_non_baby_top, non_baby_data_top_products, "Gissel's L52W $")
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Changed legend title
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_non_baby_count} products (Non-Baby Users) to plot.")


# --- New code for table and CSV download --- Modify for Consolidation

# Combine the top products dataframes for all user types
# Consolidate by grouping by 'CLEANED BRAND' and 'User' and aggregating sales and price.
combined_top_products_data = pd.concat([data_top_products, baby_data_top_products, non_baby_data_top_products])

output_df = combined_top_products_data.groupby(['CLEANED BRAND', 'User']).agg(
    {'Calculated Price/Count or Price/Bundle': 'mean',
     "Gissel's L52W $": 'sum',
     'Rating_Count Weight': 'sum' # Add Rating_Count Weight to the aggregation
    }
).reset_index()

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df = output_df.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Products Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_products_scatter_plot_data_consolidated.csv' # Changed filename to reflect products
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

In [ ]:
# display(output_df)

**Top 10% Rating_Count Weight Brand's Current Price (Include the brands that have/don't have sales $ in Profitero (Gissel's L52W $)**

In [ ]:
# Top 10% Rating_Count Weight Brand's Current Price ("Gissel's L52W $ might not be available for every brand/product)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common x-axis increment for price
x_increment = 0.1

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=x_increment):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Function to set y-axis ticks for linear scale CurrentPrice
def set_y_axis_ticks_current_price(ax, df_filtered, y_label, base_increment=10): # Adjusted base_increment for CurrentPrice
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    # Ensure ticks are well-spaced and cover the range
    y_ticks = np.arange(np.floor(min_y / base_increment) * base_increment,
                        np.ceil(max_y / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f'${t:.0f}' for t in y_ticks]) # Format as whole dollars

# DO NOT filter data for non-null 'Gissel's L52W $' values as per user request.
# Use the original dataframes: data, baby_wipe_data, other_wipe_data

# Calculate top 10% brands by Rating_Count Weight for All Users
top_brands_all_count = max(1, int(len(data['CLEANED BRAND'].unique()) * 0.1))
top_brands_all = data[data['CLEANED BRAND'].isin(
    data.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_all_count).index
)].copy()

# Calculate top 10% brands by Rating_Count Weight for Baby Users
top_brands_baby_count = max(1, int(len(baby_wipe_data['CLEANED BRAND'].unique()) * 0.1))
top_brands_baby = baby_wipe_data[baby_wipe_data['CLEANED BRAND'].isin(
    baby_wipe_data.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_baby_count).index
)].copy()

# Calculate top 10% brands by Rating_Count Weight for Non-Baby Users
top_brands_non_baby_count = max(1, int(len(other_wipe_data['CLEANED BRAND'].unique()) * 0.1))
top_brands_non_baby = other_wipe_data[other_wipe_data['CLEANED BRAND'].isin(
    other_wipe_data.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_non_baby_count).index
)].copy()

# Get hue order by 'Rating_Count Weight' sum for each group
brand_sales_all_order = top_brands_all.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index
brand_sales_baby_order = top_brands_baby.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index
brand_sales_non_baby_order = top_brands_non_baby.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index

# Plot for ALL Users - Top 10% Brands
if not top_brands_all.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_brands_all, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice", hue='CLEANED BRAND', hue_order=brand_sales_all_order, ax=ax_all_top, legend='full')
    ax_all_top.set_title(f'Top {top_brands_all_count} Brands (All Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Current Price')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Current Price ($)')
    set_x_axis_ticks(ax_all_top, top_brands_all, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_current_price(ax_all_top, top_brands_all, 'CurrentPrice')
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_brands_all_count} brands (All Users) to plot.")


# Plot for BABY Users - Top 10% Brands
if not top_brands_baby.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_brands_baby, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice", hue='CLEANED BRAND', hue_order=brand_sales_baby_order, ax=ax_baby_top, legend='full')
    ax_baby_top.set_title(f'Top {top_brands_baby_count} Brands (Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Current Price')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Current Price ($)')
    set_x_axis_ticks(ax_baby_top, top_brands_baby, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_current_price(ax_baby_top, top_brands_baby, 'CurrentPrice')
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_brands_baby_count} brands (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Brands
if not top_brands_non_baby.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_brands_non_baby, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice", hue='CLEANED BRAND', hue_order=brand_sales_non_baby_order, ax=ax_non_baby_top, legend='full')
    ax_non_baby_top.set_title(f'Top {top_brands_non_baby_count} Brands (Non-Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Current Price')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Current Price ($)')
    set_x_axis_ticks(ax_non_baby_top, top_brands_non_baby, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_current_price(ax_non_baby_top, top_brands_non_baby, 'CurrentPrice')
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_brands_non_baby_count} brands (Non-Baby Users) to plot.")


# --- New code for table and CSV download --- Modify for Consolidation

# Combine the top brands dataframes for all user types
# Consolidate by grouping by 'CLEANED BRAND' and 'User' and aggregating sales and price.
combined_top_brands_data = pd.concat([top_brands_all, top_brands_baby, top_brands_non_baby])

output_df = combined_top_brands_data.groupby(['CLEANED BRAND', 'User']).agg(
    {'Calculated Price/Count or Price/Bundle': 'mean',
     'CurrentPrice': 'mean', # Add CurrentPrice to the aggregation
     'Rating_Count Weight': 'sum' # Add Rating_Count Weight to the aggregation
    }
).reset_index()

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df = output_df.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Brands Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_brands_scatter_plot_data_consolidated.csv'
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

**Top 10% Rating_Count Weight Brand's Log of Current Price (Include the brands that have/don't have sales $ in Profitero (Gissel's L52W $)**

In [ ]:
# Top 10% Rating_Count Weight Brand's Log of (Current Price) ("Gissel's L52W $ might not be available for every brand/product)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common x-axis increment for price
x_increment = 0.1

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=x_increment):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Function to set y-axis ticks for log-transformed CurrentPrice
def set_y_axis_ticks_log_price(ax, df_filtered, y_label_log, base_increment=0.5):
    min_y_log = df_filtered[y_label_log].min()
    max_y_log = df_filtered[y_label_log].max()

    # Create ticks on the log scale
    y_ticks_log = np.arange(np.floor(min_y_log / base_increment) * base_increment,
                            np.ceil(max_y_log / base_increment) * base_increment + base_increment,
                            base_increment)
    ax.set_yticks(y_ticks_log)

    # Convert log ticks back to original price for labels
    # Use np.expm1 to revert np.log1p
    y_labels_actual_price = [f'${np.expm1(t):.0f}' for t in y_ticks_log]
    ax.set_yticklabels(y_labels_actual_price)

# Filter data for non-null 'Gissel's L52W $' values for all datasets
data_filtered_all = data[data["Gissel's L52W $"].notna()].copy()
baby_wipe_data_filtered = baby_wipe_data[baby_wipe_data["Gissel's L52W $"].notna()].copy()
other_wipe_data_filtered = other_wipe_data[other_wipe_data["Gissel's L52W $"].notna()].copy()

# Create CurrentPrice_log for these filtered dataframes
data_filtered_all['CurrentPrice_log'] = np.log1p(data_filtered_all['CurrentPrice'])
baby_wipe_data_filtered['CurrentPrice_log'] = np.log1p(baby_wipe_data_filtered['CurrentPrice'])
other_wipe_data_filtered['CurrentPrice_log'] = np.log1p(other_wipe_data_filtered['CurrentPrice'])

# Calculate top 10% brands by Rating_Count Weight for All Users
top_brands_all_count = max(1, int(len(data_filtered_all['CLEANED BRAND'].unique()) * 0.1))
top_brands_all = data_filtered_all[data_filtered_all['CLEANED BRAND'].isin(
    data_filtered_all.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_all_count).index
)].copy()

# Calculate top 10% brands by Rating_Count Weight for Baby Users
top_brands_baby_count = max(1, int(len(baby_wipe_data_filtered['CLEANED BRAND'].unique()) * 0.1))
top_brands_baby = baby_wipe_data_filtered[baby_wipe_data_filtered['CLEANED BRAND'].isin(
    baby_wipe_data_filtered.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_baby_count).index
)].copy()

# Calculate top 10% brands by Rating_Count Weight for Non-Baby Users
top_brands_non_baby_count = max(1, int(len(other_wipe_data_filtered['CLEANED BRAND'].unique()) * 0.1))
top_brands_non_baby = other_wipe_data_filtered[other_wipe_data_filtered['CLEANED BRAND'].isin(
    other_wipe_data_filtered.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(top_brands_non_baby_count).index
)].copy()

# Get hue order by 'Rating_Count Weight' sum for each group
brand_sales_all_order = top_brands_all.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index
brand_sales_baby_order = top_brands_baby.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index
brand_sales_non_baby_order = top_brands_non_baby.groupby('CLEANED BRAND')["Rating_Count Weight"].sum().sort_values(ascending=False).index

# Plot for ALL Users - Top 10% Brands
if not top_brands_all.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_brands_all, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='CLEANED BRAND', hue_order=brand_sales_all_order, ax=ax_all_top, legend='full')
    ax_all_top.set_title(f'Top {top_brands_all_count} Brands (All Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_all_top, top_brands_all, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_all_top, top_brands_all, 'CurrentPrice_log')
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_brands_all_count} brands (All Users) to plot.")


# Plot for BABY Users - Top 10% Brands
if not top_brands_baby.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_brands_baby, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='CLEANED BRAND', hue_order=brand_sales_baby_order, ax=ax_baby_top, legend='full')
    ax_baby_top.set_title(f'Top {top_brands_baby_count} Brands (Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_baby_top, top_brands_baby, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_baby_top, top_brands_baby, 'CurrentPrice_log')
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_brands_baby_count} brands (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Brands
if not top_brands_non_baby.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_brands_non_baby, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='CLEANED BRAND', hue_order=brand_sales_non_baby_order, ax=ax_non_baby_top, legend='full')
    ax_non_baby_top.set_title(f'Top {top_brands_non_baby_count} Brands (Non-Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_non_baby_top, top_brands_non_baby, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_non_baby_top, top_brands_non_baby, 'CurrentPrice_log')
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_brands_non_baby_count} brands (Non-Baby Users) to plot.")


# --- New code for table and CSV download --- Modify for Consolidation

# Combine the top brands dataframes for all user types
# Consolidate by grouping by 'CLEANED BRAND' and 'User' and aggregating sales and price.
combined_top_brands_data = pd.concat([top_brands_all, top_brands_baby, top_brands_non_baby])

output_df = combined_top_brands_data.groupby(['CLEANED BRAND', 'User']).agg(
    {'Calculated Price/Count or Price/Bundle': 'mean',
     'CurrentPrice': 'mean', # Add CurrentPrice to the aggregation
     'Rating_Count Weight': 'sum' # Add Rating_Count Weight to the aggregation
    }
).reset_index()

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df = output_df.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Brands Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_brands_scatter_plot_data_consolidated.csv'
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

**Top 10% Rating_Count Weight Products' Current Price (Include the products that have/don't have sales $ in Profitero (Gissel's L52W $)**

In [ ]:
# Top 10% Rating_Count Weight Products' Log of (Current Price) ("Gissel's L52W $ might not be available for every brand/product)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common x-axis increment for price
x_increment = 0.1

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=x_increment):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Function to set y-axis ticks for log-transformed CurrentPrice
def set_y_axis_ticks_log_price(ax, df_filtered, y_label_log, base_increment=0.5):
    min_y_log = df_filtered[y_label_log].min()
    max_y_log = df_filtered[y_label_log].max()

    # Create ticks on the log scale
    y_ticks_log = np.arange(np.floor(min_y_log / base_increment) * base_increment,
                            np.ceil(max_y_log / base_increment) * base_increment + base_increment,
                            base_increment)
    ax.set_yticks(y_ticks_log)

    # Convert log ticks back to original price for labels
    # Use np.expm1 to revert np.log1p
    y_labels_actual_price = [f'${np.expm1(t):.0f}' for t in y_ticks_log]
    ax.set_yticklabels(y_labels_actual_price)

# Prepare data for all products, including those with null 'Gissel's L52W $'
# And fill NaN 'Gissel's L52W $' with 0 for plotting and aggregation to represent no sales data
data_processed = data.copy()
baby_wipe_data_processed = baby_wipe_data.copy()
other_wipe_data_processed = other_wipe_data.copy()

data_processed["Gissel's L52W $"] = data_processed["Gissel's L52W $"].fillna(0)
baby_wipe_data_processed["Gissel's L52W $"] = baby_wipe_data_processed["Gissel's L52W $"].fillna(0)
other_wipe_data_processed["Gissel's L52W $"] = other_wipe_data_processed["Gissel's L52W $"].fillna(0)

# Create CurrentPrice_log for these processed dataframes
data_processed['CurrentPrice_log'] = np.log1p(data_processed['CurrentPrice'])
baby_wipe_data_processed['CurrentPrice_log'] = np.log1p(baby_wipe_data_processed['CurrentPrice'])
other_wipe_data_processed['CurrentPrice_log'] = np.log1p(other_wipe_data_processed['CurrentPrice'])

# Calculate top 10% products by Rating_Count Weight for All Users
top_products_all_count = max(1, int(len(data_processed) * 0.1))
top_products_all = data_processed.nlargest(top_products_all_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Baby Users
top_products_baby_count = max(1, int(len(baby_wipe_data_processed) * 0.1))
top_products_baby = baby_wipe_data_processed.nlargest(top_products_baby_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Non-Baby Users
top_products_non_baby_count = max(1, int(len(other_wipe_data_processed) * 0.1))
top_products_non_baby = other_wipe_data_processed.nlargest(top_products_non_baby_count, 'Rating_Count Weight').copy()

# Get hue order by 'Rating_Count Weight' sum for each group
# This is for ProductTitle, so we don't group by CLEANED BRAND but use ProductTitle directly if needed, or omit for now.

# Plot for ALL Users - Top 10% Products
if not top_products_all.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_products_all, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='ProductTitle', ax=ax_all_top, legend='full')
    ax_all_top.set_title(f'Top {top_products_all_count} Products (All Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_all_top, top_products_all, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_all_top, top_products_all, 'CurrentPrice_log')
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_all_count} products (All Users) to plot.")


# Plot for BABY Users - Top 10% Products
if not top_products_baby.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_products_baby, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='ProductTitle', ax=ax_baby_top, legend='full')
    ax_baby_top.set_title(f'Top {top_products_baby_count} Products (Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_baby_top, top_products_baby, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_baby_top, top_products_baby, 'CurrentPrice_log')
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_baby_count} products (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Products
if not top_products_non_baby.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_products_non_baby, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='ProductTitle', ax=ax_non_baby_top, legend='full')
    ax_non_baby_top.set_title(f'Top {top_products_non_baby_count} Products (Non-Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_non_baby_top, top_products_non_baby, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_non_baby_top, top_products_non_baby, 'CurrentPrice_log')
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_non_baby_count} products (Non-Baby Users) to plot.")


# --- New code for table and CSV download --- Modify for Consolidation

# Combine the top products dataframes for all user types
# Consolidate by grouping by 'CLEANED BRAND' and 'User' and aggregating sales and price.
combined_top_products_data = pd.concat([top_products_all, top_products_baby, top_products_non_baby])

output_df = combined_top_products_data.groupby(['ProductTitle', 'User']).agg(
    {
        'CLEANED BRAND': 'first', # To keep the brand information
        'Calculated Price/Count or Price/Bundle': 'mean',
        'CurrentPrice': 'mean',
        'Rating_Count Weight': 'sum',
        "Gissel's L52W $": 'sum'
    }
).reset_index()

# Reorder columns for better readability if needed, e.g., 'CLEANED BRAND' first
output_df = output_df[['CLEANED BRAND', 'ProductTitle', 'User', 'Calculated Price/Count or Price/Bundle', 'CurrentPrice', 'Rating_Count Weight', "Gissel's L52W $"]]

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df = output_df.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Products Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_products_scatter_plot_data_consolidated.csv' # Changed filename to reflect products
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

**Top 10% Rating_Count Weight Products' Log of Current Price (Only includes the products that have sales $ in Profitero (Gissel's L52W $)**

In [ ]:
# Top 10% Rating_Count Weight Products' Log of (Current Price) ("Gissel's L52W $ might not be available for every brand/product)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
from google.colab import files # For download

# Define a common x-axis increment for price
x_increment = 0.1

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=x_increment):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Function to set y-axis ticks for log-transformed CurrentPrice
def set_y_axis_ticks_log_price(ax, df_filtered, y_label_log, base_increment=0.5):
    min_y_log = df_filtered[y_label_log].min()
    max_y_log = df_filtered[y_label_log].max()

    # Create ticks on the log scale
    y_ticks_log = np.arange(np.floor(min_y_log / base_increment) * base_increment,
                            np.ceil(max_y_log / base_increment) * base_increment + base_increment,
                            base_increment)
    ax.set_yticks(y_ticks_log)

    # Convert log ticks back to original price for labels
    # Use np.expm1 to revert np.log1p
    y_labels_actual_price = [f'${np.expm1(t):.0f}' for t in y_ticks_log]
    ax.set_yticklabels(y_labels_actual_price)

# Filter data for non-null 'Gissel's L52W $' values for all datasets
data_processed = data[data["Gissel's L52W $"].notna()].copy()
baby_wipe_data_processed = baby_wipe_data[baby_wipe_data["Gissel's L52W $"].notna()].copy()
other_wipe_data_processed = other_wipe_data[other_wipe_data["Gissel's L52W $"].notna()].copy()

# Create CurrentPrice_log for these processed dataframes
data_processed['CurrentPrice_log'] = np.log1p(data_processed['CurrentPrice'])
baby_wipe_data_processed['CurrentPrice_log'] = np.log1p(baby_wipe_data_processed['CurrentPrice'])
other_wipe_data_processed['CurrentPrice_log'] = np.log1p(other_wipe_data_processed['CurrentPrice'])

# Calculate top 10% products by Rating_Count Weight for All Users
top_products_all_count = max(1, int(len(data_processed) * 0.1))
top_products_all = data_processed.nlargest(top_products_all_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Baby Users
top_products_baby_count = max(1, int(len(baby_wipe_data_processed) * 0.1))
top_products_baby = baby_wipe_data_processed.nlargest(top_products_baby_count, 'Rating_Count Weight').copy()

# Calculate top 10% products by Rating_Count Weight for Non-Baby Users
top_products_non_baby_count = max(1, int(len(other_wipe_data_processed) * 0.1))
top_products_non_baby = other_wipe_data_processed.nlargest(top_products_non_baby_count, 'Rating_Count Weight').copy()

# Get hue order by 'Rating_Count Weight' sum for each group
# This is for ProductTitle, so we don't group by CLEANED BRAND but use ProductTitle directly if needed, or omit for now.

# Plot for ALL Users - Top 10% Products
if not top_products_all.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_products_all, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='ProductTitle', ax=ax_all_top, legend='full')
    ax_all_top.set_title(f'Top {top_products_all_count} Products (All Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_all_top, top_products_all, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_all_top, top_products_all, 'CurrentPrice_log')
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_all_count} products (All Users) to plot.")


# Plot for BABY Users - Top 10% Products
if not top_products_baby.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_products_baby, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='ProductTitle', ax=ax_baby_top, legend='full')
    ax_baby_top.set_title(f'Top {top_products_baby_count} Products (Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_baby_top, top_products_baby, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_baby_top, top_products_baby, 'CurrentPrice_log')
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_baby_count} products (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 10% Products
if not top_products_non_baby.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(15, 10)) # Increased width for legend
    sns.scatterplot(data=top_products_non_baby, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice_log", hue='ProductTitle', ax=ax_non_baby_top, legend='full')
    ax_non_baby_top.set_title(f'Top {top_products_non_baby_count} Products (Non-Baby Users) by Rating_Count Weight\n Calculated Price/Count or Price/Bundle vs. Log of Current Price')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Log of Current Price ($)')
    set_x_axis_ticks(ax_non_baby_top, top_products_non_baby, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_log_price(ax_non_baby_top, top_products_non_baby, 'CurrentPrice_log')
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='ProductTitle') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top {top_products_non_baby_count} products (Non-Baby Users) to plot.")


# --- New code for table and CSV download --- Modify for Consolidation

# Combine the top products dataframes for all user types
# Consolidate by grouping by 'CLEANED BRAND' and 'User' and aggregating sales and price.
combined_top_products_data = pd.concat([top_products_all, top_products_baby, top_products_non_baby])

output_df = combined_top_products_data.groupby(['ProductTitle', 'User']).agg(
    {
        'CLEANED BRAND': 'first', # To keep the brand information
        'Calculated Price/Count or Price/Bundle': 'mean',
        'CurrentPrice': 'mean',
        'Rating_Count Weight': 'sum',
        "Gissel's L52W $": 'sum'
    }
).reset_index()

# Reorder columns for better readability if needed, e.g., 'CLEANED BRAND' first
output_df = output_df[['CLEANED BRAND', 'ProductTitle', 'User', 'Calculated Price/Count or Price/Bundle', 'CurrentPrice', 'Rating_Count Weight', "Gissel's L52W $"]]

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df = output_df.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top Products Data ---")
display(output_df.head())

# Save the DataFrame to a CSV file
csv_filename = 'top_products_scatter_plot_data_consolidated.csv' # Changed filename to reflect products
output_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

## *** ------------------------------------ Need to Change -------------------*** ##

## **2. Univariate analysis - Categorical Data**

## **2. Attributes vs. L52W$**

In [ ]:
# The perc_on_bar function is designed to create a bar plot for a categorical column. In this specific case, it's analyzing the 'Attribute' column in your data DataFrame. The plot will show the distribution of each unique value in the 'Attribute' column as bars, with the percentage of each category displayed on top of its respective bar.
import matplotlib.pyplot as plt
import seaborn as sns

# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z, title_suffix=""): # Added title_suffix parameter
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(25, 5)) # Increased width
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.xticks(rotation=45, ha='right')
    plt.title(f"Percentage Distribution of '{z}' ({title_suffix})") # Added title
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
perc_on_bar(data, 'Attribute', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
perc_on_bar(baby_wipe_data, 'Attribute', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
perc_on_bar(other_wipe_data, 'Attribute', 'Non-Baby Users')

In [ ]:
# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z):
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(15, 5))
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.show()

def plot_attribute_gissel_sales(df, title_suffix):
    """
    Generates a boxplot to visualize the distribution of 'Gissel's L52W $'
    across different 'Attribute' categories.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """

    # Filter data to include only non-null 'Gissel's L52W $' values
    filtered_df = df[df["Gissel's L52W $"].notna()].copy()

    # Replace 'undefined' in 'Attribute' with NaN for cleaner plotting if it's treated as a missing value
    filtered_df['Attribute'] = filtered_df['Attribute'].replace('undefined', np.nan)
    filtered_df.dropna(subset=['Attribute'], inplace=True)

    if filtered_df.empty:
        print(f"No data with non-null 'Gissel's L52W $' and defined 'Attribute' for {title_suffix}")
        return

    plt.figure(figsize=(14, 8))
    sns.boxplot(data=filtered_df, x='Attribute', y="Gissel's L52W $", palette='viridis')
    plt.title(f"Gissel's L52W $ by Attribute ({title_suffix})")
    plt.xlabel('Attribute')
    plt.ylabel("Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
plot_attribute_gissel_sales(data, 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
plot_attribute_gissel_sales(baby_wipe_data, 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
plot_attribute_gissel_sales(other_wipe_data, 'Non-Baby Users')

## **2. Benefits vs. L52W$**

In [ ]:
# The perc_on_bar function is designed to create a bar plot for a categorical column. In this specific case, it's analyzing the 'Attribute' column in your data DataFrame. The plot will show the distribution of each unique value in the 'Attribute' column as bars, with the percentage of each category displayed on top of its respective bar.
import matplotlib.pyplot as plt
import seaborn as sns

# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z, title_suffix=""): # Added title_suffix parameter
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(25, 5)) # Increased width
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.xticks(rotation=45, ha='right')
    plt.title(f"Percentage Distribution of '{z}' ({title_suffix})") # Added title
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
perc_on_bar(data, 'Benefits', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
perc_on_bar(baby_wipe_data, 'Benefits', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
perc_on_bar(other_wipe_data, 'Benefits', 'Non-Baby Users')

In [ ]:
# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z):
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(15, 5))
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.show()

def plot_gissel_sales_by_category(df, category_col, title_suffix):
    """
    Generates a boxplot to visualize the distribution of 'Gissel's L52W $'
    across different categories in a specified column.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        category_col (str): The name of the column containing categorical data (e.g., 'Benefits', 'Attribute').
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """

    # Filter data to include only non-null 'Gissel's L52W $' values
    filtered_df = df[df["Gissel's L52W $"].notna()].copy()

    # Replace 'undefined' in the specified category column with NaN for cleaner plotting
    filtered_df[category_col] = filtered_df[category_col].replace('undefined', np.nan)
    filtered_df.dropna(subset=[category_col], inplace=True)

    if filtered_df.empty:
        print(f"No data with non-null 'Gissel's L52W $' and defined '{category_col}' for {title_suffix}")
        return

    plt.figure(figsize=(14, 8))
    sns.boxplot(data=filtered_df, x=category_col, y="Gissel's L52W $", palette='viridis')
    plt.title(f"Gissel's L52W $ by {category_col} ({title_suffix})")
    plt.xlabel(category_col)
    plt.ylabel("Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
plot_gissel_sales_by_category(data, 'Benefits', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
plot_gissel_sales_by_category(baby_wipe_data, 'Benefits', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
plot_gissel_sales_by_category(other_wipe_data, 'Benefits', 'Non-Baby Users')

## **2. Scent_ingredient vs. L52W$**

In [ ]:
# The perc_on_bar function is designed to create a bar plot for a categorical column. In this specific case, it's analyzing the 'Attribute' column in your data DataFrame. The plot will show the distribution of each unique value in the 'Attribute' column as bars, with the percentage of each category displayed on top of its respective bar.
import matplotlib.pyplot as plt
import seaborn as sns

# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z, title_suffix=""): # Added title_suffix parameter
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(25, 5)) # Increased width
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.xticks(rotation=45, ha='right')
    plt.title(f"Percentage Distribution of '{z}' ({title_suffix})") # Added title
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
perc_on_bar(data, 'Scent_ingredient', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
perc_on_bar(baby_wipe_data, 'Scent_ingredient', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
perc_on_bar(other_wipe_data, 'Scent_ingredient', 'Non-Baby Users')

**Use the top 50% of the Scent_ingredient**

In [ ]:
# The perc_on_bar function is designed to create a bar plot for a categorical column. In this specific case, it's analyzing the 'Scent_ingredient' column in your data DataFrame. The plot will show the distribution of each unique value in the top 25 most frequent 'Scent_ingredient' categories as bars, with the percentage of each category displayed on top of its respective bar (relative to the full dataset).
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z, title_suffix=""):
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    # Calculate value counts and get the top 25 most frequent categories
    # Exclude 'undefined' from the top categories unless it's explicitly desired
    category_counts = data[z].value_counts()

    # Filter out 'undefined' if it's the top category and we want to show meaningful scents
    if 'undefined' in category_counts.index:
        # Option 1: Exclude 'undefined' from the top N list
        filtered_counts = category_counts[category_counts.index != 'undefined']
        top_categories = filtered_counts.head(25).index.tolist()
        # If after removing 'undefined', there are fewer than 25, add 'undefined' back if it was significant
        if len(top_categories) < 25 and 'undefined' in category_counts:
            top_categories = category_counts.head(25).index.tolist()
    else:
        top_categories = category_counts.head(25).index.tolist()

    if not top_categories:
        print(f"No categories to plot for '{z}' in '{title_suffix}' after filtering.")
        return

    total = len(data[z])  # Total length of the original column for accurate percentage calculation

    # Filter data to include only the selected top categories for plotting
    plot_data = data[data[z].isin(top_categories)].copy()

    plt.figure(figsize=(25, 5)) # Increased width
    ax = sns.countplot(data=plot_data, x=z, palette='Paired', order=top_categories)

    for p in ax.patches:
        # Percentage relative to the total length of the original column (all data points)
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.xticks(rotation=45, ha='right')
    plt.title(f"Percentage Distribution of Top {len(top_categories)} '{z}' ({title_suffix})") # Corrected title
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
perc_on_bar(data, 'Scent_ingredient', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
perc_on_bar(baby_wipe_data, 'Scent_ingredient', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
perc_on_bar(other_wipe_data, 'Scent_ingredient', 'Non-Baby Users')

In [ ]:
# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z):
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(15, 5))
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.show()

def plot_attribute_gissel_sales(df, title_suffix):
    """
    Generates a boxplot to visualize the distribution of 'Gissel's L52W $'
    across different 'Scent_ingredient' categories.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """

    # Filter data to include only non-null 'Gissel's L52W $' values
    filtered_df = df[df["Gissel's L52W $"].notna()].copy()

    # Get value counts for 'Scent_ingredient' within the filtered DataFrame
    scent_counts = filtered_df['Scent_ingredient'].value_counts()

    # Separate 'undefined' from other scent ingredients for prioritization
    meaningful_scents = scent_counts[scent_counts.index != 'undefined']
    undefined_present = 'undefined' in scent_counts.index

    top_n = 25
    top_categories = meaningful_scents.head(top_n).index.tolist()

    # If there are fewer than top_n meaningful categories, add 'undefined' if it exists to reach the limit
    if len(top_categories) < top_n and undefined_present:
        if 'undefined' not in top_categories: # Avoid adding if somehow already present
            top_categories.append('undefined')

    if not top_categories:
        print(f"No 'Scent_ingredient' categories to plot for {title_suffix} after filtering.")
        return

    # Filter data to include only the selected top categories for plotting
    plot_df = filtered_df[filtered_df['Scent_ingredient'].isin(top_categories)].copy()

    if plot_df.empty:
        print(f"No data for top 'Scent_ingredient' categories to plot for {title_suffix}.")
        return

    plt.figure(figsize=(18, 8)) # Increased figure width for better label visibility
    sns.boxplot(data=plot_df, x='Scent_ingredient', y="Gissel's L52W $", palette='viridis', order=top_categories)
    plt.title(f"Gissel's L52W $ by Top {len(top_categories)} Scent Ingredients ({title_suffix})")
    plt.xlabel('Scent Ingredient')
    plt.ylabel("Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
plot_attribute_gissel_sales(data, 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
plot_attribute_gissel_sales(baby_wipe_data, 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
plot_attribute_gissel_sales(other_wipe_data, 'Non-Baby Users')

**Free_From vs. L52W$**

In [ ]:
# The perc_on_bar function is designed to create a bar plot for a categorical column. In this specific case, it's analyzing the 'Attribute' column in your data DataFrame. The plot will show the distribution of each unique value in the 'Attribute' column as bars, with the percentage of each category displayed on top of its respective bar.
import matplotlib.pyplot as plt
import seaborn as sns

# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z, title_suffix=""): # Added title_suffix parameter
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(25, 5)) # Increased width
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.xticks(rotation=45, ha='right')
    plt.title(f"Percentage Distribution of '{z}' ({title_suffix})") # Added title
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
perc_on_bar(data, 'Free_From', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
perc_on_bar(baby_wipe_data, 'Free_From', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
perc_on_bar(other_wipe_data, 'Free_From', 'Non-Baby Users')

In [ ]:
# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z):
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(15, 5))
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.show()

def plot_attribute_gissel_sales(df, title_suffix):
    """
    Generates a boxplot to visualize the distribution of 'Gissel's L52W $'
    across different 'Free_From' categories.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """

    # Filter data to include only non-null 'Gissel's L52W $' values
    filtered_df = df[df["Gissel's L52W $"].notna()].copy()

    # Replace 'undefined' in 'Free_From' with NaN for cleaner plotting if it's treated as a missing value
    filtered_df['Free_From'] = filtered_df['Free_From'].replace('undefined', np.nan)
    filtered_df.dropna(subset=['Free_From'], inplace=True)

    if filtered_df.empty:
        print(f"No data with non-null 'Gissel's L52W $' and defined 'Free_From' for {title_suffix}")
        return

    plt.figure(figsize=(14, 8))
    sns.boxplot(data=filtered_df, x='Free_From', y="Gissel's L52W $", palette='viridis')
    plt.title(f"Gissel's L52W $ by Free_From ({title_suffix})")
    plt.xlabel('Free_From')
    plt.ylabel("Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
plot_attribute_gissel_sales(data, 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
plot_attribute_gissel_sales(baby_wipe_data, 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
plot_attribute_gissel_sales(other_wipe_data, 'Non-Baby Users')

**Feature vs L52W$**

In [ ]:
# The perc_on_bar function is designed to create a bar plot for a categorical column. In this specific case, it's analyzing the 'Attribute' column in your data DataFrame. The plot will show the distribution of each unique value in the 'Attribute' column as bars, with the percentage of each category displayed on top of its respective bar.
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z, title_suffix=""): # Added title_suffix parameter
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    # Calculate value counts and get the top 25 most frequent categories
    # Exclude 'undefined' from the top categories unless it's explicitly desired
    category_counts = data[z].value_counts()

    # Filter out 'undefined' if it's the top category and we want to show meaningful features
    if 'undefined' in category_counts.index:
        filtered_counts = category_counts[category_counts.index != 'undefined']
        top_categories = filtered_counts.head(25).index.tolist()
        # If after removing 'undefined', there are fewer than 25, add 'undefined' back if it was significant
        if len(top_categories) < 25 and 'undefined' in category_counts:
            top_categories = category_counts.head(25).index.tolist()
    else:
        top_categories = category_counts.head(25).index.tolist()

    if not top_categories:
        print(f"No categories to plot for '{z}' in '{title_suffix}' after filtering.")
        return

    total = len(data[z])  # Total length of the original column for accurate percentage calculation

    # Filter data to include only the selected top categories for plotting
    plot_data = data[data[z].isin(top_categories)].copy()

    plt.figure(figsize=(25, 5)) # Increased width
    ax = sns.countplot(data=plot_data, x=z, palette='Paired', order=top_categories)

    for p in ax.patches:
        # Percentage relative to the total length of the original column (all data points)
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.xticks(rotation=45, ha='right')
    plt.title(f"Percentage Distribution of Top {len(top_categories)} '{z}' ({title_suffix})") # Corrected title
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
perc_on_bar(data, 'Features', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
perc_on_bar(baby_wipe_data, 'Features', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
perc_on_bar(other_wipe_data, 'Features', 'Non-Baby Users')

In [ ]:
# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z):
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(15, 5))
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.show()

def plot_attribute_gissel_sales(df, title_suffix):
    """
    Generates a boxplot to visualize the distribution of 'Gissel's L52W $'
    across different 'Features' categories.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """

    # Filter data to include only non-null 'Gissel's L52W $' values
    filtered_df = df[df["Gissel's L52W $"].notna()].copy()

    # Replace 'undefined' in 'Features' with NaN for cleaner plotting if it's treated as a missing value
    filtered_df['Features'] = filtered_df['Features'].replace('undefined', np.nan)
    filtered_df.dropna(subset=['Features'], inplace=True)

    if filtered_df.empty:
        print(f"No data with non-null 'Gissel's L52W $' and defined 'Features' for {title_suffix}")
        return

    plt.figure(figsize=(14, 8))
    sns.boxplot(data=filtered_df, x='Features', y="Gissel's L52W $", palette='viridis')
    plt.title(f"Gissel's L52W $ by Features ({title_suffix})")
    plt.xlabel('Features')
    plt.ylabel("Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
plot_attribute_gissel_sales(data, 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
plot_attribute_gissel_sales(baby_wipe_data, 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
plot_attribute_gissel_sales(other_wipe_data, 'Non-Baby Users')

**Skin_Type vs. L52W$**

In [ ]:
# The perc_on_bar function is designed to create a bar plot for a categorical column. In this specific case, it's analyzing the 'Attribute' column in your data DataFrame. The plot will show the distribution of each unique value in the 'Attribute' column as bars, with the percentage of each category displayed on top of its respective bar.
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z, title_suffix=""): # Added title_suffix parameter
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    # Calculate value counts and get the top 25 most frequent categories
    # Exclude 'undefined' from the top categories unless it's explicitly desired
    category_counts = data[z].value_counts()

    # Filter out 'undefined' if it's the top category and we want to show meaningful features
    if 'undefined' in category_counts.index:
        filtered_counts = category_counts[category_counts.index != 'undefined']
        top_categories = filtered_counts.head(25).index.tolist()
        # If after removing 'undefined', there are fewer than 25, add 'undefined' back if it was significant
        if len(top_categories) < 25 and 'undefined' in category_counts:
            top_categories = category_counts.head(25).index.tolist()
    else:
        top_categories = category_counts.head(25).index.tolist()

    if not top_categories:
        print(f"No categories to plot for '{z}' in '{title_suffix}' after filtering.")
        return

    total = len(data[z])  # Total length of the original column for accurate percentage calculation

    # Filter data to include only the selected top categories for plotting
    plot_data = data[data[z].isin(top_categories)].copy()

    plt.figure(figsize=(25, 5)) # Increased width
    ax = sns.countplot(data=plot_data, x=z, palette='Paired', order=top_categories)

    for p in ax.patches:
        # Percentage relative to the total length of the original column (all data points)
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.xticks(rotation=45, ha='right')
    plt.title(f"Percentage Distribution of Top {len(top_categories)} '{z}' ({title_suffix})") # Corrected title
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
perc_on_bar(data, 'Skin_Type', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
perc_on_bar(baby_wipe_data, 'Skin_Type', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
perc_on_bar(other_wipe_data, 'Skin_Type', 'Non-Baby Users')

In [ ]:
# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z):
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(15, 5))
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.show()

def plot_attribute_gissel_sales(df, title_suffix):
    """
    Generates a boxplot to visualize the distribution of 'Gissel's L52W $'
    across different 'Skin_Type' categories.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """

    # Filter data to include only non-null 'Gissel's L52W $' values
    filtered_df = df[df["Gissel's L52W $"].notna()].copy()

    # Replace 'undefined' in 'Skin_Type' with NaN for cleaner plotting if it's treated as a missing value
    filtered_df['Skin_Type'] = filtered_df['Skin_Type'].replace('undefined', np.nan)
    filtered_df.dropna(subset=['Skin_Type'], inplace=True)

    if filtered_df.empty:
        print(f"No data with non-null 'Gissel's L52W $' and defined 'Skin_Type' for {title_suffix}")
        return

    plt.figure(figsize=(14, 8))
    sns.boxplot(data=filtered_df, x='Skin_Type', y="Gissel's L52W $", palette='viridis')
    plt.title(f"Gissel's L52W $ by Skin_Type ({title_suffix})")
    plt.xlabel('Skin_Type')
    plt.ylabel("Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
plot_attribute_gissel_sales(data, 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
plot_attribute_gissel_sales(baby_wipe_data, 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
plot_attribute_gissel_sales(other_wipe_data, 'Non-Baby Users')

**Container vs L52W$**

In [ ]:
# The perc_on_bar function is designed to create a bar plot for a categorical column.
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z, title_suffix=""): # Added title_suffix parameter
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    # Calculate value counts and get the top 25 most frequent categories
    # Exclude 'undefined' from the top categories unless it's explicitly desired
    category_counts = data[z].value_counts()

    # Filter out 'undefined' if it's the top category and we want to show meaningful features
    if 'undefined' in category_counts.index:
        filtered_counts = category_counts[category_counts.index != 'undefined']
        top_categories = filtered_counts.head(25).index.tolist()
        # If after removing 'undefined', there are fewer than 25, add 'undefined' back if it was significant
        if len(top_categories) < 25 and 'undefined' in category_counts:
            top_categories = category_counts.head(25).index.tolist()
    else:
        top_categories = category_counts.head(25).index.tolist()

    if not top_categories:
        print(f"No categories to plot for '{z}' in '{title_suffix}' after filtering.")
        return

    total = len(data[z])  # Total length of the original column for accurate percentage calculation

    # Filter data to include only the selected top categories for plotting
    plot_data = data[data[z].isin(top_categories)].copy()

    plt.figure(figsize=(25, 5)) # Increased width
    ax = sns.countplot(data=plot_data, x=z, palette='Paired', order=top_categories)

    for p in ax.patches:
        # Percentage relative to the total length of the original column (all data points)
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.xticks(rotation=45, ha='right')
    plt.title(f"Percentage Distribution of Top {len(top_categories)} '{z}' ({title_suffix})") # Corrected title
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
perc_on_bar(data, 'Container', 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
perc_on_bar(baby_wipe_data, 'Container', 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
perc_on_bar(other_wipe_data, 'Container', 'Non-Baby Users')

In [ ]:
# Let us write a function that will help us create barplots that indicate the percentage for each category.
# This function takes the categorical column as the input and returns the barplots for the variable.

def perc_on_bar(data, z):
    '''
    plot
    data: DataFrame or Series
    z: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(data[z])  # Length of the column
    plt.figure(figsize=(15, 5))
    ax = sns.countplot(data=data, x=z, palette='Paired', order=data[z].value_counts().index)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)  # Percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05  # Width of the plot
        y = p.get_y() + p.get_height()  # Height of the plot

        ax.annotate(percentage, (x, y), size=12, ha='center')  # Annotate the percentage with center alignment

    plt.show()

def plot_attribute_gissel_sales(df, title_suffix):
    """
    Generates a boxplot to visualize the distribution of 'Gissel's L52W $'
    across different 'Container' categories.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """

    # Filter data to include only non-null 'Gissel's L52W $' values
    filtered_df = df[df["Gissel's L52W $"].notna()].copy()

    # Replace 'undefined' in 'Container' with NaN for cleaner plotting if it's treated as a missing value
    filtered_df['Container'] = filtered_df['Container'].replace('undefined', np.nan)
    filtered_df.dropna(subset=['Container'], inplace=True)

    if filtered_df.empty:
        print(f"No data with non-null 'Gissel's L52W $' and defined 'Container' for {title_suffix}")
        return

    plt.figure(figsize=(14, 8))
    sns.boxplot(data=filtered_df, x='Container', y="Gissel's L52W $", palette='viridis')
    plt.title(f"Gissel's L52W $ by Container ({title_suffix})")
    plt.xlabel('Container')
    plt.ylabel("Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Call the function for All Users
print("\n--- Analysis for All Users ---")
plot_attribute_gissel_sales(data, 'All Users')

# Call the function for Baby Users
print("\n--- Analysis for Baby Users ---")
plot_attribute_gissel_sales(baby_wipe_data, 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Analysis for Non-Baby Users ---")
plot_attribute_gissel_sales(other_wipe_data, 'Non-Baby Users')

# ---------------------------**Need to change------------------------------------**

# ---------------------------**Top 20 Baby/Non-baby Wipes------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from google.colab import files # For download

# Function to set x-axis ticks for linear scale price
def set_x_axis_ticks(ax, df_filtered, x_label, base_increment=0.1):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    # Ensure ticks are well-spaced and cover the range
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.1f}' for t in x_ticks], rotation=45, ha='right')

# Function to set y-axis ticks for linear scale CurrentPrice
def set_y_axis_ticks_current_price(ax, df_filtered, y_label, base_increment=10): # Adjusted base_increment for CurrentPrice
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    # Ensure ticks are well-spaced and cover the range
    y_ticks = np.arange(np.floor(min_y / base_increment) * base_increment,
                        np.ceil(max_y / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f'${t:.0f}' for t in y_ticks]) # Format as whole dollars

# Prepare dataframes and fill NaN for 'Gissel\'s L52W $' with 0 for consistent aggregation later
data_processed = data.copy()
baby_wipe_data_processed = baby_wipe_data.copy()
other_wipe_data_processed = other_wipe_data.copy()

data_processed["Gissel's L52W $"] = data_processed["Gissel's L52W $"].fillna(0)
baby_wipe_data_processed["Gissel's L52W $"] = baby_wipe_data_processed["Gissel's L52W $"].fillna(0)
other_wipe_data_processed["Gissel's L52W $"] = other_wipe_data_processed["Gissel's L52W $"].fillna(0)


# --- Identify Top 20 Brands by Rating_Count Weight for each user segment ---

# All Users
top_brands_all_names = data_processed.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(20).index.tolist()
top_20_brands_all_products = data_processed[data_processed['CLEANED BRAND'].isin(top_brands_all_names)].copy()

# Baby Users
top_brands_baby_names = baby_wipe_data_processed.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(20).index.tolist()
top_20_brands_baby_products = baby_wipe_data_processed[baby_wipe_data_processed['CLEANED BRAND'].isin(top_brands_baby_names)].copy()

# Non-Baby Users
top_brands_non_baby_names = other_wipe_data_processed.groupby('CLEANED BRAND')['Rating_Count Weight'].sum().nlargest(20).index.tolist()
top_20_brands_non_baby_products = other_wipe_data_processed[other_wipe_data_processed['CLEANED BRAND'].isin(top_brands_non_baby_names)].copy()


# --- Generate Scatter Plots for individual products within top 20 brands ---

# Plot for ALL Users - Top 20 Brands
if not top_20_brands_all_products.empty:
    fig_all_top, ax_all_top = plt.subplots(figsize=(15, 10))
    sns.scatterplot(data=top_20_brands_all_products, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice", hue='CLEANED BRAND', ax=ax_all_top, legend='full', s=50, alpha=0.7) # Each point is a product
    ax_all_top.set_title(f'Top {len(top_brands_all_names)} Brands (All Users) by Rating_Count Weight\n Individual Products: Price/Count vs. Current Price')
    ax_all_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_all_top.set_ylabel('Current Price ($)')
    set_x_axis_ticks(ax_all_top, top_20_brands_all_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_current_price(ax_all_top, top_20_brands_all_products, 'CurrentPrice')
    ax_all_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND', fontsize='small') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top brands (All Users) to plot.")


# Plot for BABY Users - Top 20 Brands
if not top_20_brands_baby_products.empty:
    fig_baby_top, ax_baby_top = plt.subplots(figsize=(15, 10))
    sns.scatterplot(data=top_20_brands_baby_products, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice", hue='CLEANED BRAND', ax=ax_baby_top, legend='full', s=50, alpha=0.7) # Each point is a product
    ax_baby_top.set_title(f'Top {len(top_brands_baby_names)} Brands (Baby Users) by Rating_Count Weight\n Individual Products: Price/Count vs. Current Price')
    ax_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_baby_top.set_ylabel('Current Price ($)')
    set_x_axis_ticks(ax_baby_top, top_20_brands_baby_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_current_price(ax_baby_top, top_20_brands_baby_products, 'CurrentPrice')
    ax_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND', fontsize='small') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top brands (Baby Users) to plot.")

# Plot for NON-BABY Users - Top 20 Brands
if not top_20_brands_non_baby_products.empty:
    fig_non_baby_top, ax_non_baby_top = plt.subplots(figsize=(15, 10))
    sns.scatterplot(data=top_20_brands_non_baby_products, x='Calculated Price/Count or Price/Bundle', y="CurrentPrice", hue='CLEANED BRAND', ax=ax_non_baby_top, legend='full', s=50, alpha=0.7) # Each point is a product
    ax_non_baby_top.set_title(f'Top {len(top_brands_non_baby_names)} Brands (Non-Baby Users) by Rating_Count Weight\n Individual Products: Price/Count vs. Current Price')
    ax_non_baby_top.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax_non_baby_top.set_ylabel('Current Price ($)')
    set_x_axis_ticks(ax_non_baby_top, top_20_brands_non_baby_products, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_current_price(ax_non_baby_top, top_20_brands_non_baby_products, 'CurrentPrice')
    ax_non_baby_top.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='CLEANED BRAND', fontsize='small') # Place legend outside
    plt.tight_layout()
    plt.show()
else:
    print(f"No data for top brands (Non-Baby Users) to plot.")


# --- Create and Download Consolidated CSV ---

# Combine the data for the top brands across all user types for the downloadable CSV
# Keep all relevant columns for individual products
combined_top_products_data_for_csv = pd.concat([
    top_20_brands_all_products[['CLEANED BRAND', 'ProductTitle', 'User', 'Calculated Price/Count or Price/Bundle', 'CurrentPrice', 'Rating_Count Weight', "Gissel's L52W $"]],
    top_20_brands_baby_products[['CLEANED BRAND', 'ProductTitle', 'User', 'Calculated Price/Count or Price/Bundle', 'CurrentPrice', 'Rating_Count Weight', "Gissel's L52W $"]],
    top_20_brands_non_baby_products[['CLEANED BRAND', 'ProductTitle', 'User', 'Calculated Price/Count or Price/Bundle', 'CurrentPrice', 'Rating_Count Weight', "Gissel's L52W $"]]
]).drop_duplicates().reset_index(drop=True) # Drop duplicates in case a product is in multiple "top brand" lists (e.g. from the general 'data' and specific 'baby_wipe_data' if it's a baby product).

# Sort the output DataFrame by 'Rating_Count Weight' in descending order
output_df_final = combined_top_products_data_for_csv.sort_values(by='Rating_Count Weight', ascending=False).reset_index(drop=True)

# Display the head of the DataFrame as a table
print("\n--- Consolidated Top 20 Brands Data (Individual Products) ---")
display(output_df_final.head())

# Save the DataFrame to a CSV file
csv_filename_final = 'top_20_brands_individual_products_data.csv'
output_df_final.to_csv(csv_filename_final, index=False)

# Provide a download link
files.download(csv_filename_final)
print(f"\n'{csv_filename_final}' has been generated and is ready for download.")

# ---------------------------**Baby User - Honest Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'HONEST' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'HONEST'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=1000000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.0f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'HONEST' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 12)) # Increased figure height to 12
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("HONEST Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, Attribute, Benefits, Scent_ingredients, and Rating)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'HONEST' brand in 'Baby Users' segment.")

print("\n--- Head of 'HONEST' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Pampers Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'PAMPERS' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'PAMPERS'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'PAMPERS' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("PAMPERS Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'PAMPERS' brand in 'Baby Users' segment.")

print("\n--- Head of 'PAMPERS' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Private Label Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    print("Expected path: /content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
    # Exit or handle the error gracefully, for now, we'll just print and stop further processing in this block.
    raise

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'PRIVATE LABEL' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'PRIVATE LABEL'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'PRIVATE LABEL' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("PRIVATE LABEL Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'PRIVATE LABEL' brand in 'Baby Users' segment.")

print("\n--- Head of 'PRIVATE LABEL' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Boogie Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'BOOGIE' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'BOOGIE'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'BOOGIE' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("BOOGIE Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'BOOGIE' brand in 'Baby Users' segment.")

print("\n--- Head of 'BOOGIE' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Huggies Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'HUGGIES' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'HUGGIES'].copy()

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=250000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.2f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'HUGGIES' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 28)) # Increased figure height to 28
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("HUGGIES Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'HUGGIES' brand in 'Baby Users' segment.")

print("\n--- Head of 'HUGGIES' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Aveeno Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'AVEENO' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'AVEENO'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'AVEENO' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("AVEENO Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'AVEENO' brand in 'Baby Users' segment.")

print("\n--- Head of 'AVEENO' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Hello Bello Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'HELLO BELLO' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'HELLO BELLO'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'HELLO BELLO' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("HELLO BELLO Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'HELLO BELLO' brand in 'Baby Users' segment.")

print("\n--- Head of 'HELLO BELLO' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Waterwipes Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'WATERWIPES' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'WATERWIPES'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'WATERWIPES' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("WATERWIPES Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'WATERWIPES' brand in 'Baby Users' segment.")

print("\n--- Head of 'WATERWIPES' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Coterie Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'COTERIE' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'COTERIE'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'COTERIE' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("COTERIE Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'COTERIE' brand in 'Baby Users' segment.")

print(
"--- Head of 'COTERIE' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Naty Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'NATY' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'NATY'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'NATY' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("NATY Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'NATY' brand in 'Baby Users' segment.")

print("\n--- Head of 'NATY' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Babyganics Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'BABYGANICS' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'BABYGANICS'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'BABYGANICS' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("BABYGANICS Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'BABYGANICS' brand in 'Baby Users' segment.")

print("\n--- Head of 'BABYGANICS' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ------------------**Baby User - Seventh Generation Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'SEVENTH GENERATION' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'SEVENTH GENERATION'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'SEVENTH GENERATION' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("SEVENTH GENERATION Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'SEVENTH GENERATION' brand in 'Baby Users' segment.")

print("\n--- Head of 'SEVENTH GENERATION' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Kandoo Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'KANDOO' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'KANDOO'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=100000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'KANDOO' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 15)) # Increased figure height for readability
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("KANDOO Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'KANDOO' brand in 'Baby Users' segment.")

print("\n--- Head of 'KANDOO' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Pura Baby Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'PURA BABY' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'PURA BABY'].copy()

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'PURA BABY' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("PURA BABY Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'PURA BABY' brand in 'Baby Users' segment.")

print("\n--- Head of 'PURA BABY' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Dr Brown's Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'DR.BROWN'S' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'DR.BROWN\'S'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'DR.BROWN'S' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("DR.BROWN'S Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'DR.BROWN'S' brand in 'Baby Users' segment.")

print("\n--- Head of 'DR.BROWN'S' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Momcozy Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'MOMCOZY' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'MOMCOZY'].copy()

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'MOMCOZY' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("MOMCOZY Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'MOMCOZY' brand in 'Baby Users' segment.")

print("\n--- Head of 'MOMCOZY' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Johnson's Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'JOHNSON\'S' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == "JOHNSON'S"].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'JOHNSON\'S' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("JOHNSON'S Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'JOHNSON\'S' brand in 'Baby Users' segment.")

print("\n--- Head of 'JOHNSON\'S' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Mother Sparsh Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'MOTHER SPARSH' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'MOTHER SPARSH'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'MOTHER SPARSH' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("MOTHER SPARSH Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'MOTHER SPARSH' brand in 'Baby Users' segment.")

print("\n--- Head of 'MOTHER SPARSH' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Munchkin Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'MUNCHKIN' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'MUNCHKIN'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'MUNCHKIN' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("MUNCHKIN Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'MUNCHKIN' brand in 'Baby Users' segment.")

print("\n--- Head of 'MUNCHKIN' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Supples Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'SUPPLES' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'SUPPLES'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'SUPPLES' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("SUPPLES Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'SUPPLES' brand in 'Baby Users' segment.")

print("\n--- Head of 'SUPPLES' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Baby User - Dapple Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create baby_wipe_data as done in earlier cells
baby_wipe_data = data[data['User'] == 'Baby']

# Filter for 'DAPPLE' brand in baby_wipe_data
honest_baby_data = baby_wipe_data[baby_wipe_data['CLEANED BRAND'] == 'DAPPLE'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honest_baby_data["Gissel's L52W $"] = honest_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'DAPPLE' brand (Baby Users)
if not honest_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honest_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("DAPPLE Brand (Baby Users): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honest_baby_data.shape[0]):
        x_val = honest_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honest_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honest_baby_data['TotalCount'].iloc[line]
        current_price_val = honest_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honest_baby_data['Attribute'].iloc[line]
        benefits_val = honest_baby_data['Benefits'].iloc[line]
        scent_val = honest_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honest_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honest_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honest_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honest_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honest_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'DAPPLE' brand in 'Baby Users' segment.")

print("\n--- Head of 'DAPPLE' Brand Data (Baby Users) ---")
display(honest_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Dude Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'DUDE' brand in non_baby_data
dude_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'DUDE'].copy()

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
dude_non_baby_data["Gissel's L52W $"] = dude_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'DUDE' brand (Users Except Baby)
if not dude_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=dude_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("DUDE Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,dude_non_baby_data.shape[0]):
        x_val = dude_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = dude_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = dude_non_baby_data['TotalCount'].iloc[line]
        current_price_val = dude_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = dude_non_baby_data['Attribute'].iloc[line]
        benefits_val = dude_non_baby_data['Benefits'].iloc[line]
        scent_val = dude_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = dude_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, dude_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, dude_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=dude_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=dude_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'DUDE' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'DUDE' Brand Data (Users Except Baby) ---")
display(dude_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Waterwipes Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'WATERWIPES' brand in non_baby_data
waterwipes_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'WATERWIPES'].copy()

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
waterwipes_non_baby_data["Gissel's L52W $"] = waterwipes_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'WATERWIPES' brand (Users Except Baby)
if not waterwipes_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=waterwipes_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("WATERWIPES Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,waterwipes_non_baby_data.shape[0]):
        x_val = waterwipes_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = waterwipes_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = waterwipes_non_baby_data['TotalCount'].iloc[line]
        current_price_val = waterwipes_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = waterwipes_non_baby_data['Attribute'].iloc[line]
        benefits_val = waterwipes_non_baby_data['Benefits'].iloc[line]
        scent_val = waterwipes_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = waterwipes_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, waterwipes_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, waterwipes_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=waterwipes_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=waterwipes_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'WATERWIPES' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'WATERWIPES' Brand Data (Users Except Baby) ---")
display(waterwipes_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Cottonelle Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'COTTONELLE' brand in non_baby_data
cottonelle_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'COTTONELLE'].copy()

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
cottonelle_non_baby_data["Gissel's L52W $"] = cottonelle_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=250000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.2f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'COTTONELLE' brand (Users Except Baby)
if not cottonelle_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 30)) # Increased figure height to 30
    sns.scatterplot(
        data=cottonelle_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("COTTONELLE Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,cottonelle_non_baby_data.shape[0]):
        x_val = cottonelle_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = cottonelle_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = cottonelle_non_baby_data['TotalCount'].iloc[line]
        current_price_val = cottonelle_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = cottonelle_non_baby_data['Attribute'].iloc[line]
        benefits_val = cottonelle_non_baby_data['Benefits'].iloc[line]
        scent_val = cottonelle_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = cottonelle_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, cottonelle_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, cottonelle_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=cottonelle_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=cottonelle_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'COTTONELLE' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'COTTONELLE' Brand Data (Users Except Baby) ---")
display(cottonelle_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Wet Ones Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'WET ONES' brand in non_baby_data
wetones_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'WET ONES'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
wetones_non_baby_data["Gissel's L52W $"] = wetones_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'WET ONES' brand (Users Except Baby)
if not wetones_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=wetones_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("WET ONES Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,wetones_non_baby_data.shape[0]):
        x_val = wetones_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = wetones_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = wetones_non_baby_data['TotalCount'].iloc[line]
        current_price_val = wetones_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = wetones_non_baby_data['Attribute'].iloc[line]
        benefits_val = wetones_non_baby_data['Benefits'].iloc[line]
        scent_val = wetones_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = wetones_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, wetones_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, wetones_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=wetones_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=wetones_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'WET ONES' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'WET ONES' Brand Data (Users Except Baby) ---")
display(wetones_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Private Label Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'PRIVATE LABEL' brand in non_baby_data
wetones_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'PRIVATE LABEL'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
wetones_non_baby_data["Gissel's L52W $"] = wetones_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'PRIVATE LABEL' brand (Users Except Baby)
if not wetones_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=wetones_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("PRIVATE LABEL Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,wetones_non_baby_data.shape[0]):
        x_val = wetones_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = wetones_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = wetones_non_baby_data['TotalCount'].iloc[line]
        current_price_val = wetones_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = wetones_non_baby_data['Attribute'].iloc[line]
        benefits_val = wetones_non_baby_data['Benefits'].iloc[line]
        scent_val = wetones_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = wetones_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, wetones_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, wetones_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=wetones_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=wetones_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'PRIVATE LABEL' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'PRIVATE LABEL' Brand Data (Users Except Baby) ---")
display(wetones_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Honey Pot Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'HONEY POT' brand in non_baby_data
honey_pot_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == "HONEY POT"].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honey_pot_non_baby_data["Gissel's L52W $"] = honey_pot_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'HONEY POT' brand (Users Except Baby)
if not honey_pot_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honey_pot_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("HONEY POT Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honey_pot_non_baby_data.shape[0]):
        x_val = honey_pot_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honey_pot_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honey_pot_non_baby_data['TotalCount'].iloc[line]
        current_price_val = honey_pot_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honey_pot_non_baby_data['Attribute'].iloc[line]
        benefits_val = honey_pot_non_baby_data['Benefits'].iloc[line]
        scent_val = honey_pot_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honey_pot_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honey_pot_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honey_pot_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honey_pot_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honey_pot_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'HONEY POT' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'HONEY POT' Brand Data (Users Except Baby) ---")
display(honey_pot_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Vagisil Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'VAGISIL' brand in non_baby_data
honey_pot_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == "VAGISIL"].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
honey_pot_non_baby_data["Gissel's L52W $"] = honey_pot_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'VAGISIL' brand (Users Except Baby)
if not honey_pot_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=honey_pot_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("VAGISIL Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,honey_pot_non_baby_data.shape[0]):
        x_val = honey_pot_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = honey_pot_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = honey_pot_non_baby_data['TotalCount'].iloc[line]
        current_price_val = honey_pot_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = honey_pot_non_baby_data['Attribute'].iloc[line]
        benefits_val = honey_pot_non_baby_data['Benefits'].iloc[line]
        scent_val = honey_pot_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = honey_pot_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, honey_pot_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, honey_pot_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=honey_pot_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=honey_pot_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'VAGISIL' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'VAGISIL' Brand Data (Users Except Baby) ---")
display(honey_pot_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Summer's Eve Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'SUMMER'S EVE' brand in non_baby_data
summers_eve_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == "SUMMER'S EVE"].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
summers_eve_non_baby_data["Gissel's L52W $"] = summers_eve_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'SUMMER'S EVE' brand (Users Except Baby)
if not summers_eve_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=summers_eve_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("SUMMER'S EVE Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,summers_eve_non_baby_data.shape[0]):
        x_val = summers_eve_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = summers_eve_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = summers_eve_non_baby_data['TotalCount'].iloc[line]
        current_price_val = summers_eve_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = summers_eve_non_baby_data['Attribute'].iloc[line]
        benefits_val = summers_eve_non_baby_data['Benefits'].iloc[line]
        scent_val = summers_eve_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = summers_eve_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, summers_eve_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, summers_eve_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=summers_eve_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=summers_eve_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'SUMMER'S EVE' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'SUMMER'S EVE' Brand Data (Users Except Baby) ---")
display(summers_eve_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Preparation H Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'PREPARATION H' brand in non_baby_data
preparationh_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'PREPARATION H'].copy()

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting
preparationh_non_baby_data["Gissel's L52W $"] = preparationh_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'PREPARATION H' brand (Users Except Baby)
if not preparationh_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=preparationh_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("PREPARATION H Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,preparationh_non_baby_data.shape[0]):
        x_val = preparationh_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = preparationh_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = preparationh_non_baby_data['TotalCount'].iloc[line]
        current_price_val = preparationh_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = preparationh_non_baby_data['Attribute'].iloc[line]
        benefits_val = preparationh_non_baby_data['Benefits'].iloc[line]
        scent_val = preparationh_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = preparationh_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, preparationh_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, preparationh_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=preparationh_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=preparationh_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'PREPARATION H' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'PREPARATION H' Brand Data (Users Except Baby) ---")
display(preparationh_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Wet Ones Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'WET ONES' brand in non_baby_data
wetones_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'WET ONES'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
wetones_non_baby_data["Gissel's L52W $"] = wetones_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'WET ONES' brand (Users Except Baby)
if not wetones_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=wetones_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("WET ONES Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,wetones_non_baby_data.shape[0]):
        x_val = wetones_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = wetones_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = wetones_non_baby_data['TotalCount'].iloc[line]
        current_price_val = wetones_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = wetones_non_baby_data['Attribute'].iloc[line]
        benefits_val = wetones_non_baby_data['Benefits'].iloc[line]
        scent_val = wetones_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = wetones_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, wetones_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, wetones_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=wetones_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=wetones_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'WET ONES' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'WET ONES' Brand Data (Users Except Baby) ---")
display(wetones_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Honest Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'HONEST' brand in non_baby_data
wetones_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'HONEST'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
wetones_non_baby_data["Gissel's L52W $"] = wetones_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'HONEST' brand (Users Except Baby)
if not wetones_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=wetones_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("HONEST Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,wetones_non_baby_data.shape[0]):
        x_val = wetones_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = wetones_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = wetones_non_baby_data['TotalCount'].iloc[line]
        current_price_val = wetones_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = wetones_non_baby_data['Attribute'].iloc[line]
        benefits_val = wetones_non_baby_data['Benefits'].iloc[line]
        scent_val = wetones_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = wetones_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, wetones_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, wetones_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=wetones_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=wetones_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'HONEST' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'HONEST' Brand Data (Users Except Baby) ---")
display(wetones_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Lume Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'LUME' brand in non_baby_data
wetones_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'LUME'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
wetones_non_baby_data["Gissel's L52W $"] = wetones_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'LUME' brand (Users Except Baby)
if not wetones_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=wetones_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("LUME Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,wetones_non_baby_data.shape[0]):
        x_val = wetones_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = wetones_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = wetones_non_baby_data['TotalCount'].iloc[line]
        current_price_val = wetones_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = wetones_non_baby_data['Attribute'].iloc[line]
        benefits_val = wetones_non_baby_data['Benefits'].iloc[line]
        scent_val = wetones_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = wetones_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, wetones_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, wetones_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=wetones_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=wetones_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'LUME' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'LUME' Brand Data (Users Except Baby) ---")
display(wetones_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

# ---------------------------**Non-Baby User - Mckesson Wipe------------------------**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys

try:
    from adjustText import adjust_text
except ImportError:
    print("adjustText not found. Installing...")
    !{sys.executable} -m pip install adjustText
    from adjustText import adjust_text

# Re-introduce data loading and preprocessing to ensure dataframes are defined
# This assumes the drive is mounted, which is handled in a previous cell.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")

# Standardize 'CLEANED BRAND' column as done in earlier cells
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Create non_baby_data to include all users EXCEPT 'Baby'
non_baby_data = data[data['User'] != 'Baby']

# Filter for 'MCKESSON' brand in non_baby_data
wetones_non_baby_data = non_baby_data[non_baby_data['CLEANED BRAND'] == 'MCKESSON'].copy()

# Ensure 'Gissel\\'s L52W $' has no NaN values for plotting
wetones_non_baby_data["Gissel's L52W $"] = wetones_non_baby_data["Gissel's L52W $"].fillna(0)

# Define custom tick functions for consistency
def set_y_axis_ticks_gissel_l52w(ax, df_filtered, y_label, y_increment=500000):
    min_y = df_filtered[y_label].min()
    max_y = df_filtered[y_label].max()
    if max_y == 0: # Handle cases where all sales are zero
        y_ticks = np.array([0, y_increment])
    else:
        y_ticks = np.arange(np.floor(min_y / y_increment) * y_increment, np.ceil(max_y / y_increment) * y_increment + y_increment, y_increment)
    y_labels = [f'${t/1000000:.1f}M' for t in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

def set_x_axis_ticks_price_per_count(ax, df_filtered, x_label, base_increment=0.05):
    min_x = df_filtered[x_label].min()
    max_x = df_filtered[x_label].max()
    x_ticks = np.arange(np.floor(min_x / base_increment) * base_increment,
                        np.ceil(max_x / base_increment) * base_increment + base_increment,
                        base_increment)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in x_ticks], rotation=45, ha='right')

# Create the scatter plot for 'MCKESSON' brand (Users Except Baby)
if not wetones_non_baby_data.empty:
    fig, ax = plt.subplots(figsize=(24, 20)) # Increased figure height to 20
    sns.scatterplot(
        data=wetones_non_baby_data,
        x='Calculated Price/Count or Price/Bundle',
        y="Gissel's L52W $",
        hue='ProductTitle',
        ax=ax,
        legend='full',
        s=100,
        alpha=0.8
    )
    ax.set_title("MCKESSON Brand (Users Except Baby): Calculated Price/Count or Price/Bundle vs. Gissel's L52W $ (with TotalCount, CurrentPrice, Rating, Attribute, Benefits, and Scent_ingredients)")
    ax.set_xlabel('Calculated Price/Count or Price/Bundle')
    ax.set_ylabel("Gissel's L52W $")

    # Add TotalCount, Attribute, Benefits, Scent_ingredients, and Rating as text annotation to each dot
    texts = []
    for line in range(0,wetones_non_baby_data.shape[0]):
        x_val = wetones_non_baby_data['Calculated Price/Count or Price/Bundle'].iloc[line]
        y_val = wetones_non_baby_data["Gissel's L52W $"].iloc[line]
        total_count = wetones_non_baby_data['TotalCount'].iloc[line]
        current_price_val = wetones_non_baby_data['CurrentPrice'].iloc[line] # Added CurrentPrice
        attribute_val = wetones_non_baby_data['Attribute'].iloc[line]
        benefits_val = wetones_non_baby_data['Benefits'].iloc[line]
        scent_val = wetones_non_baby_data['Scent_ingredient'].iloc[line]
        rating_val = wetones_non_baby_data['Rating'].iloc[line]
        # Always include all relevant information
        annotation_text = f"({total_count}, ${current_price_val:.2f}, {rating_val}, {attribute_val}, {benefits_val}, {scent_val})"

        texts.append(ax.text(x_val, y_val, annotation_text,
                             horizontalalignment='left',
                             size='medium',
                             color='black',
                             weight='semibold'))

    set_x_axis_ticks_price_per_count(ax, wetones_non_baby_data, 'Calculated Price/Count or Price/Bundle')
    set_y_axis_ticks_gissel_l52w(ax, wetones_non_baby_data, "Gissel's L52W $")

    ax.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(0.5, -0.35), loc='upper center', borderaxespad=0., title='ProductTitle', fontsize='small', ncol=2)
    plt.subplots_adjust(bottom=0.3)

    # Adjust text to prevent overlapping and draw lines
    adjust_text(texts,
                x=wetones_non_baby_data['Calculated Price/Count or Price/Bundle'],
                y=wetones_non_baby_data["Gissel's L52W $"],
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    plt.tight_layout()
    plt.show()
else:
    print("No data found for 'MCKESSON' brand in 'Users Except Baby' segment.")

print("\n--- Head of 'MCKESSON' Brand Data (Users Except Baby) ---")
display(wetones_non_baby_data[['ProductTitle', 'Calculated Price/Count or Price/Bundle', "Gissel's L52W $", 'TotalCount', 'CurrentPrice' , 'Rating', 'Attribute', 'Benefits', 'Scent_ingredient']].head())

#** Flushable Wipes Deep Dive **

In [ ]:
#Need to move 1
flushable_wipes_data = data[data['Flushable'].str.contains('Flushable', na=False)]
display(flushable_wipes_data.head())

In [ ]:
#Need to move 2
flushable_wipes_data.info()

In [ ]:
#Need to move 3
#Explore basic statistics about flushable wipes
flushable_wipes_data.describe()


In [ ]:
#Need to move 4
flushable_wipes_data.describe(include = ['object'])

In [ ]:
#Need to move 5
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Function to plot a boxplot and a histogram along the same scale

def histogram_boxplot(data, feature, figsize = (12, 7), kde = False, bins = None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows = 2,      # Number of rows of the subplot grid = 2
        sharex = True,  # x-axis will be shared among all subplots
        gridspec_kw = {"height_ratios": (0.25, 0.75)},
        figsize = figsize,
    )                   # Creating the 2 subplots
    sns.boxplot(
        data = data, x = feature, ax = ax_box2, showmeans = True, color = "violet"
    )                   # Boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2, bins = bins, palette = "winter"
    ) if bins else sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2
    )                   # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color = "green", linestyle = "--"
    )                   # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color = "black", linestyle = "-"
    )                   # Add median to the histogram

    # Add data labels for mean and median
    mean_val = data[feature].mean()
    median_val = data[feature].median()

    # Annotate mean
    ax_hist2.text(mean_val + (ax_hist2.get_xlim()[1] - ax_hist2.get_xlim()[0]) * 0.01,
                  ax_hist2.get_ylim()[1] * 0.9,
                  f'Mean: {mean_val:.2f}', color='green', ha='left', va='center')
    # Annotate median
    ax_hist2.text(median_val + (ax_hist2.get_xlim()[1] - ax_hist2.get_xlim()[0]) * 0.01,
                  ax_hist2.get_ylim()[1] * 0.8,
                  f'Median: {median_val:.2f}', color='black', ha='left', va='center')

    if feature == 'CurrentPrice':
        min_price = data[feature].min()
        max_price = data[feature].max()
        start_tick = np.floor(min_price / 5) * 5
        end_tick = np.ceil(max_price / 5) * 5

        # Adjust ticks for better readability, especially if range is small
        if end_tick - start_tick < 15: # Arbitrary threshold, adjust as needed
            # If the range is small, create ticks more frequently, e.g., every $1 or $2
            ticks = np.arange(start_tick, end_tick + 1, 2.5) # Try $2.5 increments for smaller ranges
        else:
            ticks = np.arange(start_tick, end_tick + 5, 5)

        # Filter ticks to be within reasonable bounds if necessary
        ticks = [t for t in ticks if t >= min_price and t <= max_price + 5] # Ensure ticks are relevant

        labels = [f'${t:.0f}' for t in ticks] # Format as whole dollars for clarity

        if len(ticks) > 15: # Avoid too many labels on the axis
             ticks = np.arange(start_tick, end_tick + 10, 10)
             labels = [f'${t:.0f}' for t in ticks]

        ax_hist2.set_xticks(ticks)
        ax_hist2.set_xticklabels(labels, rotation=45, ha='right')
        ax_box2.set_xticks(ticks)
        ax_box2.set_xticklabels(labels, rotation=45, ha='right')
        f2.suptitle(f'Distribution of {feature} with $5 Increments', fontsize=16)
    else:
        f2.suptitle(f'Distribution of {feature}', fontsize=16)


### ** Flushable wipes pricing by Major brands **

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os # Import os module
import matplotlib.ticker as mticker # Import for custom ticker formatting

# Define the file path
file_path = "/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx"

# --- Data Loading and Preprocessing (Ensuring 'data' is defined) ---
if 'data' not in locals(): # Check if 'data' is already defined
    if not os.path.exists('/content/drive/MyDrive'):
        print("Error: Google Drive is not mounted. Please run the cell that mounts Google Drive (e.g., `colab.drive.mount('/content/drive')`) and ensure it completes successfully.")
        # Do not proceed if drive is not mounted
        raise FileNotFoundError(f"Google Drive not mounted, cannot access {file_path}")
    elif not os.path.exists(file_path):
        print(f"Error: The data file was not found at the specified path: {file_path}")
        print("Please ensure the file exists at this exact location in your Google Drive.")
        # Do not proceed if file is missing
        raise FileNotFoundError(f"Data file missing at {file_path}")
    else:
        # If drive is mounted and file path exists, proceed to read
        try:
            data = pd.read_excel(file_path)
            data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
            data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
            total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
            data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct
            # Ensure baby_wipe_data and other_wipe_data are defined once 'data' is loaded
            baby_wipe_data = data[data['User'] == 'Baby'].copy()
            other_wipe_data = data[data['User'] != 'Baby'].copy()
        except Exception as e:
            print(f"An unexpected error occurred while loading or preprocessing the data: {e}")
            raise # Re-raise other exceptions

# If 'data' was already in locals (from a previous successful run), ensure derived dfs are also set up
if 'data' in locals() and 'baby_wipe_data' not in locals():
    # This block ensures baby_wipe_data and other_wipe_data are created even if data was pre-existing
    # without running the 'if 'data' not in locals():' block again.
    # It assumes data has the necessary columns.
    baby_wipe_data = data[data['User'] == 'Baby'].copy()
    other_wipe_data = data[data['User'] != 'Baby'].copy()


# Filter for flushable wipes across all datasets
flushable_wipes_all = data[data['Flushable'].str.contains('Flushable', na=False)].copy()
flushable_wipes_baby = baby_wipe_data[baby_wipe_data['Flushable'].str.contains('Flushable', na=False)].copy()
flushable_wipes_non_baby = other_wipe_data[other_wipe_data['Flushable'].str.contains('Flushable', na=False)].copy()

# Define user segments with flushable wipes data
user_segments_flushable = {
    'All Users': flushable_wipes_all,
    'Baby Users': flushable_wipes_baby,
    'Non-Baby Users': flushable_wipes_non_baby
}

for segment_name, segment_df in user_segments_flushable.items():
    if segment_df.empty:
        print(f"No flushable wipes data available for {segment_name}. Skipping plot.")
        continue

    # Fill NaN values in 'Gissel\'s L52W $' with 0 for plotting and aggregation
    segment_df["Gissel's L52W $"] = segment_df["Gissel's L52W $"].fillna(0)
    # Also fill NaN for 'Calculated Price/Count or Price/Bundle' if it could happen.
    segment_df['Calculated Price/Count or Price/Bundle'] = segment_df['Calculated Price/Count or Price/Bundle'].fillna(0)

    # Aggregate data by CLEANED BRAND for plotting
    plot_data = segment_df.groupby('CLEANED BRAND').agg(
        avg_price_per_count=('Calculated Price/Count or Price/Bundle', 'mean'),
        total_l52w_sales=("Gissel's L52W $", 'sum')
    ).reset_index()

    # Sort by total_l52w_sales in descending order and select top 50 brands
    plot_data = plot_data.sort_values(by='total_l52w_sales', ascending=False).head(50)

    if plot_data.empty:
        print(f"No top brands with sales data found for {segment_name} after aggregation. Skipping plot.")
        continue

    plt.figure(figsize=(18, 9)) # Adjust figure size for bar plot
    ax = sns.barplot(
        data=plot_data,
        x='CLEANED BRAND',
        y='total_l52w_sales',
        palette='viridis'
    )

    # Add average price/count on top of each bar
    for i, p in enumerate(ax.patches):
        avg_price_val = plot_data['avg_price_per_count'].iloc[i]
        # Place annotation slightly above the bar
        ax.annotate(f'${avg_price_val:.2f}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=9, rotation=45, color='black',
                    xytext=(0, 5), textcoords='offset points')

    plt.title(f'Flushable Wipes: Total L52W$ Sales by Brand (Top {len(plot_data)}) - {segment_name}\n(Avg. Price/Count annotated on bars)')
    plt.xlabel('CLEANED BRAND')
    plt.ylabel('Total Gissel\'s L52W $ Sales')
    plt.xticks(rotation=90, ha='right') # Rotate x-axis labels for readability

    # Format y-axis labels to show in millions
    formatter = mticker.FuncFormatter(lambda x, p: '${:,.0f}M'.format(x / 1_000_000) if x >= 1_000_000 else '${:,.0f}'.format(x))
    ax.yaxis.set_major_formatter(formatter)

    plt.grid(axis='y', linestyle='--', alpha=0.7) # Grid only on y-axis for bar plots
    plt.tight_layout() # Adjust layout to make space for labels and annotations
    plt.show()


** The "Count" in histogram" represents the number of products within that price bracket, not Count/pack in the "Count" column.

In [ ]:
#Need to move 6
histogram_boxplot(flushable_wipes_data, 'Calculated Price/Count or Price/Bundle')
histogram_boxplot(flushable_wipes_data, 'CurrentPrice')
histogram_boxplot(flushable_wipes_data, 'TotalCount')

### **Let's visualize the relationship between 'CurrentPrice' and 'TotalCount' using a scatter plot.**

In [ ]:
#Need to move 7
plt.figure(figsize=(10, 6))
sns.scatterplot(data=flushable_wipes_data, x='CurrentPrice', y='TotalCount')
plt.title('CurrentPrice vs TotalCount for Flushable Wipes')
plt.xlabel('CurrentPrice')
plt.ylabel('TotalCount')

# Set x-axis ticks for CurrentPrice at $10 increments
min_price = flushable_wipes_data['CurrentPrice'].min()
max_price = flushable_wipes_data['CurrentPrice'].max()
x_ticks = np.arange(np.floor(min_price / 10) * 10, np.ceil(max_price / 10) * 10 + 10, 10)
plt.xticks(x_ticks, [f'${t:.0f}' for t in x_ticks], rotation=45, ha='right')

# Set y-axis ticks for TotalCount at 100 increments
min_total_count = flushable_wipes_data['TotalCount'].min()
max_total_count = flushable_wipes_data['TotalCount'].max()
y_ticks = np.arange(np.floor(min_total_count / 100) * 100, np.ceil(max_total_count / 100) * 100 + 100, 100)
plt.yticks(y_ticks)

plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

### **Pearson Correlation Coefficient between CurrentPrice and TotalCount**

In [ ]:
#Need to move 8
correlation = flushable_wipes_data['CurrentPrice'].corr(flushable_wipes_data['TotalCount'])
print(f"Pearson correlation coefficient between CurrentPrice and TotalCount: {correlation}")

correlation coefficient between CurrentPrice and TotalCount: 0.4990247755904606
1. There is weak-to-moderate positive linear relationship between TotalCount and CurrentPrice, but there are other variants impacting CurrentPrice.
2. Most Flushable wipes products are between \$0-\$60 unit price range and 0-700 TotalCounts range.
3. Visually, within the main cluster, price and count don't move together very strongly; the apparent “moderate” correlation number is mostly about the combined effect of the main group plus those extreme points.

### **Let's visualize the relationship between 'CurrentPrice' and 'TotalCount' using a scatter plot for \$0-\$60 CurrentPrice flushable wipes.**

In [ ]:
#Need to move 9
# Filter flushable wipes data for products with CurrentPrice between $0 and $60
flushable_wipes_data_filtered_price = flushable_wipes_data[
    (flushable_wipes_data['CurrentPrice'] >= 0) &
    (flushable_wipes_data['CurrentPrice'] <= 60)
].copy()

print(f"Number of flushable wipes products with CurrentPrice between $0 and $60: {len(flushable_wipes_data_filtered_price)}")

plt.figure(figsize=(10, 6))
sns.scatterplot(data=flushable_wipes_data_filtered_price, x='CurrentPrice', y='TotalCount')
plt.title('CurrentPrice vs TotalCount for Flushable Wipes (Price $0-60)')
plt.xlabel('CurrentPrice')
plt.ylabel('TotalCount')

# Set x-axis ticks for CurrentPrice at $5 increments
min_price_filtered = flushable_wipes_data_filtered_price['CurrentPrice'].min()
max_price_filtered = flushable_wipes_data_filtered_price['CurrentPrice'].max()
x_ticks_filtered = np.arange(np.floor(min_price_filtered / 5) * 5, np.ceil(max_price_filtered / 5) * 5 + 5, 5)
plt.xticks(x_ticks_filtered, [f'${t:.0f}' for t in x_ticks_filtered], rotation=45, ha='right')

# Set y-axis ticks for TotalCount at appropriate increments (e.g., 50 or 100)
min_total_count_filtered = flushable_wipes_data_filtered_price['TotalCount'].min()
max_total_count_filtered = flushable_wipes_data_filtered_price['TotalCount'].max()
y_ticks_filtered = np.arange(0, np.ceil(max_total_count_filtered / 100) * 100 + 100, 100)

# Adjust for smaller ranges if needed
if (max_total_count_filtered - min_total_count_filtered) < 200 and (max_total_count_filtered - min_total_count_filtered) > 0:
    y_ticks_filtered = np.arange(0, np.ceil(max_total_count_filtered / 50) * 50 + 50, 50)

plt.yticks(y_ticks_filtered)

plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Pearson Correlation Coefficient between CurrentPrice and TotalCount for Flushable Wipes between \$0-\$60**

In [ ]:
#Need to move 10
correlation_flushable_wipes_data_filtered_price = flushable_wipes_data_filtered_price['CurrentPrice'].corr(flushable_wipes_data_filtered_price['TotalCount'])
print(f"Pearson correlation coefficient between CurrentPrice and TotalCount: {correlation_flushable_wipes_data_filtered_price}")

1. CurrentPrice and TotalCount for Flushable Wipes between $0-$60 indicates moderately strong positive linear relationship between CurrentPrice and TotalCount.
2. As the total number of wipes per ASIN increases, the current price tends to increase as well, and they move together in a roughly linear way.

In [ ]:
#Need to move 11
average_price_by_brand = flushable_wipes_data.groupby('CLEANED BRAND')['CurrentPrice'].mean().reset_index()

print("Average CurrentPrice by CLEANED BRAND for Flushable Wipes:")
display(average_price_by_brand.sort_values(by='CurrentPrice', ascending=False))

### **Let's visualize the relationship between 'Rating' and 'Calculated Price/Count or Price/Bundle' using a scatter plot.**

In [ ]:
#Need to move 12
plt.figure(figsize=(10, 6))
sns.scatterplot(data=flushable_wipes_data, x='Rating', y='Calculated Price/Count or Price/Bundle')
plt.title('Rating vs. Calculated Price/Count or Price/Bundle for Flushable Wipes')
plt.xlabel('Rating')
plt.ylabel('Calculated Price/Count or Price/Bundle')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### **Pearson Correlation Coefficient between Rating and Calculated Price/Count or Price/Bundle**

In [ ]:
#Need to move 13
correlation_rating_price_per_count = flushable_wipes_data['Rating'].corr(flushable_wipes_data['Calculated Price/Count or Price/Bundle'])
print(f"Pearson correlation coefficient between Rating and Calculated Price/Count or Price/Bundle: {correlation_rating_price_per_count}")

1. No linear relationship between rating and price per count/bundle for these flushable wipes.
2. The scatter points are spread over a wide band of prices for ratings clustered mostly between about 3.5 and 5.0.

In [ ]:
#Need to move 15
import sys
!{sys.executable} -m pip install plotly

In [ ]:
##Need to move 16
import plotly.express as px

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting and size scaling
products_with_matching_total_count['Gissel\'s L52W $'] = products_with_matching_total_count['Gissel\'s L52W $'].fillna(0)

# Create the 3D scatter plot using Plotly Express
fig = px.scatter_3d(products_with_matching_total_count,
                    x='TotalCount',
                    y='CurrentPrice',
                    z='Gissel\'s L52W $',
                    color='CLEANED BRAND', # Color points by brand
                    size='Gissel\'s L52W $', # Size points by Gissel's L52W $
                    hover_name='CLEANED BRAND', # Display brand name on hover
                    title='3D Ball Graph: TotalCount vs. CurrentPrice vs. Gissel\'s L52W $ by Brand',
                    labels={'TotalCount': 'Total Count (Wipes)',
                            'CurrentPrice': 'Current Price ($)',
                            'Gissel\'s L52W $': 'Gissel\'s L52W Sales ($)'},
                    height=700)

# Adjust marker size scaling for better visibility if needed
fig.update_traces(marker=dict(sizemode='diameter', sizeref=1.0))

fig.show()

In [ ]:
#Need to move 17
import plotly.express as px
import numpy as np

# Ensure products_with_matching_total_count is available (already created and 'Gissel's L52W $' NaNs filled with 0)

# Create a new column for plot size: apply log1p transformation for proportional sizing.
# For values that are 0 (representing original NaNs or actual 0 sales), assign a minimum visible size (e.g., 2 after log1p)
# so they appear as visible reference points.
products_with_matching_total_count['Plot_Size'] = products_with_matching_total_count["Gissel's L52W $"].apply(
    lambda x: np.log1p(x) if x > 0 else 2 # np.log1p(0) is 0, so 'else 2' ensures visibility for 0 sales
)

# Create the 2D scatter plot using Plotly Express
fig = px.scatter(products_with_matching_total_count,
                 x='CurrentPrice',
                 y='Rating',
                 color='CLEANED BRAND', # Color points by brand
                 size='Plot_Size', # Size points by the transformed Gissel's L52W $
                 hover_name='CLEANED BRAND', # Display brand name on hover
                 hover_data={'CurrentPrice': ':.2f', 'Rating': True, "Gissel's L52W $": ':.2f', 'Plot_Size': False}, # Show original Gissel's L52W $ on hover
                 title='2D Scatter Plot: CurrentPrice vs. Rating by Brand (Size by Gissel\'s L52W $)',
                 labels={'CurrentPrice': 'Current Price ($)',
                         'Rating': 'Rating',
                         'CLEANED BRAND': 'Brand',
                         'Plot_Size': 'Gissel\'s L52W $ (Transformed)'},
                 size_max=60, # Adjust maximum dot size for better visualization
                 height=600)

fig.show()

### **Investigating 'BOOGIE' Brand Entries**

To understand why 'BOOGIE' might have appeared multiple times in the plots, let's examine the data for this brand closely, specifically looking at variations in brand name (if any, despite prior cleaning), associated user types, product titles, and relevant sales metrics.

In [ ]:
# Filter data for 'BOOGIE' brand
boogie_brand_data = data[data['CLEANED BRAND'] == 'BOOGIE'].copy()

print("Unique 'CLEANED BRAND' entries for 'BOOGIE':")
display(boogie_brand_data['CLEANED BRAND'].value_counts())

print("\nUnique 'User' types associated with 'BOOGIE':")
display(boogie_brand_data['User'].value_counts())

print("\nProduct Titles for 'BOOGIE' brand (top 20 if many):")
display(boogie_brand_data['ProductTitle'].value_counts().head(20))

print("\nRelevant data for 'BOOGIE' brand entries:")
display(boogie_brand_data[['ProductTitle', 'User', 'Calculated Price/Count or Price/Bundle', 'Gissel\'s L52W $']].sort_values(by='User').head(20))

### **Observations regarding 'BOOGIE' brand:**

Based on the analysis of 'BOOGIE' brand entries:

1.  **'CLEANED BRAND' Consistency**: The 'CLEANED BRAND' column for 'BOOGIE' is consistent, showing only one unique value 'BOOGIE'. This confirms that the initial standardization successfully grouped all 'BOOGIE' variations under a single brand name.

2.  **User Types**: The 'BOOGIE' brand is associated with multiple 'User' types, including 'Baby', 'Non-Baby', and 'Feminine'. However, since the plot in question was specifically for 'Baby Users', the issue of 'BOOGIE' appearing twice likely stems from distinct product offerings within the 'Baby' user segment.

3.  **Product Titles**: There are multiple distinct 'ProductTitle' entries for the 'BOOGIE' brand. For 'Baby Users' specifically, there might be different products (e.g., 'Boogie Wipes Saline Wipes for Baby & Kids, Wet Nose Wipe, Grape Scent, 30 Count (Pack of 3)' vs. 'Boogie Bottoms Diaper Rash Spray, No-Mess Diaper Rash Cream Alternative, 3.4oz, 1 Count') which, when aggregated or selected based on top metrics, could lead to the brand appearing multiple times if the plotting logic considers distinct product instances for the 'hue' or if certain metrics are not unique at the brand level across all products.

4.  **Sales and Pricing Data**: It's possible that different products under the 'BOOGIE' brand, even within the 'Baby' user segment, have different 'Calculated Price/Count or Price/Bundle' and 'Gissel's L52W $' values. If the plotting is done at a product level and then aggregated, or if the aggregation method retains multiple entries (e.g., if there are multiple products with very high 'Rating_Count Weight' that fall into the 'top 10%' calculation), it could result in multiple points for the same brand.

To fully clarify this, the plotting logic for 'Top 20 Brands (Baby Users) by Rating_Count Weight' might need to be revisited to see how `CLEANED BRAND` and `ProductTitle` are handled in the aggregation or selection process for the plot, especially if `hue` was set to `CLEANED BRAND` directly without further aggregation.

### **2D Scatter Plots for Each Unique `TotalCount` in `products_baby_users.csv`**

These plots visualize the relationship between `CurrentPrice` and `Rating` for products specific to 'Baby' users that share the same `TotalCount` values. Each point is colored by its `CLEANED BRAND` and sized according to `Gissel's L52W $` to indicate sales performance.

In [ ]:
#Need to move 18
import plotly.express as px
import numpy as np
import pandas as pd

# Load products_baby_users.csv (assuming it was saved previously)
baby_users_df = pd.read_csv('products_baby_users.csv')

# Ensure 'Gissel\'s L52W $' has no NaN values for plotting and size scaling
baby_users_df['Gissel\'s L52W $'] = baby_users_df['Gissel\'s L52W $'].fillna(0)

# Create a new column for plot size: apply log1p transformation for proportional sizing.
# For values that are 0 (representing original NaNs or actual 0 sales), assign a minimum visible size (e.g., 2 after log1p)
# so they appear as visible reference points.
baby_users_df['Plot_Size'] = baby_users_df["Gissel's L52W $"].apply(
    lambda x: np.log1p(x) if x > 0 else 2 # np.log1p(0) is 0, so 'else 2' ensures visibility for 0 sales
)

unique_total_counts_baby = baby_users_df['TotalCount'].unique()

if not unique_total_counts_baby.size == 0:
    for total_count_value in unique_total_counts_baby:
        subset_data = baby_users_df[
            baby_users_df['TotalCount'] == total_count_value
        ].copy()

        if not subset_data.empty:
            fig = px.scatter(subset_data,
                             x='CurrentPrice',
                             y='Rating',
                             color='CLEANED BRAND', # Color points by brand
                             size='Plot_Size', # Size points by the transformed Gissel's L52W $
                             hover_name='CLEANED BRAND', # Display brand name on hover
                             hover_data={'CurrentPrice': ':.2f', 'Rating': True, "Gissel's L52W $": ':.2f', 'Plot_Size': False}, # Show original Gissel's L52W $ on hover
                             title=f'2D Scatter Plot for Baby Users: TotalCount = {total_count_value}',
                             labels={'CurrentPrice': 'Current Price ($)',
                                     'Rating': 'Rating',
                                     'CLEANED BRAND': 'Brand',
                                     'Plot_Size': 'Gissel\'s L52W $ (Transformed)'},
                             size_max=60, # Adjust maximum dot size for better visualization
                             height=600)

            fig.show()
        else:
            print(f"No data for TotalCount = {total_count_value} in baby_users_df.")
else:
    print("No unique 'TotalCount' values found in baby_users_df to plot.")

### **Compare Average Price of Flushable vs. Non-Flushable Products**

In [ ]:
# Create copies of the data for each user segment to avoid modifying the original DataFrames
product_data_all = data.copy()
product_data_baby = baby_wipe_data.copy()
product_data_non_baby = other_wipe_data.copy()

# List of dataframes and their corresponding titles
data_segments = [
    (product_data_all, "All Users"),
    (product_data_baby, "Baby Users"),
    (product_data_non_baby, "Non-Baby Users")
]

for df_segment, title_suffix in data_segments:
    # Fill NaN values in 'Flushable' with 'Non-Flushable' for explicit categorization
    df_segment['Flushable'] = df_segment['Flushable'].fillna('Non-Flushable')

    # --- Analysis for 'Calculated Price/Count or Price/Bundle' ---
    # Calculate the average 'Calculated Price/Count or Price/Bundle' for flushable and non-flushable products
    average_price_per_unit = df_segment.groupby('Flushable')['Calculated Price/Count or Price/Bundle'].mean().reset_index()

    # Display the average prices per unit
    print(f"\nAverage Calculated Price/Count or Price/Bundle for Flushable vs. Non-Flushable Products ({title_suffix}):")
    display(average_price_per_unit)

    # Visualize the comparison for Price/Count
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Flushable', y='Calculated Price/Count or Price/Bundle', data=average_price_per_unit, palette='viridis')
    plt.title(f'Average Calculated Price/Count or Price/Bundle: Flushable vs. Non-Flushable ({title_suffix})')
    plt.xlabel('Product Type')
    plt.ylabel('Average Calculated Price/Count or Price/Bundle')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

    # --- Analysis for 'CurrentPrice' ---
    # Calculate the average 'CurrentPrice' for flushable and non-flushable products
    average_current_price = df_segment.groupby('Flushable')['CurrentPrice'].mean().reset_index()

    # Display the average current prices
    print(f"\nAverage Current Price for Flushable vs. Non-Flushable Products ({title_suffix}):")
    display(average_current_price)

    # Visualize the comparison for CurrentPrice
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Flushable', y='CurrentPrice', data=average_current_price, palette='magma')
    plt.title(f'Average Current Price: Flushable vs. Non-Flushable ({title_suffix})')
    plt.xlabel('Product Type')
    plt.ylabel('Average Current Price ($)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

### **Compare Average Price/Count of Flushable vs. Non-Flushable Products**

In [ ]:
# Create copies of the data for each user segment to avoid modifying the original DataFrames
product_data_all = data.copy()
product_data_baby = baby_wipe_data.copy()
product_data_non_baby = other_wipe_data.copy()

# List of dataframes and their corresponding titles
data_segments = [
    (product_data_all, "All Users"),
    (product_data_baby, "Baby Users"),
    (product_data_non_baby, "Non-Baby Users")
]

for df_segment, title_suffix in data_segments:
    # Fill NaN values in 'Flushable' with 'Non-Flushable' for explicit categorization
    df_segment['Flushable'] = df_segment['Flushable'].fillna('Non-Flushable')

    # Calculate the average 'Calculated Price/Count or Price/Bundle' for flushable and non-flushable products
    average_prices = df_segment.groupby('Flushable')['Calculated Price/Count or Price/Bundle'].mean().reset_index()

    # Display the average prices
    print(f"Average Calculated Price/Count or Price/Bundle for Flushable vs. Non-Flushable Products ({title_suffix}):")
    display(average_prices)

    # Visualize the comparison
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Flushable', y='Calculated Price/Count or Price/Bundle', data=average_prices, palette='viridis')
    plt.title(f'Average Calculated Price/Count or Price/Bundle: Flushable vs. Non-Flushable ({title_suffix})')
    plt.xlabel('Product Type')
    plt.ylabel('Average Calculated Price/Count or Price/Bundle')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

### **Compare Average Price/Count of XL vs. Non-XL Products**

In [ ]:
# Create copies of the data for each user segment to avoid modifying the original DataFrames
product_data_all = data.copy()
product_data_baby = baby_wipe_data.copy()
product_data_non_baby = other_wipe_data.copy()

# List of dataframes and their corresponding titles
data_segments = [
    (product_data_all, "All Users"),
    (product_data_baby, "Baby Users"),
    (product_data_non_baby, "Non-Baby Users")
]

for df_segment, title_suffix in data_segments:
    # Fill NaN values in 'XL' with 'Non-XL' for explicit categorization
    df_segment['XL'] = df_segment['XL'].fillna('Non-XL')

    # Calculate the average 'Calculated Price/Count or Price/Bundle' for XL and non-XL products
    average_prices = df_segment.groupby('XL')['Calculated Price/Count or Price/Bundle'].mean().reset_index()

    # Display the average prices
    print(f"Average Calculated Price/Count or Price/Bundle for XL vs. Non-XL Products ({title_suffix}):")
    display(average_prices)

    # Visualize the comparison
    plt.figure(figsize=(8, 6))
    sns.barplot(x='XL', y='Calculated Price/Count or Price/Bundle', data=average_prices, palette='viridis')
    plt.title(f'Average Calculated Price/Count or Price/Bundle: XL vs. Non-XL ({title_suffix})')
    plt.xlabel('Product Type')
    plt.ylabel('Average Calculated Price/Count or Price/Bundle')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os # Import os module
import matplotlib.ticker as mticker # Import for custom ticker formatting

# Define the file path
file_path = "/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx"

# --- Data Loading and Preprocessing (Ensuring 'data' is defined) ---
if 'data' not in locals(): # Check if 'data' is already defined
    if not os.path.exists('/content/drive/MyDrive'):
        print("Error: Google Drive is not mounted. Please run the cell that mounts Google Drive (e.g., `colab.drive.mount('/content/drive')`) and ensure it completes successfully.")
        # Do not proceed if drive is not mounted
        raise FileNotFoundError(f"Google Drive not mounted, cannot access {file_path}")
    elif not os.path.exists(file_path):
        print(f"Error: The data file was not found at the specified path: {file_path}")
        print("Please ensure the file exists at this exact location in your Google Drive.")
        # Do not proceed if file is missing
        raise FileNotFoundError(f"Data file missing at {file_path}")
    else:
        # If drive is mounted and file path exists, proceed to read
        try:
            data = pd.read_excel(file_path)
            data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
            data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
            total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
            data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct
            # Ensure baby_wipe_data and other_wipe_data are defined once 'data' is loaded
            baby_wipe_data = data[data['User'] == 'Baby'].copy()
            other_wipe_data = data[data['User'] != 'Baby'].copy()
        except Exception as e:
            print(f"An unexpected error occurred while loading or preprocessing the data: {e}")
            raise # Re-raise other exceptions

# If 'data' was already in locals (from a previous successful run), ensure derived dfs are also set up
if 'data' in locals() and 'baby_wipe_data' not in locals():
    # This block ensures baby_wipe_data and other_wipe_data are created even if data was pre-existing
    # without running the 'if 'data' not in locals():' block again.
    # It assumes data has the necessary columns.
    baby_wipe_data = data[data['User'] == 'Baby'].copy()
    other_wipe_data = data[data['User'] != 'Baby'].copy()


# Filter for XL wipes across all datasets
XL_wipes_all = data[data['XL'].str.contains('XL', na=False)].copy()
XL_wipes_baby = baby_wipe_data[baby_wipe_data['XL'].str.contains('XL', na=False)].copy()
XL_wipes_non_baby = other_wipe_data[other_wipe_data['XL'].str.contains('XL', na=False)].copy()

# Define user segments with XL wipes data
user_segments_XL = {
    'All Users': XL_wipes_all,
    'Baby Users': XL_wipes_baby,
    'Non-Baby Users': XL_wipes_non_baby
}

for segment_name, segment_df in user_segments_XL.items():
    if segment_df.empty:
        print(f"No XL wipes data available for {segment_name}. Skipping plot.")
        continue

    # Fill NaN values in 'Gissel\'s L52W $' with 0 for plotting and aggregation
    segment_df["Gissel's L52W $"] = segment_df["Gissel's L52W $"].fillna(0)
    # Also fill NaN for 'Calculated Price/Count or Price/Bundle' if it could happen.
    segment_df['Calculated Price/Count or Price/Bundle'] = segment_df['Calculated Price/Count or Price/Bundle'].fillna(0)

    # Aggregate data by CLEANED BRAND for plotting
    plot_data = segment_df.groupby('CLEANED BRAND').agg(
        avg_price_per_count=('Calculated Price/Count or Price/Bundle', 'mean'),
        total_l52w_sales=("Gissel's L52W $", 'sum')
    ).reset_index()

    # Sort by total_l52w_sales in descending order and select top 50 brands
    plot_data = plot_data.sort_values(by='total_l52w_sales', ascending=False).head(50)

    if plot_data.empty:
        print(f"No top brands with sales data found for {segment_name} after aggregation. Skipping plot.")
        continue

    plt.figure(figsize=(18, 9)) # Adjust figure size for bar plot
    ax = sns.barplot(
        data=plot_data,
        x='CLEANED BRAND',
        y='total_l52w_sales',
        palette='viridis'
    )

    # Add average price/count on top of each bar
    for i, p in enumerate(ax.patches):
        avg_price_val = plot_data['avg_price_per_count'].iloc[i]
        # Place annotation slightly above the bar
        ax.annotate(f'${avg_price_val:.2f}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=9, rotation=45, color='black',
                    xytext=(0, 5), textcoords='offset points')

    plt.title(f'XL Wipes: Total L52W$ Sales by Brand (Top {len(plot_data)}) - {segment_name}\n(Avg. Price/Count annotated on bars)')
    plt.xlabel('CLEANED BRAND')
    plt.ylabel('Total Gissel\'s L52W $ Sales')
    plt.xticks(rotation=90, ha='right') # Rotate x-axis labels for readability

    # Format y-axis labels to show in millions
    formatter = mticker.FuncFormatter(lambda x, p: '${:,.0f}M'.format(x / 1_000_000) if x >= 1_000_000 else '${:,.0f}'.format(x))
    ax.yaxis.set_major_formatter(formatter)

    plt.grid(axis='y', linestyle='--', alpha=0.7) # Grid only on y-axis for bar plots
    plt.tight_layout() # Adjust layout to make space for labels and annotations
    plt.show()

In [ ]:
import nbconvert
import os

# Define the output HTML filename
html_filename = "Amazon Wipes Price Analysis.html"

# Construct the command to convert the current notebook to HTML
# The current notebook path can be obtained from the `_notebook_path` global variable if available, or assume it's the current working directory.
# For Colab, the notebook path is usually `/content/drive/MyDrive/path/to/notebook.ipynb` if mounted, or a temporary path if not.
# Assuming the user is running the notebook, `_notebook_path` is usually available.

try:
    # This attempts to get the current notebook's path in Colab
    from google.colab import _message as _colab_message
    notebook_path = _colab_message.blocking_request(f'get_filename')['filename']
except Exception:
    notebook_path = "/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Amazon Wipes Price Analysis V1 05292026.ipynb"

# Execute the nbconvert command
!jupyter nbconvert --to html "{notebook_path}" --output "{html_filename}"

print(f"Notebook converted to HTML: {html_filename}")

** Average Pricing Index, Premium Price Index (Average Price/Count for major baby and non baby brand and the average and median price of these major brands)**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_top_brands_avg_price_per_count(df, title_suffix):
    """
    Generates a bar plot showing the average 'Calculated Price/Count or Price/Bundle'
    for the top 10% brands by 'Rating_Count Weight' for a given DataFrame.
    Includes mean and median price lines.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """
    # Ensure 'Rating_Count Weight' is available (it's created in previous cells)
    if 'Rating_Count Weight' not in df.columns:
        print(f"Error: 'Rating_Count Weight' column not found in {title_suffix} DataFrame.")
        return

    if df.empty:
        print(f"No data available for {title_suffix}.")
        return

    # Calculate the sum of 'Rating_Count Weight' per brand
    brand_rating_weight = df.groupby('CLEANED BRAND')['Rating_Count Weight'].sum()

    # Determine the number of top brands (top 10%)
    num_unique_brands = len(brand_rating_weight)
    top_n_brands = max(1, int(num_unique_brands * 0.10)) # Ensure at least 1 brand if data exists

    # Get the names of the top N brands by Rating_Count Weight
    top_brands_names = brand_rating_weight.nlargest(top_n_brands).index

    if top_brands_names.empty:
        print(f"No top brands found for {title_suffix} based on Rating_Count Weight.")
        return

    # Filter the DataFrame to include only these top brands
    top_brands_data = df[df['CLEANED BRAND'].isin(top_brands_names)].copy()

    if top_brands_data.empty:
        print(f"No data for products from top brands in {title_suffix}.")
        return

    # Aggregate for plotting: Calculate mean 'Calculated Price/Count or Price/Bundle' for each top brand
    plot_data = top_brands_data.groupby('CLEANED BRAND')['Calculated Price/Count or Price/Bundle'].mean().reset_index()

    # Sort the brands by their average price/count for better visualization
    plot_data = plot_data.sort_values(by='Calculated Price/Count or Price/Bundle', ascending=False)

    plt.figure(figsize=(20, 8)) # Increased figure width for longer x-axis
    ax = sns.barplot(x='CLEANED BRAND', y='Calculated Price/Count or Price/Bundle', data=plot_data, palette='viridis')

    # Add average price/count on top of each bar
    for p in ax.patches:
        ax.annotate(f'${p.get_height():.3f}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points')

    # Calculate mean and median for the displayed brands
    mean_price = plot_data['Calculated Price/Count or Price/Bundle'].mean()
    median_price = plot_data['Calculated Price/Count or Price/Bundle'].median()

    # Add mean line (green dashed)
    plt.axhline(mean_price, color='green', linestyle='--', linewidth=2, label=f'Mean Price: ${mean_price:.2f}')
    # Add median line (black solid)
    plt.axhline(median_price, color='black', linestyle='-', linewidth=2, label=f'Median Price: ${median_price:.2f}')

    plt.title(f'Average Price/Count for Top {top_n_brands} Brands by Rating_Count Weight ({title_suffix})')
    plt.xlabel('CLEANED BRAND')
    plt.ylabel('Average Calculated Price/Count or Price/Bundle')
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Ensure data and derived dataframes have 'Rating_Count Weight' column
# Standardize 'CLEANED BRAND' column
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Recalculate 'Rating_x_RatingCount' and 'Rating_Count Weight' for the main 'data' DataFrame
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

# Re-create baby_wipe_data and other_wipe_data to ensure they inherit the new column
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Call the function for All Users
print("\n--- Plotting for All Users ---")
plot_top_brands_avg_price_per_count(data, 'All Users')

# Call the function for Baby Users
print("\n--- Plotting for Baby Users ---")
plot_top_brands_avg_price_per_count(baby_wipe_data, 'Baby Users')

# Call the function for Non-Baby Users
print("\n--- Plotting for Non-Baby Users ---")
plot_top_brands_avg_price_per_count(other_wipe_data, 'Non-Baby Users')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd # Ensure pandas is imported as it's used for DataFrame operations

def plot_top_brands_avg_price_per_count(df, title_suffix):
    """
    Generates a bar plot showing the average 'Calculated Price/Count or Price/Bundle'
    for the top 10% brands by 'Rating_Count Weight' for a given DataFrame.
    Includes mean and median price lines.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """
    # Ensure 'Rating_Count Weight' is available (it's created in previous cells)
    if 'Rating_Count Weight' not in df.columns:
        print(f"Error: 'Rating_Count Weight' column not found in {title_suffix} DataFrame.")
        return

    if df.empty:
        print(f"No data available for {title_suffix}.")
        return

    # Calculate the sum of 'Rating_Count Weight' per brand
    brand_rating_weight = df.groupby('CLEANED BRAND')['Rating_Count Weight'].sum()

    # Determine the number of top brands (top 10%)
    num_unique_brands = len(brand_rating_weight)
    top_n_brands = max(1, int(num_unique_brands * 0.10)) # Ensure at least 1 brand if data exists

    # Get the names of the top N brands by Rating_Count Weight
    top_brands_names = brand_rating_weight.nlargest(top_n_brands).index

    if top_brands_names.empty:
        print(f"No top brands found for {title_suffix} based on Rating_Count Weight.")
        return

    # Filter the DataFrame to include only these top brands
    top_brands_data = df[df['CLEANED BRAND'].isin(top_brands_names)].copy()

    if top_brands_data.empty:
        print(f"No data for products from top brands in {title_suffix}.")
        return

    # Fill NaN values in 'Gissel's L52W $' with 0 before aggregation
    top_brands_data["Gissel's L52W $"] = top_brands_data["Gissel's L52W $"].fillna(0)

    # Aggregate for plotting: Calculate sum of 'Gissel's L52W $' for each top brand
    plot_data = top_brands_data.groupby('CLEANED BRAND')["Gissel's L52W $"].sum().reset_index()

    # Sort the brands by their total sales for better visualization
    plot_data = plot_data.sort_values(by="Gissel's L52W $", ascending=False)

    plt.figure(figsize=(20, 8)) # Increased figure width for longer x-axis
    ax = sns.barplot(x='CLEANED BRAND', y="Gissel's L52W $", data=plot_data, palette='viridis')

    # Add total sales on top of each bar, formatted in millions
    for p in ax.patches:
        ax.annotate(f'${p.get_height()/1000000:.2f}M',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points')

    # Calculate mean and median for the displayed brands' sales
    mean_sales = plot_data["Gissel's L52W $"].mean()
    median_sales = plot_data["Gissel's L52W $"].median()

    # Add mean line (green dashed)
    plt.axhline(mean_sales, color='green', linestyle='--', linewidth=2, label=f'Mean Sales: ${mean_sales/1000000:.2f}M')
    # Add median line (black solid)
    plt.axhline(median_sales, color='black', linestyle='-', linewidth=2, label=f'Median Sales: ${median_sales/1000000:.2f}M')

    plt.title(f"Total Gissel's L52W $ for Top {top_n_brands} Brands by Rating_Count Weight ({title_suffix})")
    plt.xlabel('CLEANED BRAND')
    plt.ylabel("Total Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Ensure data and derived dataframes have 'Rating_Count Weight' column
# Standardize 'CLEANED BRAND' column
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Recalculate 'Rating_x_RatingCount' and 'Rating_Count Weight' for the main 'data' DataFrame
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

# Re-create baby_wipe_data and other_wipe_data to ensure they inherit the new column
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define Count ranges
count_ranges = {
    '1-39 counts': (1, 39),
    '40-79 counts': (40, 79),
    '80-119 counts': (80, 119),
    '120-159 counts': (120, 159),
    '160-199 counts': (160, 199),
    '200+ counts': (200, float('inf')) # Use float('inf') for the upper bound of 200+
}

# Define user segments
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

# Iterate through each user segment
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plotting for {segment_name} ---")

    # Iterate through each Count range
    for range_name, (lower_bound, upper_bound) in count_ranges.items():
        # Filter the segment DataFrame for the current Count range
        if range_name == '200+ counts':
            filtered_df = segment_df[segment_df['Count'] >= lower_bound].copy()
        else:
            filtered_df = segment_df[(segment_df['Count'] >= lower_bound) & (segment_df['Count'] <= upper_bound)].copy()

        if not filtered_df.empty:
            plot_title_suffix = f"{segment_name} ({range_name})";
            plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
        else:
            print(f"No data available for {segment_name} within {range_name}.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd # Ensure pandas is imported as it's used for DataFrame operations

def plot_top_brands_avg_price_per_count(df, title_suffix):
    """
    Generates a bar plot showing the total 'Gissel's L52W $'
    for the top 10% brands by 'Rating_Count Weight' for a given DataFrame.
    Annotations and lines represent mean/median 'Calculated Price/Count or Price/Bundle'.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """
    # Ensure 'Rating_Count Weight' is available (it's created in previous cells)
    if 'Rating_Count Weight' not in df.columns:
        print(f"Error: 'Rating_Count Weight' column not found in {title_suffix} DataFrame.")
        return

    if df.empty:
        print(f"No data available for {title_suffix}.")
        return

    # Calculate the sum of 'Rating_Count Weight' per brand
    brand_rating_weight = df.groupby('CLEANED BRAND')['Rating_Count Weight'].sum()

    # Determine the number of top brands (top 10%)
    num_unique_brands = len(brand_rating_weight)
    top_n_brands = max(1, int(num_unique_brands * 0.10)) # Ensure at least 1 brand if data exists

    # Get the names of the top N brands by Rating_Count Weight
    top_brands_names = brand_rating_weight.nlargest(top_n_brands).index

    if top_brands_names.empty:
        print(f"No top brands found for {title_suffix} based on Rating_Count Weight.")
        return

    # Filter the DataFrame to include only these top brands
    top_brands_data = df[df['CLEANED BRAND'].isin(top_brands_names)].copy()

    if top_brands_data.empty:
        print(f"No data for products from top brands in {title_suffix}.")
        return

    # Fill NaN values in 'Gissel's L52W $' with 0 before aggregation
    top_brands_data["Gissel's L52W $"] = top_brands_data["Gissel's L52W $"].fillna(0)

    # Aggregate for plotting: Calculate sum of 'Gissel's L52W $' and mean 'Calculated Price/Count or Price/Bundle' for each top brand
    plot_data = top_brands_data.groupby('CLEANED BRAND').agg(
        total_gissel_sales=("Gissel's L52W $", 'sum'),
        avg_price_per_count=('Calculated Price/Count or Price/Bundle', 'mean')
    ).reset_index()

    # Sort the brands by their total sales for better visualization
    plot_data = plot_data.sort_values(by="total_gissel_sales", ascending=False)

    plt.figure(figsize=(20, 8)) # Increased figure width for longer x-axis
    ax = sns.barplot(x='CLEANED BRAND', y="total_gissel_sales", data=plot_data, palette='viridis')

    # Add average price/count on top of each bar
    for index, p in enumerate(ax.patches):
        # Get the corresponding avg_price_per_count for the brand represented by the current bar
        brand_name = plot_data['CLEANED BRAND'].iloc[index]
        avg_price_val = plot_data[plot_data['CLEANED BRAND'] == brand_name]['avg_price_per_count'].iloc[0]
        ax.annotate(f'${avg_price_val:.2f}', # Format to 2 decimal places as it's a price/count
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points')

    # Calculate mean and median for 'Calculated Price/Count or Price/Bundle' from the aggregated data
    mean_avg_price_per_count = plot_data['avg_price_per_count'].mean()
    median_avg_price_per_count = plot_data['avg_price_per_count'].median()

    # Add mean line (green dashed)
    plt.axhline(mean_avg_price_per_count, color='green', linestyle='--', linewidth=2, label=f'Mean Avg Price/Count: ${mean_avg_price_per_count:.2f}')
    # Add median line (black solid)
    plt.axhline(median_avg_price_per_count, color='black', linestyle='-', linewidth=2, label=f'Median Avg Price/Count: ${median_avg_price_per_count:.2f}')

    plt.title(f"Total Gissel's L52W $ for Top {top_n_brands} Brands by Rating_Count Weight ({title_suffix})\nAnnotated with Average Price/Count and Mean/Median Price/Count Lines")
    plt.xlabel('CLEANED BRAND')
    plt.ylabel("Total Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Ensure data and derived dataframes have 'Rating_Count Weight' column
# Standardize 'CLEANED BRAND' column
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Recalculate 'Rating_x_RatingCount' and 'Rating_Count Weight' for the main 'data' DataFrame
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

# Re-create baby_wipe_data and other_wipe_data to ensure they inherit the new column
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define TotalCount ranges
total_count_ranges = {
    '1-39 counts': (1, 39),
    '40-79 counts': (40, 79),
    '80-119 counts': (80, 119),
    '120-159 counts': (120, 159),
    '160-199 counts': (160, 199),
    '200+ counts': (200, float('inf')) # Use float('inf') for the upper bound of 200+
}

# Define user segments
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

# Iterate through each user segment
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plotting for {segment_name} ---")

    # Iterate through each TotalCount range
    for range_name, (lower_bound, upper_bound) in total_count_ranges.items():
        # Filter the segment DataFrame for the current TotalCount range
        if range_name == '200+ counts':
            filtered_df = segment_df[segment_df['Count'] >= lower_bound].copy()
        else:
            filtered_df = segment_df[(segment_df['Count'] >= lower_bound) & (segment_df['Count'] <= upper_bound)].copy()

        if not filtered_df.empty:
            plot_title_suffix = f"{segment_name} ({range_name})";
            plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
        else:
            print(f"No data available for {segment_name} within {range_name}.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Re-defining histogram_boxplot here to ensure it's available within this cell's context,
# even though it's defined elsewhere. This is safer if the notebook execution order
# isn't strictly linear or if the agent only focuses on the target cell.
def histogram_boxplot(data, feature, figsize = (12, 7), kde = False, bins = None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows = 2,      # Number of rows of the subplot grid = 2
        sharex = True,  # x-axis will be shared among all subplots
        gridspec_kw = {"height_ratios": (0.25, 0.75)},
        figsize = figsize,
    )                   # Creating the 2 subplots
    sns.boxplot(
        data = data, x = feature, ax = ax_box2, showmeans = True, color = "violet"
    )                   # Boxplot will be created and a star will indicate the mean value of the column

    # Histogram plotting
    sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2, bins = bins, palette = "viridis"
    ) if bins else sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2, palette = "viridis"
    )                   # For histogram

    # Add percentage labels to the histogram bars
    total_count_for_percentages = len(data[feature])
    if total_count_for_percentages > 0: # Avoid division by zero if DataFrame is empty
        for p in ax_hist2.patches:
            height = p.get_height()
            if height > 0: # Only label non-empty bars
                percentage = (height / total_count_for_percentages) * 100
                ax_hist2.annotate(f'{percentage:.1f}%', # Display percentage with one decimal place
                                 (p.get_x() + p.get_width() / 2., height),
                                 ha='center', va='bottom', fontsize=8, color='black',
                                 xytext=(0, 5), textcoords='offset points') # Offset text slightly above bar

    ax_hist2.axvline(
        data[feature].mean(), color = "green", linestyle = "--"
    )                   # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color = "black", linestyle = "-"
    )                   # Add median to the histogram

    # Add data labels for mean and median
    mean_val = data[feature].mean()
    median_val = data[feature].median()

    # Annotate mean
    ax_hist2.text(mean_val + (ax_hist2.get_xlim()[1] - ax_hist2.get_xlim()[0]) * 0.01,
                  ax_hist2.get_ylim()[1] * 0.9,
                  f'Mean: {mean_val:.2f}', color='green', ha='left', va='center')
    # Annotate median
    ax_hist2.text(median_val + (ax_hist2.get_xlim()[1] - ax_hist2.get_xlim()[0]) * 0.01,
                  ax_hist2.get_ylim()[1] * 0.8,
                  f'Median: {median_val:.2f}', color='black', ha='left', va='center')

    # Explicitly set x-axis labels for both subplots
    ax_hist2.set_xlabel(feature)
    ax_box2.set_xlabel(feature)

    # Add custom tick logic for TotalCount at 50 increments
    if feature == 'TotalCount':
        min_count = data[feature].min()
        max_count = data[feature].max()
        # Calculate ticks with 50 increments
        # Ensure that min_count is not negative before floor division
        start_tick = np.floor(min_count / 50) * 50 if min_count >= 0 else 0
        end_tick = np.ceil(max_count / 50) * 50
        ticks = np.arange(start_tick, end_tick + 50, 50)

        # Filter ticks to only include values within a reasonable range of the actual data
        # and ensure no redundant labels if the range is small
        if len(ticks) > 2: # Only apply custom ticks if there's a meaningful range
            ax_hist2.set_xticks(ticks)
            ax_hist2.set_xticklabels([f'{int(t)}' for t in ticks], rotation=45, ha='right')
            ax_box2.set_xticks(ticks)
            ax_box2.set_xticklabels([f'{int(t)}' for t in ticks], rotation=45, ha='right')

        f2.suptitle(f'Distribution of {feature} with 50 Count Increments', fontsize=16)
    else:
        f2.suptitle(f'Distribution of {feature}', fontsize=16)

    plt.show()

# Ensure data and derived dataframes have 'Rating_Count Weight' column
# This recalculation ensures consistency, as previous steps might have altered 'data' or its derivatives.
data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define user segments
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

print("Generating TotalCount distribution for top 50% Rating_Count Weight products:")

for segment_name, segment_df in user_segments.items():
    if segment_df.empty:
        print(f"No data available for {segment_name}.")
        continue

    # Calculate the 50th percentile for 'Rating_Count Weight' in the current segment
    percentile_50_threshold = segment_df['Rating_Count Weight'].quantile(0.50)

    # Filter the DataFrame to include only products in the top 50% by 'Rating_Count Weight'
    top_50_percent_products = segment_df[
        segment_df['Rating_Count Weight'] >= percentile_50_threshold
    ].copy()

    if not top_50_percent_products.empty:
        # Exclude the top 2% of 'TotalCount' values (outliers)
        total_count_98th_percentile = top_50_percent_products['TotalCount'].quantile(0.98)
        filtered_products_no_outliers = top_50_percent_products[
            top_50_percent_products['TotalCount'] <= total_count_98th_percentile
        ].copy()

        if not filtered_products_no_outliers.empty:
            print(f"\n--- TotalCount Distribution for Top 50% Rating_Count Weight Products in {segment_name} (excluding top 2% TotalCount outliers) ---")
            histogram_boxplot(filtered_products_no_outliers, 'TotalCount')
        else:
            print(f"No products remaining after filtering top 50% Rating_Count Weight and excluding top 2% TotalCount outliers for {segment_name}.")
    else:
        print(f"No products found in the top 50% Rating_Count Weight for {segment_name}.")

print("\nFinished generating TotalCount distributions for top 50% Rating_Count Weight products.")

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# import pandas as pd # Ensure pandas is imported as it's used for DataFrame operations

# def plot_top_brands_avg_price_per_count(df, title_suffix):
#     """
#     Generates a bar plot showing the total 'Gissel's L52W $'
#     for all brands for a given DataFrame.
#     Annotations and lines represent mean/median 'Calculated Price/Count or Price/Bundle'.

#     Args:
#         df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
#         title_suffix (str): A string to append to the plot title (e.g., 'All Users').
#     """
#     # Ensure 'Rating_Count Weight' is available (it's created in previous cells)
#     if 'Rating_Count Weight' not in df.columns:
#         print(f"Error: 'Rating_Count Weight' column not found in {title_suffix} DataFrame.")
#         return

#     if df.empty:
#         print(f"No data available for {title_suffix}.")
#         return

#     # Use the entire filtered DataFrame for plotting, no top brand selection
#     processed_df = df.copy()

#     # Fill NaN values in 'Gissel's L52W $' with 0 before aggregation
#     processed_df["Gissel's L52W $"] = processed_df["Gissel's L52W $"].fillna(0)

#     # Aggregate for plotting: Calculate sum of 'Gissel's L52W $' and mean 'Calculated Price/Count or Price/Bundle' for each brand
#     plot_data = processed_df.groupby('CLEANED BRAND').agg(
#         total_gissel_sales=("Gissel's L52W $", 'sum'),
#         avg_price_per_count=('Calculated Price/Count or Price/Bundle', 'mean')
#     ).reset_index()

#     # Sort the brands by their total sales for better visualization
#     plot_data = plot_data.sort_values(by="total_gissel_sales", ascending=False)

#     plt.figure(figsize=(20, 8)) # Increased figure width for longer x-axis
#     ax = sns.barplot(x='CLEANED BRAND', y="total_gissel_sales", data=plot_data, palette='viridis')

#     # Add average price/count on top of each bar
#     for index, p in enumerate(ax.patches):
#         # Get the corresponding avg_price_per_count for the brand represented by the current bar
#         brand_name = plot_data['CLEANED BRAND'].iloc[index]
#         avg_price_val = plot_data[plot_data['CLEANED BRAND'] == brand_name]['avg_price_per_count'].iloc[0]
#         ax.annotate(f'${avg_price_val:.2f}', # Format to 2 decimal places as it's a price/count
#                     (p.get_x() + p.get_width() / 2., p.get_height()),
#                     ha='center', va='center',
#                     xytext=(0, 9),
#                     textcoords='offset points')

#     # Calculate mean and median for 'Calculated Price/Count or Price/Bundle' from the aggregated data
#     mean_avg_price_per_count = plot_data['avg_price_per_count'].mean()
#     median_avg_price_per_count = plot_data['avg_price_per_count'].median()

#     # Add mean line (green dashed)
#     plt.axhline(mean_avg_price_per_count, color='green', linestyle='--', linewidth=2, label=f'Mean Avg Price/Count: ${mean_avg_price_per_count:.2f}')
#     # Add median line (black solid)
#     plt.axhline(median_avg_price_per_count, color='black', linestyle='-', linewidth=2, label=f'Median Avg Price/Count: ${median_avg_price_per_count:.2f}')

#     plt.title(f"Total Gissel's L52W $ for All Brands ({title_suffix})\nAnnotated with Average Price/Count and Mean/Median Price/Count Lines")
#     plt.xlabel('CLEANED BRAND')
#     plt.ylabel("Total Gissel's L52W $")
#     plt.xticks(rotation=45, ha='right')
#     plt.legend()
#     plt.grid(axis='y', linestyle='--', alpha=0.7)
#     plt.tight_layout()
#     plt.show()

# # Ensure data and derived dataframes have 'Rating_Count Weight' column
# # Standardize 'CLEANED BRAND' column
# data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# # Recalculate 'Rating_x_RatingCount' and 'Rating_Count Weight' for the main 'data' DataFrame
# data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
# total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
# data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

# # Re-create baby_wipe_data and other_wipe_data to ensure they inherit the new column
# baby_wipe_data = data[data['User'] == 'Baby'].copy()
# other_wipe_data = data[data['User'] != 'Baby'].copy()

# # Define TotalCount ranges
# total_count_ranges = {
#     '1-39 counts': (1, 39),
#     '40-79 counts': (40, 79),
#     '80-119 counts': (80, 119),
#     '120-159 counts': (120, 159),
#     '160-199 counts': (160, 199),
#     '200+ counts': (200, float('inf')) # Use float('inf') for the upper bound of 200+
# }

# # Define user segments
# user_segments = {
#     'All Users': data,
#     'Baby Users': baby_wipe_data,
#     'Non-Baby Users': other_wipe_data
# }

# # Iterate through each user segment
# for segment_name, segment_df in user_segments.items():
#     print(f"\n--- Plotting for {segment_name} ---")

#     # Iterate through each TotalCount range
#     for range_name, (lower_bound, upper_bound) in total_count_ranges.items():
#         # Filter the segment DataFrame for the current TotalCount range
#         if range_name == '200+ counts':
#             filtered_df = segment_df[segment_df['Count'] >= lower_bound].copy()
#         else:
#             filtered_df = segment_df[(segment_df['Count'] >= lower_bound) & (segment_df['Count'] <= upper_bound)].copy()

#         if not filtered_df.empty:
#             plot_title_suffix = f"{segment_name} ({range_name})";
#             plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
#         else:
#             print(f"No data available for {segment_name} within {range_name}.")

###** 1-39 Count various packs **

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd # Ensure pandas is imported as it's used for DataFrame operations

def plot_top_brands_avg_price_per_count(df, title_suffix):
    """
    Generates a bar plot showing the total 'Gissel's L52W $'
    for the top 50% brands by 'Rating_Count Weight' for a given DataFrame.
    Annotations and lines represent mean/median 'Calculated Price/Count or Price/Bundle'.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """
    # Ensure 'Rating_Count Weight' is available (it's created in previous cells)
    if 'Rating_Count Weight' not in df.columns:
        print(f"Error: 'Rating_Count Weight' column not found in {title_suffix} DataFrame.")
        return

    if df.empty:
        print(f"No data available for {title_suffix}.")
        return

    # Calculate the sum of 'Rating_Count Weight' per brand
    brand_rating_weight = df.groupby('CLEANED BRAND')['Rating_Count Weight'].sum()

    # Determine the number of top brands (top 50% or maximum 50)
    num_unique_brands = len(brand_rating_weight)
    top_n_brands = min(50, max(1, num_unique_brands)) # Changed to limit to 50 brands

    # Get the names of the top N brands by Rating_Count Weight
    top_brands_names = brand_rating_weight.nlargest(top_n_brands).index

    if top_brands_names.empty:
        print(f"No top brands found for {title_suffix} based on Rating_Count Weight.")
        return

    # Filter the DataFrame to include only these top brands
    top_brands_data = df[df['CLEANED BRAND'].isin(top_brands_names)].copy()

    if top_brands_data.empty:
        print(f"No data for products from top brands in {title_suffix}.")
        return

    # Fill NaN values in 'Gissel's L52W $' with 0 before aggregation
    top_brands_data["Gissel's L52W $"] = top_brands_data["Gissel's L52W $"].fillna(0)

    # Aggregate for plotting: Calculate sum of 'Gissel's L52W $' and mean 'Calculated Price/Count or Price/Bundle' for each top brand
    plot_data = top_brands_data.groupby('CLEANED BRAND').agg(
        total_gissel_sales=("Gissel's L52W $", 'sum'),
        avg_price_per_count=('Calculated Price/Count or Price/Bundle', 'mean')
    ).reset_index()

    # Sort the brands by their total sales for better visualization
    plot_data = plot_data.sort_values(by="total_gissel_sales", ascending=False)

    plt.figure(figsize=(20, 8)) # Increased figure width for longer x-axis
    ax = sns.barplot(x='CLEANED BRAND', y="total_gissel_sales", data=plot_data, palette='viridis')

    # Add average price/count on top of each bar
    for index, p in enumerate(ax.patches):
        # Get the corresponding avg_price_per_count for the brand represented by the current bar
        brand_name = plot_data['CLEANED BRAND'].iloc[index]
        avg_price_val = plot_data[plot_data['CLEANED BRAND'] == brand_name]['avg_price_per_count'].iloc[0]
        ax.annotate(f'${avg_price_val:.2f}', # Format to 2 decimal places as it's a price/count
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points')

    # Calculate mean and median for 'Calculated Price/Count or Price/Bundle' from the aggregated data
    mean_avg_price_per_count = plot_data['avg_price_per_count'].mean()
    median_avg_price_per_count = plot_data['avg_price_per_count'].median()

    # Add mean line (green dashed)
    plt.axhline(mean_avg_price_per_count, color='green', linestyle='--', linewidth=2, label=f'Mean Avg Price/Count: ${mean_avg_price_per_count:.2f}')
    # Add median line (black solid)
    plt.axhline(median_avg_price_per_count, color='black', linestyle='-', linewidth=2, label=f'Median Avg Price/Count: ${median_avg_price_per_count:.2f}')

    plt.title(f"Total Gissel's L52W $ for Top {top_n_brands} Brands by Rating_Count Weight ({title_suffix})\nAnnotated with Average Price/Count and Mean/Median Price/Count Lines")
    plt.xlabel('CLEANED BRAND')
    plt.ylabel("Total Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Ensure data and derived dataframes have 'Rating_Count Weight' column
# Standardize 'CLEANED BRAND' column
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Recalculate 'Rating_x_RatingCount' and 'Rating_Count Weight' for the main 'data' DataFrame
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

# Re-create baby_wipe_data and other_wipe_data to ensure they inherit the new column
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define Count ranges for the 'Count' column as requested (1-39)
count_ranges_for_count_plot = {
    'Count: 1-39 counts': (1, 39)
}

# Define Pack ranges for the 'Pack' column as requested (1-20)
# Updated as per user request to separate 1-10 individually and then 11+.
pack_ranges_for_pack_plot = {
    'Pack: 1 pack': (1, 1),
    'Pack: 2 packs': (2, 2),
    'Pack: 3 packs': (3, 3),
    'Pack: 4 packs': (4, 4),
    'Pack: 5 packs': (5, 5),
    'Pack: 6 packs': (6, 6),
    'Pack: 7 packs': (7, 7),
    'Pack: 8 packs': (8, 8),
    'Pack: 9 packs': (9, 9),
    'Pack: 10 packs': (10, 10),
    'Pack: 11+ packs': (11, float('inf')) # All packs 11 and above
}

# Define user segments
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

# Iterate through each user segment for 'Count' ranges
print("\n--- Generating plots for 'Count' column ranges ---")
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plotting for {segment_name} (Count column) ---")

    # Iterate through each Count range
    for range_label, (lower_bound, upper_bound) in count_ranges_for_count_plot.items():
        # Filter the segment DataFrame for the current Count range
        filtered_df = segment_df[(segment_df['Count'] >= lower_bound) & (segment_df['Count'] <= upper_bound)].copy()

        if not filtered_df.empty:
            plot_title_suffix = f"{segment_name} ({range_label})";
            plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
        else:
            print(f"No data available for {segment_name} within {range_label}.")

# Iterate through each user segment for 'Pack' ranges
print("\n--- Generating plots for 'Pack' column ranges ---")
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plotting for {segment_name} (Pack column) ---")

    # First, filter the segment_df for 'Count' between 1 and 39
    filtered_by_count_df = segment_df[(segment_df['Count'] >= 1) & (segment_df['Count'] <= 39)].copy()

    # Iterate through each Pack range on the already filtered DataFrame
    for range_label, (lower_bound, upper_bound) in pack_ranges_for_pack_plot.items():
        # Filter the already count-filtered DataFrame for the current Pack range
        if upper_bound == float('inf'): # For open-ended ranges like 11+ packs
            filtered_df = filtered_by_count_df[filtered_by_count_df['Pack'] >= lower_bound].copy()
        else:
            filtered_df = filtered_by_count_df[(filtered_by_count_df['Pack'] >= lower_bound) & (filtered_by_count_df['Pack'] <= upper_bound)].copy()

        if not filtered_df.empty:
            plot_title_suffix = f"{segment_name} (Count: 1-39 counts, {range_label})";
            plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
        else:
            print(f"No data available for {segment_name} within Count: 1-39 and {range_label}.")

# Iterate through each user segment for 'TotalCount' ranges
# print("\n--- Generating plots for 'TotalCount' column ranges ---")
# for segment_name, segment_df in user_segments.items():
#     print(f"\n--- Plotting for {segment_name} (TotalCount column) ---")

#     # Iterate through each TotalCount range
#     for range_label, (lower_bound, upper_bound) in total_count_ranges_for_totalcount_plot.items():
#         # Filter the segment DataFrame for the current TotalCount range
#         if upper_bound == float('inf'): # For open-ended ranges like 200+
#             filtered_df = segment_df[segment_df['TotalCount'] >= lower_bound].copy()
#         else:
#             filtered_df = segment_df[(segment_df['TotalCount'] >= lower_bound) & (segment_df['TotalCount'] <= upper_bound)].copy()

#         if not filtered_df.empty:
#             plot_title_suffix = f"{segment_name} ({range_label})";
#             plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
#         else:
#             print(f"No data available for {segment_name} within {range_label}.")

### Test Derivation of 'Honest 1-39 count baby user, 1 pack'

In [ ]:
import pandas as pd

# Ensure data is loaded and baby_wipe_data is correctly filtered
# (Assuming `data` and `baby_wipe_data` are already defined and preprocessed from previous cells)
if 'data' not in locals():
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
    data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
    baby_wipe_data = data[data['User'] == 'Baby'].copy()

# Filter for the specific criteria:
# - CLEANED BRAND is 'HONEST'
# - User is 'Baby' (already handled by `baby_wipe_data`)
# - Count is between 1 and 39
# - Pack is 1
honest_baby_filtered_products = baby_wipe_data[
    (baby_wipe_data['CLEANED BRAND'] == 'HONEST') &
    (baby_wipe_data['Count'] >= 1) &
    (baby_wipe_data['Count'] <= 39) &
    (baby_wipe_data['Pack'] == 1)
].copy()

if not honest_baby_filtered_products.empty:
    # Calculate the average 'Calculated Price/Count or Price/Bundle'
    average_price_for_data_point = honest_baby_filtered_products['Calculated Price/Count or Price/Bundle'].mean()

    print(f"Average Calculated Price/Count or Price/Bundle for 'Honest 1-39 count baby user, 1 pack': ${average_price_for_data_point:.2f}")
    print("\nProducts contributing to this average:")
    display(honest_baby_filtered_products[['ProductTitle', 'Count', 'Pack', 'Calculated Price/Count or Price/Bundle']])
else:
    print("No products found matching the criteria for 'Honest 1-39 count baby user, 1 pack'.")

** 40-79 Count, Various packs"

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd # Ensure pandas is imported as it's used for DataFrame operations

def plot_top_brands_avg_price_per_count(df, title_suffix):
    """
    Generates a bar plot showing the total 'Gissel's L52W $'
    for the top 50% brands by 'Rating_Count Weight' for a given DataFrame.
    Annotations and lines represent mean/median 'Calculated Price/Count or Price/Bundle'.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """
    # Ensure 'Rating_Count Weight' is available (it's created in previous cells)
    if 'Rating_Count Weight' not in df.columns:
        print(f"Error: 'Rating_Count Weight' column not found in {title_suffix} DataFrame.")
        return

    if df.empty:
        print(f"No data available for {title_suffix}.")
        return

    # Calculate the sum of 'Rating_Count Weight' per brand
    brand_rating_weight = df.groupby('CLEANED BRAND')['Rating_Count Weight'].sum()

    # Determine the number of top brands (top 10% or maximum 50)
    num_unique_brands = len(brand_rating_weight)
    # Modified to limit to a maximum of 50 brands, or 10% of unique brands if that's less than 50
    top_n_brands = min(50, max(1, int(num_unique_brands * 0.10)))

    # Get the names of the top N brands by Rating_Count Weight
    top_brands_names = brand_rating_weight.nlargest(top_n_brands).index

    if top_brands_names.empty:
        print(f"No top brands found for {title_suffix} based on Rating_Count Weight.")
        return

    # Filter the DataFrame to include only these top brands
    top_brands_data = df[df['CLEANED BRAND'].isin(top_brands_names)].copy()

    if top_brands_data.empty:
        print(f"No data for products from top brands in {title_suffix}.")
        return

    # Fill NaN values in 'Gissel's L52W $' with 0 before aggregation
    top_brands_data["Gissel's L52W $"] = top_brands_data["Gissel's L52W $"].fillna(0)

    # Aggregate for plotting: Calculate sum of 'Gissel's L52W $' and mean 'Calculated Price/Count or Price/Bundle' for each top brand
    plot_data = top_brands_data.groupby('CLEANED BRAND').agg(
        total_gissel_sales=("Gissel's L52W $", 'sum'),
        avg_price_per_count=('Calculated Price/Count or Price/Bundle', 'mean')
    ).reset_index()

    # Sort the brands by their total sales for better visualization
    plot_data = plot_data.sort_values(by="total_gissel_sales", ascending=False)

    plt.figure(figsize=(20, 8)) # Increased figure width for longer x-axis
    ax = sns.barplot(x='CLEANED BRAND', y="total_gissel_sales", data=plot_data, palette='viridis')

    # Add average price/count on top of each bar
    for index, p in enumerate(ax.patches):
        # Get the corresponding avg_price_per_count for the brand represented by the current bar
        brand_name = plot_data['CLEANED BRAND'].iloc[index]
        avg_price_val = plot_data[plot_data['CLEANED BRAND'] == brand_name]['avg_price_per_count'].iloc[0]
        ax.annotate(f'${avg_price_val:.2f}', # Format to 2 decimal places as it's a price/count
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points')

    # Calculate mean and median for 'Calculated Price/Count or Price/Bundle' from the aggregated data
    mean_avg_price_per_count = plot_data['avg_price_per_count'].mean()
    median_avg_price_per_count = plot_data['avg_price_per_count'].median()

    # Add mean line (green dashed)
    plt.axhline(mean_avg_price_per_count, color='green', linestyle='--', linewidth=2, label=f'Mean Avg Price/Count: ${mean_avg_price_per_count:.2f}')
    # Add median line (black solid)
    plt.axhline(median_avg_price_per_count, color='black', linestyle='-', linewidth=2, label=f'Median Avg Price/Count: ${median_avg_price_per_count:.2f}')

    plt.title(f"Total Gissel's L52W $ for Top {top_n_brands} Brands by Rating_Count Weight ({title_suffix})\nAnnotated with Average Price/Count and Mean/Median Price/Count Lines")
    plt.xlabel('CLEANED BRAND')
    plt.ylabel("Total Gissel's L52W $")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# --- Data Loading and Preprocessing (Ensuring 'data' is defined) ---
# This block is added to ensure that 'data' and its derived dataframes are always defined
# when this cell is executed, regardless of previous cell execution order.
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    print("Expected path: /content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
    # Exit or handle the error gracefully, for now, we'll just print and stop further processing in this block.
    raise

# Standardize 'CLEANED BRAND' column
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Recalculate 'Rating_x_RatingCount' and 'Rating_Count Weight' for the main 'data' DataFrame
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

# Re-create baby_wipe_data and other_wipe_data to ensure they inherit the new column
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define Count ranges for the 'Count' column as requested (40-79)
count_ranges_for_count_plot = {
    'Count: 40-79 counts': (40, 79)
}

# Define Pack ranges for the 'Pack' column as requested (1-20)
# Updated as per user request to separate 1-10 individually and then 11+.
pack_ranges_for_pack_plot = {
    'Pack: 1 pack': (1, 1),
    'Pack: 2 packs': (2, 2),
    'Pack: 3 packs': (3, 3),
    'Pack: 4 packs': (4, 4),
    'Pack: 5 packs': (5, 5),
    'Pack: 6 packs': (6, 6),
    'Pack: 7 packs': (7, 7),
    'Pack: 8 packs': (8, 8),
    'Pack: 9 packs': (9, 9),
    'Pack: 10 packs': (10, 10),
    'Pack: 11+ packs': (11, float('inf')) # All packs 11 and above
}

# Define user segments
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

# Iterate through each user segment for 'Count' ranges
print("\n--- Generating plots for 'Count' column ranges ---")
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plotting for {segment_name} (Count column) ---")

    # Iterate through each Count range
    for range_label, (lower_bound, upper_bound) in count_ranges_for_count_plot.items():
        # Filter the segment DataFrame for the current Count range
        filtered_df = segment_df[(segment_df['Count'] >= lower_bound) & (segment_df['Count'] <= upper_bound)].copy()

        if not filtered_df.empty:
            plot_title_suffix = f"{segment_name} ({range_label})";
            plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
        else:
            print(f"No data available for {segment_name} within {range_label}.")

# Iterate through each user segment for 'Pack' ranges
print("\n--- Generating plots for 'Pack' column ranges ---")
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plotting for {segment_name} (Pack column) ---")

    # First, filter the segment_df for 'Count' between 40 and 79
    filtered_by_count_df = segment_df[(segment_df['Count'] >= 40) & (segment_df['Count'] <= 79)].copy()

    # Iterate through each Pack range on the already filtered DataFrame
    for range_label, (lower_bound, upper_bound) in pack_ranges_for_pack_plot.items():
        # Filter the already count-filtered DataFrame for the current Pack range
        if upper_bound == float('inf'): # For open-ended ranges like 11+ packs
            filtered_df = filtered_by_count_df[filtered_by_count_df['Pack'] >= lower_bound].copy()
        else:
            filtered_df = filtered_by_count_df[(filtered_by_count_df['Pack'] >= lower_bound) & (filtered_by_count_df['Pack'] <= upper_bound)].copy()

        if not filtered_df.empty:
            plot_title_suffix = f"{segment_name} (Count: 40-79 counts, {range_label})";
            plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
        else:
            print(f"No data available for {segment_name} within Count: 40-79 and {range_label}.")

# # Iterate through each user segment for 'TotalCount' ranges
# # print("\n--- Generating plots for 'TotalCount' column ranges ---")
# # for segment_name, segment_df in user_segments.items():
# #     print(f"\n--- Plotting for {segment_name} (TotalCount column) ---")

# #     # Iterate through each TotalCount range
# #     for range_label, (lower_bound, upper_bound) in total_count_ranges_for_totalcount_plot.items():
# #         # Filter the segment DataFrame for the current TotalCount range
# #         if upper_bound == float('inf'): # For open-ended ranges like 200+
# #             filtered_df = segment_df[segment_df['TotalCount'] >= lower_bound].copy()
# #         else:
# #             filtered_df = segment_df[(segment_df['TotalCount'] >= lower_bound) & (segment_df['TotalCount'] <= upper_bound)].copy()

# #         if not filtered_df.empty:
# #             plot_title_suffix = f"{segment_name} ({range_label})";
# #             plot_top_brands_avg_price_per_count(filtered_df, plot_title_suffix)
# #         else:
# #             print(f"No data available for {segment_name} within {range_label}.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd # Ensure pandas is imported as it's used for DataFrame operations
import matplotlib.ticker as mticker # Import for ScalarFormatter

def plot_brands_gissel_units(df, title_suffix):
    """
    Generates a bar plot showing 'Total Gissel's L52W Units'
    by 'CLEANED BRAND' for all brands in a given DataFrame.

    The y-axis ('Total Gissel's L52W Units') is formatted as plain numbers without currency symbols.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """
    # Ensure 'Gissel's Units' is available
    if "Gissel's Units" not in df.columns:
        print(f"Error: 'Gissel's Units' column not found in {title_suffix} DataFrame. Please ensure data is loaded with this column.")
        return

    if df.empty:
        print(f"No data available for {title_suffix}.")
        return

    # Fill NaN values in 'Gissel's Units' with 0 before aggregation for consistent calculation
    df_copy = df.copy()
    df_copy["Gissel's Units"] = df_copy["Gissel's Units"].fillna(0)

    # Aggregate for plotting: Calculate sum of 'Gissel's Units' for each brand
    plot_data = df_copy.groupby('CLEANED BRAND').agg(
        total_gissel_units=("Gissel's Units", 'sum')
    ).reset_index()

    # Sort by total_gissel_units for better visualization
    plot_data = plot_data.sort_values(by="total_gissel_units", ascending=False)

    # Filter to show only brands with non-zero total_gissel_units
    plot_data = plot_data[plot_data['total_gissel_units'] > 0]

    if plot_data.empty:
        print(f"No brands with Gissel's L52W Units > 0 found for {title_suffix}. Skipping plot.")
        return

    plt.figure(figsize=(15, 9)) # Adjust figure size for bar plot
    ax = sns.barplot(
        x='CLEANED BRAND',
        y='total_gissel_units',
        data=plot_data,
        palette='viridis'
    )

    # Calculate mean and median for total_gissel_units
    mean_total_units = plot_data['total_gissel_units'].mean()
    median_total_units = plot_data['total_gissel_units'].median()

    # Add mean line (green dashed)
    plt.axhline(mean_total_units, color='green', linestyle='--', linewidth=2, label=f'Mean Total Units: {mean_total_units:,.0f}')
    # Add median line (black solid)
    plt.axhline(median_total_units, color='black', linestyle='-', linewidth=2, label=f'Median Total Units: {median_total_units:,.0f}')

    plt.title(f"Total Gissel's L52W Units by Brand ({title_suffix})")
    plt.xlabel('CLEANED BRAND')
    plt.ylabel("Total Gissel's L52W Units") # Y-axis label

    # Set y-axis tick formatter to plain numbers with comma separators
    formatter = mticker.FuncFormatter(lambda x, p: f'{int(x):,}')
    ax.yaxis.set_major_formatter(formatter)

    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
    plt.legend(loc='upper right') # Show legend for mean and median lines
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

##40-79 Count, Various packs Total Units

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd # Ensure pandas is imported as it's used for DataFrame operations
import matplotlib.ticker as mticker # Import for ScalarFormatter
from google.colab import files # Import for file downloads

def plot_bottom_brands_gissel_units(df, title_suffix):
    """
    Generates a bar plot showing 'Total Gissel's L52W Units'
    by 'CLEANED BRAND' for the bottom 20% brands by 'Gissel's Units' for a given DataFrame.
    Includes mean and median horizontal lines for total units.
    Provides a downloadable CSV of the plot data.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., data, baby_wipe_data, other_wipe_data).
        title_suffix (str): A string to append to the plot title (e.g., 'All Users').
    """
    # Ensure 'Gissel's Units' is available
    if "Gissel's Units" not in df.columns:
        print(f"Error: 'Gissel's Units' column not found in {title_suffix} DataFrame. Please ensure data is loaded with this column.")
        return

    if df.empty:
        print(f"No data available for {title_suffix}.")
        return

    # Fill NaN values in 'Gissel's Units' with 0 before aggregation for consistent ranking
    df_copy = df.copy()
    df_copy["Gissel's Units"] = df_copy["Gissel's Units"].fillna(0)

    # Calculate the sum of 'Gissel's Units' per brand
    brand_sales_units = df_copy.groupby('CLEANED BRAND')["Gissel's Units"].sum()

    # Determine the number of bottom brands (bottom 20% or maximum 50)
    num_unique_brands = len(brand_sales_units)
    # Changed to bottom 20%, up to a maximum of 50 brands
    bottom_n_brands = min(50, max(1, int(num_unique_brands * 0.20)))

    # Get the names of the bottom N brands by Gissel's Units
    # Using nsmallest for bottom N
    bottom_brands_names = brand_sales_units.nsmallest(bottom_n_brands).index

    if bottom_brands_names.empty:
        print(f"No bottom brands found for {title_suffix} based on Gissel's Units.")
        return

    # Filter the DataFrame to include only these selected brands
    selected_brands_data = df_copy[df_copy['CLEANED BRAND'].isin(bottom_brands_names)].copy()

    if selected_brands_data.empty:
        print(f"No data for products from selected brands in {title_suffix}.")
        return

    # Aggregate for plotting: Calculate sum of 'Gissel's Units' for each brand
    plot_data = selected_brands_data.groupby('CLEANED BRAND').agg(
        total_gissel_units=("Gissel's Units", 'sum')
    ).reset_index()

    # Sort by total_gissel_units for better visualization (ascending for bottom brands)
    plot_data = plot_data.sort_values(by="total_gissel_units", ascending=True)

    # Filter to show only brands with non-zero total_gissel_units
    plot_data = plot_data[plot_data['total_gissel_units'] > 0]

    if plot_data.empty:
        print(f"No brands with Gissel's L52W Units > 0 found for {title_suffix} after filtering for bottom brands. Skipping plot.")
        return

    plt.figure(figsize=(15, 9)) # Adjust figure size for bar plot
    ax = sns.barplot(
        x='CLEANED BRAND',
        y='total_gissel_units',
        data=plot_data,
        palette='viridis'
    )

    # Calculate mean and median for 'total_gissel_units' from the aggregated data
    mean_total_units = plot_data['total_gissel_units'].mean()
    median_total_units = plot_data['total_gissel_units'].median()

    # Add mean line (green dashed)
    plt.axhline(mean_total_units, color='green', linestyle='--', linewidth=2, label=f'Mean Total Units: {mean_total_units:,.0f}')
    # Add median line (black solid)
    plt.axhline(median_total_units, color='black', linestyle='-', linewidth=2, label=f'Median Total Units: {median_total_units:,.0f}')

    plt.title(f"Total Gissel's L52W Units by Brand for Bottom {len(plot_data)} Brands ({title_suffix})")
    plt.xlabel('CLEANED BRAND')
    plt.ylabel("Total Gissel's L52W Units") # Y-axis label

    # Set y-axis tick formatter to plain numbers (not currency) with comma separators
    formatter = mticker.FuncFormatter(lambda x, p: f'{int(x):,}')
    ax.yaxis.set_major_formatter(formatter)

    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
    plt.legend(loc='upper right') # Place legend in the upper right
    plt.grid(axis='y', linestyle='--', alpha=0.7) # Add grid to y-axis for bar plot
    plt.tight_layout() # Adjust layout to make space for labels and annotations
    plt.show()

    # Save plot data to CSV
    csv_filename = f"bottom_brands_gissel_units_{title_suffix.replace(' ', '_').lower()}.csv"
    plot_data.to_csv(csv_filename, index=False)
    files.download(csv_filename)
    print(f"'{csv_filename}' has been generated and is ready for download.")

# --- Data Loading and Preprocessing (Ensuring 'data' is defined) ---
# This block is added to ensure that 'data' and its derived dataframes are always defined
# when this cell is executed, regardless of previous cell execution order.
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    print("Expected path: /content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
    # Exit or handle the error gracefully, for now, we'll just print and stop further processing in this block.
    raise

# Standardize 'CLEANED BRAND' column
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

# Recalculate 'Rating_x_RatingCount' and 'Rating_Count Weight' for the main 'data' DataFrame
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

# Re-create baby_wipe_data and other_wipe_data to ensure they inherit the new column
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define Count ranges for the 'Count' column as requested (40-79)
count_ranges_for_count_plot = {
    'Count: 40-79 counts': (40, 79)
}

# Define Pack ranges for the 'Pack' column as requested (1-20)
# Updated as per user request to separate 1-10 individually and then 11+.
pack_ranges_for_pack_plot = {
    'Pack: 1 pack': (1, 1),
    'Pack: 2 packs': (2, 2),
    'Pack: 3 packs': (3, 3),
    'Pack: 4 packs': (4, 4),
    'Pack: 5 packs': (5, 5),
    'Pack: 6 packs': (6, 6),
    'Pack: 7 packs': (7, 7),
    'Pack: 8 packs': (8, 8),
    'Pack: 9 packs': (9, 9),
    'Pack: 10 packs': (10, 10),
    'Pack: 11+ packs': (11, float('inf')) # All packs 11 and above
}

# Define user segments
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

# Iterate through each user segment for 'Count' ranges
print("\n--- Generating plots for 'Count' column ranges ---")
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plotting for {segment_name} (Count column) ---")

    # Iterate through each Count range
    for range_label, (lower_bound, upper_bound) in count_ranges_for_count_plot.items():
        # Filter the segment DataFrame for the current Count range
        filtered_df = segment_df[(segment_df['Count'] >= lower_bound) & (segment_df['Count'] <= upper_bound)].copy()

        if not filtered_df.empty:
            plot_title_suffix = f"{segment_name} ({range_label})";
            plot_bottom_brands_gissel_units(filtered_df, plot_title_suffix)
        else:
            print(f"No data available for {segment_name} within {range_label}.")

# Iterate through each user segment for 'Pack' ranges
print("\n--- Generating plots for 'Pack' column ranges ---")
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plotting for {segment_name} (Pack column) ---")

    # First, filter the segment_df for 'Count' between 40 and 79
    filtered_by_count_df = segment_df[(segment_df['Count'] >= 40) & (segment_df['Count'] <= 79)].copy()

    # Iterate through each Pack range on the already filtered DataFrame
    for range_label, (lower_bound, upper_bound) in pack_ranges_for_pack_plot.items():
        # Filter the already count-filtered DataFrame for the current Pack range
        if upper_bound == float('inf'): # For open-ended ranges like 11+ packs
            filtered_df = filtered_by_count_df[filtered_by_count_df['Pack'] >= lower_bound].copy()
        else:
            filtered_df = filtered_by_count_df[(filtered_by_count_df['Pack'] >= lower_bound) & (filtered_by_count_df['Pack'] <= upper_bound)].copy()

        if not filtered_df.empty:
            plot_title_suffix = f"{segment_name} (Count: 40-79 counts, {range_label})";
            plot_bottom_brands_gissel_units(filtered_df, plot_title_suffix)
        else:
            print(f"No data available for {segment_name} within Count: 40-79 and {range_label}.")


In [ ]:
import pandas as pd
import numpy as np

# --- Ensure dataframes are loaded and preprocessed for consistent calls ---
# This block ensures 'data', 'baby_wipe_data', and 'other_wipe_data' are properly initialized
# assuming the excel file is in the specified Google Drive path.
try:
    # If 'data' is not already in locals, load it. Otherwise, assume it's already processed.
    if 'data' not in locals():
        data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
        data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
        # Recalculate 'Rating_x_RatingCount' and 'Rating_Count Weight' if they are needed for filtering/ordering not implemented here.
        data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
        total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
        data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

    # Ensure baby_wipe_data and other_wipe_data are defined
    if 'baby_wipe_data' not in locals():
        baby_wipe_data = data[data['User'] == 'Baby'].copy()
    if 'other_wipe_data' not in locals():
        other_wipe_data = data[data['User'] != 'Baby'].copy()

except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    print("Expected path: /content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
    raise # Re-raise the error to stop execution if data is critical

# Define Count ranges
count_ranges = {
    '1-39 counts': (1, 39),
    '40-79 counts': (40, 79),
    '80-119 counts': (80, 119),
    '120-159 counts': (120, 159),
    '160-199 counts': (160, 199),
    '200+ counts': (200, float('inf')) # Use float('inf') for the upper bound of 200+
}

# Define Pack ranges (as per previous cells, breaking down 1-10 individually)
pack_ranges = {
    '1 pack': (1, 1),
    '2 packs': (2, 2),
    '3 packs': (3, 3),
    '4 packs': (4, 4),
    '5 packs': (5, 5),
    '6 packs': (6, 6),
    '7 packs': (7, 7),
    '8 packs': (8, 8),
    '9 packs': (9, 9),
    '10 packs': (10, 10),
    '11+ packs': (11, float('inf')) # All packs 11 and above
}

# Define user segments
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

print("Generating 'Total Gissel's L52W Units' bar charts...")

for segment_name, segment_df in user_segments.items():
    print(f"\n--- Processing for {segment_name} ---")

    if segment_df.empty:
        print(f"No data available for {segment_name}. Skipping.")
        continue

    # Iterate through each Count range
    for count_range_name, (count_lower_bound, count_upper_bound) in count_ranges.items():
        # Filter by Count range
        if count_upper_bound == float('inf'):
            filtered_by_count_df = segment_df[segment_df['Count'] >= count_lower_bound].copy()
        else:
            filtered_by_count_df = segment_df[(segment_df['Count'] >= count_lower_bound) & (segment_df['Count'] <= count_upper_bound)].copy()

        if filtered_by_count_df.empty:
            print(f"No data for {segment_name} within Count {count_range_name}. Skipping Pack range analysis for this Count.")
            continue

        # Now, for each Count range, iterate through Pack ranges
        for pack_range_name, (pack_lower_bound, pack_upper_bound) in pack_ranges.items():
            # Filter by Pack range (on the already count-filtered DataFrame)
            if pack_upper_bound == float('inf'):
                final_filtered_df = filtered_by_count_df[filtered_by_count_df['Pack'] >= pack_lower_bound].copy()
            else:
                final_filtered_df = filtered_by_count_df[(filtered_by_count_df['Pack'] >= pack_lower_bound) & (filtered_by_count_df['Pack'] <= pack_upper_bound)].copy()

            if not final_filtered_df.empty:
                title = f"{segment_name} (Count: {count_range_name}, Pack: {pack_range_name})";
                plot_brands_gissel_units(final_filtered_df, title)
            else:
                print(f"No data for {segment_name} within Count {count_range_name} and Pack {pack_range_name}.")

print("Finished generating all 'Total Gissel's L52W Units' bar charts.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Ensure dataframes are loaded and preprocessed ---
# This block ensures 'data', 'baby_wipe_data', and 'other_wipe_data' are properly initialized
# assuming the excel file is in the specified Google Drive path.
try:
    if 'data' not in locals(): # Check if 'data' is already defined
        data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
        data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
        data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
        total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
        data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

    if 'baby_wipe_data' not in locals():
        baby_wipe_data = data[data['User'] == 'Baby'].copy()
    if 'other_wipe_data' not in locals():
        other_wipe_data = data[data['User'] != 'Baby'].copy()

except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    print("Expected path: /content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
    raise # Re-raise the error to stop execution if data is critical


# 1. Define keywords for "individually wrapped" and its synonyms
individually_wrapped_keywords = [
    'individually wrapped',
    'single wrap',
    'single-wrapped',
    'each wrapped',
    'pocket size',
    'travel size'
    # Add more synonyms as needed based on data exploration
]

# Function to categorize products based on keywords in 'Features'
def categorize_individually_wrapped(df):
    df_copy = df.copy()
    # Convert 'Features' to string to handle potential non-string entries and NaN
    df_copy['Features_lower'] = df_copy['Features'].astype(str).str.lower()

    def check_keywords(features_str):
        for keyword in individually_wrapped_keywords:
            if keyword in features_str:
                return 'Individually Wrapped'
        return 'Not Individually Wrapped'

    df_copy['Individually_Wrapped_Status'] = df_copy['Features_lower'].apply(check_keywords)
    df_copy.drop(columns=['Features_lower'], inplace=True)
    return df_copy

# Apply categorization to all user segments
data_categorized = categorize_individually_wrapped(data)
baby_wipe_data_categorized = categorize_individually_wrapped(baby_wipe_data)
other_wipe_data_categorized = categorize_individually_wrapped(other_wipe_data)


# Define user segments for iteration
user_segments = {
    'All Users': data_categorized,
    'Baby Users': baby_wipe_data_categorized,
    'Non-Baby Users': other_wipe_data_categorized
}

print("--- Analysis of 'Individually Wrapped' Feature ---")

for segment_name, segment_df in user_segments.items():
    if segment_df.empty:
        print(f"No data available for {segment_name}.")
        continue

    print(f"\n--- {segment_name} ---")

    # 2. Count comparison
    count_distribution = segment_df['Individually_Wrapped_Status'].value_counts().reset_index()
    count_distribution.columns = ['Individually_Wrapped_Status', 'Count']

    print(f"Product Count Distribution for {segment_name}:")
    display(count_distribution)

    plt.figure(figsize=(8, 5))
    sns.barplot(x='Individually_Wrapped_Status', y='Count', data=count_distribution, palette='coolwarm')
    plt.title(f'Count of Individually Wrapped vs. Not ({segment_name})')
    plt.xlabel('Individually Wrapped Status')
    plt.ylabel('Number of Products')
    plt.tight_layout()
    plt.show()

    # 3. Price comparison: Calculated Price/Count or Price/Bundle
    price_per_unit_comparison = segment_df.groupby('Individually_Wrapped_Status')['Calculated Price/Count or Price/Bundle'].mean().reset_index()
    print(f"\nAverage Calculated Price/Count or Price/Bundle for {segment_name}:")
    display(price_per_unit_comparison)

    plt.figure(figsize=(8, 5))
    sns.barplot(x='Individually_Wrapped_Status', y='Calculated Price/Count or Price/Bundle', data=price_per_unit_comparison, palette='rocket')
    plt.title(f'Avg. Price/Count or Price/Bundle: Individually Wrapped vs. Not ({segment_name})')
    plt.xlabel('Individually Wrapped Status')
    plt.ylabel('Average Calculated Price/Count or Price/Bundle')
    plt.tight_layout()
    plt.show()

    # 4. Price comparison: CurrentPrice
    current_price_comparison = segment_df.groupby('Individually_Wrapped_Status')['CurrentPrice'].mean().reset_index()
    print(f"\nAverage CurrentPrice for {segment_name}:")
    display(current_price_comparison)

    plt.figure(figsize=(8, 5))
    sns.barplot(x='Individually_Wrapped_Status', y='CurrentPrice', data=current_price_comparison, palette='mako')
    plt.title(f'Average CurrentPrice: Individually Wrapped vs. Not ({segment_name})')
    plt.xlabel('Individually Wrapped Status')
    plt.ylabel('Average CurrentPrice ($)')
    plt.tight_layout()
    plt.show()


## **Bivariate Analysis**

### **1. Scatter plot**
A scatter plot allows us to see relationships between two variables.

**Note: Use log transformed values 'kilometers_driven_log' and 'price_log'**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.ticker as mticker # Import for custom ticker formatting

# --- Ensure dataframes are loaded and preprocessed ---
# This block ensures 'data', 'baby_wipe_data', and 'other_wipe_data' are properly initialized
# assuming the excel file is in the specified Google Drive path.
try:
    # Load data if not already loaded (check for 'data' in locals() to avoid reloading)
    if 'data' not in locals():
        data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
        data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()

    # Ensure derived dataframes exist or recreate them
    if 'baby_wipe_data' not in locals():
        baby_wipe_data = data[data['User'] == 'Baby'].copy()
    if 'other_wipe_data' not in locals():
        other_wipe_data = data[data['User'] != 'Baby'].copy()

except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    print("Expected path: /content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
    raise # Re-raise the error to stop execution if data is critical

# Define the variables to plot against 'Gissel's L52W $'
x_variables = ['Count', 'Pack', 'TotalCount', 'Rating', 'RatingCount', 'Rating_Count Weight']

# Define user segments for iteration
user_segments = {
    'All Users': data.copy(),
    'Baby Users': baby_wipe_data.copy(),
    'Non-Baby Users': other_wipe_data.copy()
}

# New function to set ticks and labels for log-transformed axes
def set_log_axis_ticks(ax, data_series_log, original_label, is_currency=False):
    # Calculate ticks based on the log scale range
    min_log = data_series_log.min()
    max_log = data_series_log.max()
    # Choose a reasonable number of ticks, e.g., 5-7 ticks
    num_ticks = 7
    log_ticks = np.linspace(min_log, max_log, num_ticks)

    # Convert log ticks back to original scale for display
    original_ticks = np.expm1(log_ticks)

    # Format labels
    if is_currency:
        # For currency, use dollar format, potentially in millions for large numbers
        labels = []
        for val in original_ticks:
            if val >= 1_000_000:
                labels.append(f"${val/1_000_000:.1f}M")
            elif val >= 1_000:
                labels.append(f"${val/1_000:.0f}K")
            else:
                labels.append(f"${val:.0f}")
    else:
        # For other numeric values, format as integers or with one decimal
        labels = [f"{val:.0f}" if val >= 1 else f"{val:.1f}" for val in original_ticks]

    ax.set_xticks(log_ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right')

# Define a similar function for the y-axis
def set_log_y_axis_ticks(ax, data_series_log, original_label, is_currency=False):
    # Calculate ticks based on the log scale range
    min_log = data_series_log.min()
    max_log = data_series_log.max()
    num_ticks = 7
    log_ticks = np.linspace(min_log, max_log, num_ticks)
    original_ticks = np.expm1(log_ticks)

    if is_currency:
        labels = []
        for val in original_ticks:
            if val >= 1_000_000:
                labels.append(f"${val/1_000_000:.1f}M")
            elif val >= 1_000:
                labels.append(f"${val/1_000:.0f}K")
            else:
                labels.append(f"${val:.0f}")
    else:
        labels = [f"{val:.0f}" if val >= 1 else f"{val:.1f}" for val in original_ticks]

    ax.set_yticks(log_ticks)
    ax.set_yticklabels(labels)


# Function to plot scatter plots with log transformation
def plot_gissel_sales_scatter_log(df, segment_name, x_col):
    df_plot = df.copy()

    # Fill NaN values for 'Gissel's L52W $' and x_col with 0 before log transformation
    # Log1p handles 0 by returning 0, which is good for visualization of "no sales" or "no count"
    df_plot["Gissel's L52W $"] = df_plot["Gissel's L52W $"].fillna(0)
    df_plot[x_col] = df_plot[x_col].fillna(0) # Also fillna for x_col if it could have NaNs

    # Apply log1p transformation
    df_plot[f'Log_{x_col}'] = np.log1p(df_plot[x_col])
    df_plot[f'Log_Gissel_L52W_$'] = np.log1p(df_plot["Gissel's L52W $"])

    if df_plot.empty:
        print(f"No data available for {segment_name} for {x_col}.")
        return

    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    sns.scatterplot(data=df_plot, x=f'Log_{x_col}', y=f'Log_Gissel_L52W_$', alpha=0.7, ax=ax)
    plt.title(f"Log(Gissel's L52W $) vs. Log({x_col}) ({segment_name})")
    plt.xlabel(f"Log({x_col})")
    plt.ylabel(f"Log(Gissel's L52W $)")

    # Set x-axis ticks to display original values. is_currency is always False for x_variables.
    set_log_axis_ticks(ax, df_plot[f'Log_{x_col}'], x_col, is_currency=False)
    # Set y-axis ticks to display original currency values
    set_log_y_axis_ticks(ax, df_plot[f'Log_Gissel_L52W_$'], "Gissel's L52W $", is_currency=True)

    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Generate plots for each segment and each x_variable using log transformation
print("Generating scatter plots for Log(Gissel's L52W $) against Log(various features)...")
for segment_name, segment_df in user_segments.items():
    print(f"\n--- Plots for {segment_name} ---")
    for x_col in x_variables:
        # Ensure 'Rating_x_RatingCount' is available for calculating 'Rating_Count Weight'
        if 'Rating_x_RatingCount' not in segment_df.columns:
            segment_df['Rating_x_RatingCount'] = segment_df['Rating'] * segment_df['RatingCount']
            total_rating_count_sumproduct = segment_df['Rating_x_RatingCount'].sum()
            segment_df['Rating_Count Weight'] = segment_df['Rating_x_RatingCount'] / total_rating_count_sumproduct

        plot_gissel_sales_scatter_log(segment_df, segment_name, x_col)

print("Finished generating all requested scatter plots with log transformations.")

Conclusions: Among the numerical variables Count, Pack, TotalCount, Rating, RatingCount tend to have a positive relationship to the $L52W \$

### **Heat map**
Heat map shows a 2D correlation matrix between two numerical features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Assuming 'data', 'baby_wipe_data', and 'other_wipe_data' are already defined
# from previous cells in the notebook.
# If not, ensure they are loaded/created here for standalone execution.

# Define user segments (using existing dataframes from the notebook state)
user_segments = {
    'All Users': data,
    'Baby Users': baby_wipe_data,
    'Non-Baby Users': other_wipe_data
}

for segment_name, segment_df in user_segments.items():
    if segment_df.empty:
        print(f"No data available for {segment_name}. Skipping heatmap.")
        continue

    # Select only numerical columns from the DataFrame for the current segment
    # This will automatically exclude the newly added log-transformed columns if they are not numerical,
    # or include them if they are numerical and relevant for correlation.
    numerical_data = segment_df.select_dtypes(include=np.number)

    # Drop columns that are entirely NaN or have too many unique values to be meaningful in a correlation matrix
    # For instance, if 'Gissel's L52W $' is all NaN for a segment, its correlation won't be useful.
    # Also, some log-transformed columns might have been created with NaNs if the original was NaN and was not filled.
    numerical_data = numerical_data.dropna(axis=1, how='all')

    # Remove columns with very low variance (e.g., if a column becomes all 0s after filtering)
    numerical_data = numerical_data.loc[:, numerical_data.nunique() > 1]

    if numerical_data.empty:
        print(f"No numerical data available for {segment_name} after filtering. Skipping heatmap.")
        continue

    # Calculate the correlation matrix
    correlation_matrix = numerical_data.corr()

    # Plot the heatmap
    plt.figure(figsize=(16, 12)) # Adjusted figure size for better readability
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title(f'Correlation Matrix of Numerical Variables ({segment_name})', fontsize=18)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

### **Split the Data**

## **Building Various Models**

1. What we want to predict is the "Price". We will use the normalized version 'price_log' for modeling.
2. Before we proceed to the model, we'll have to encode categorical features. We will drop categorical features like Name.
3. We'll split the data into train and test, to be able to evaluate the model that we build on the train data.
4. Build Regression models using train data.
5. Evaluate the model performance.


### **Split the Data**
<li>Step1: Seperating the indepdent variables (X) and the dependent variable (y).
<li>Step2: Encode the categorical variables in X using pd.dummies.
<li>Step3: Split the data into train and test using train_test_split.

In [ ]:
df_model = data.copy()

In [ ]:
columns_to_drop_from_X = [
    'ASIN', 'ProductTitle',
    'Count', 'Pack', 'TotalCount',
    'CurrentPrice', 'Calculated Price/Count or Price/Bundle',
    "Gissel's L52W $", "Gissel's YAG $", "Gissel's Units",

    'CurrentPrice_log',
    'Calculated Price/Count or Price/Bundle_log' # This is our target variable 'y'
]

In [ ]:
# # Step-2 Use pd.get_dummies(drop_first = True)
# X = pd.get_dummies(X, drop_first = True)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

# Make a copy of the data for modeling
df_model = data.copy()

# Create the log-transformed target variable
df_model['Calculated Price/Count or Price/Bundle_log'] = np.log1p(df_model['Calculated Price/Count or Price/Bundle'])

# Create log-transformed versions of other numerical features on df_model
df_model['Rating_x_RatingCount_log'] = np.log1p(df_model['Rating_x_RatingCount'])
df_model['Rating_Count_Weight_log'] = np.log1p(df_model['Rating_Count Weight'])
df_model['CurrentPrice_log'] = np.log1p(df_model['CurrentPrice'])

# Step 1: Separating the independent variables (X) and the dependent variable (y)
# The dependent variable (y) is the log-transformed 'Calculated Price/Count or Price/Bundle'
y = df_model['Calculated Price/Count or Price/Bundle_log']

# Identify columns to drop from X:
# - Identifiers: ASIN, ProductTitle
# - Original numerical columns that have log-transformed versions being used or are directly related to y:
#   Count, Pack, TotalCount (use TotalCount_log instead, assuming it will be created or is already in data)
#   CurrentPrice, Calculated Price/Count or Price/Bundle (original price, y is log-transformed version)
# - Sales-related metrics: Gissel's L52W $, Gissel's YAG $, Gissel's Units (target leakage for price prediction)
# - Redundant/derived rating metrics: Rating_x_RatingCount, Rating_Count Weight
# - CurrentPrice_log (highly correlated with y: Calculated Price/Count or Price/Bundle_log)

columns_to_drop_from_X = [
    'ASIN', 'ProductTitle',
    'Count', 'Pack', 'TotalCount',
    'CurrentPrice', 'Calculated Price/Count or Price/Bundle',
    "Gissel's L52W $", "Gissel's YAG $", "Gissel's Units",
    'CurrentPrice_log',
    'Calculated Price/Count or Price/Bundle_log' # This is our target variable 'y'
]

X_raw = df_model.drop(columns=columns_to_drop_from_X, errors='ignore') # Use errors='ignore' for robustness

# Handle NaN values in 'XL' and 'Flushable' before one-hot encoding
X_raw['XL'] = X_raw['XL'].fillna('Non-XL')
X_raw['Flushable'] = X_raw['Flushable'].fillna('Non-Flushable')

# Identify categorical columns. Explicitly include 'CLEANED BRAND' for one-hot encoding.
categorical_cols = X_raw.select_dtypes(include=['object']).columns.tolist()

# Remove 'CLEANED BRAND' from columns to drop if it was implicitly added and needs to be kept
# No need to remove 'CLEANED BRAND' if it's not in columns_to_drop_from_X. The previous step
# already correctly drops the specified columns.

# Apply one-hot encoding with drop_first=True as requested
X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True)

# Ensure 'TotalCount_log' is present in X, if it was intended to be included as a feature
# It might have been dropped if it was numeric and not explicitly kept, or if it wasn't in df_model
# Re-add it if it's missing and intended for X
if 'TotalCount_log' not in X.columns and 'TotalCount_log' in df_model.columns:
    X['TotalCount_log'] = df_model['TotalCount_log']

# Step 3: Split the data into train and test using train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data preparation complete!")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# Display the first few rows of X and y to confirm
print("\nX (features) head:")

# Reorder columns for display: 'CLEANED BRAND' columns to the end
cleaned_brand_cols = [col for col in X.columns if col.startswith('CLEANED BRAND_')]
other_cols = [col for col in X.columns if not col.startswith('CLEANED BRAND_')]
new_column_order = other_cols + cleaned_brand_cols
display(X[new_column_order].head())

print("\ny (target) head:")
display(y.head())

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor

# Bagging, also kown as bootstrap aggregation, is the ensemble learning method that is commonly used to reduce variance within a noisy dataset.
from sklearn.model_selection import GridSearchCV
# GridSearchCV is an internal cross-validation technique which is used to calculate the scope for each combination of parameters on the grid.
# Mean Absolute Error (MAE) is the sum of the absolute difference between actual and predicted values.
# Mean Squared Error (MSE) is defined as the mean or average of the square of the difference between actual and estimated values.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def get_model_score(model, flag = True):
    '''
    model : regressor to predict values of X

    '''
    # Defining an empty list to store train and test results
    score_list = []

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    # If flag is True, convert predictions and actuals back to original scale
    if flag:
        pred_train_transformed = np.expm1(pred_train) # Inverse of log1p
        pred_test_transformed = np.expm1(pred_test) # Inverse of log1p
        y_train_transformed = np.expm1(y_train) # Inverse of log1p for actuals
        y_test_transformed = np.expm1(y_test) # Inverse of log1p for actuals
    else:
        pred_train_transformed = pred_train
        pred_test_transformed = pred_test
        y_train_transformed = y_train
        y_test_transformed = y_test

    train_r2 = r2_score(y_train_transformed, pred_train_transformed)
    test_r2 = r2_score(y_test_transformed, pred_test_transformed)
    train_rmse = np.sqrt(mean_squared_error(y_train_transformed, pred_train_transformed))
    test_rmse = np.sqrt(mean_squared_error(y_test_transformed, pred_test_transformed))

    # Adding all scores in the list
    score_list.extend((train_r2, test_r2, train_rmse, test_rmse))

    # If the flag is set to True then only the following print statements will be dispayed, the default value is True
    if flag == True:

        print("R-square on training set : ", train_r2)

        print("R-square on test set : ", test_r2)

        print("RMSE on training set : ", train_rmse)

        print("RMSE on test set : ", test_rmse)

    # Returning the list with train and test scores
    return score_list

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

# Make a copy of the data for modeling
df_model = data.copy()

# Ensure all necessary log-transformed features are created on df_model
# These were previously created on 'data' but 'df_model' needs them explicitly
# as it's a fresh copy for this cell's processing.
df_model['TotalCount_log'] = np.log1p(df_model['TotalCount'])
df_model['Calculated Price/Count or Price/Bundle_log'] = np.log1p(df_model['Calculated Price/Count or Price/Bundle'])
df_model['Rating_x_RatingCount_log'] = np.log1p(df_model['Rating_x_RatingCount'])
df_model['Rating_Count_Weight_log'] = np.log1p(df_model['Rating_Count Weight'])
df_model['CurrentPrice_log'] = np.log1p(df_model['CurrentPrice'])


# Step 1: Separating the independent variables (X) and the dependent variable (y)
# The dependent variable (y) is the log-transformed 'Calculated Price/Count or Price/Bundle'
y = df_model['Calculated Price/Count or Price/Bundle_log']

# Identify columns to drop from X:
# - Identifiers: ASIN, ProductTitle
# - Original numerical columns that have log-transformed versions being used or are directly related to y:
#   Count, Pack, TotalCount (use TotalCount_log instead)
#   CurrentPrice, Calculated Price/Count or Price/Bundle (original price, y is log-transformed version)
# - Sales-related metrics: Gissel's L52W $, Gissel's YAG $, Gissel's Units (target leakage for price prediction)
# - Redundant/derived rating metrics: Rating_x_RatingCount, Rating_Count Weight
# - CurrentPrice_log (highly correlated with y: Calculated Price/Count or Price/Bundle_log)
columns_to_drop_from_X = [
    'ASIN', 'ProductTitle',
    'Count', 'Pack', 'TotalCount',
    'CurrentPrice', 'Calculated Price/Count or Price/Bundle',
    "Gissel's L52W $", "Gissel's YAG $", "Gissel's Units",
    'Rating_x_RatingCount', 'Rating_Count Weight',
    'Rating_x_RatingCount_log',
    'Rating_Count_Weight_log',
    'CurrentPrice_log',
    'Calculated Price/Count or Price/Bundle_log' # This is our target variable 'y'
]

X_raw = df_model.drop(columns=columns_to_drop_from_X, errors='ignore') # Use errors='ignore' for robustness

# Handle NaN values in 'XL' and 'Flushable' before one-hot encoding
X_raw['XL'] = X_raw['XL'].fillna('Non-XL')
X_raw['Flushable'] = X_raw['Flushable'].fillna('Non-Flushable')

# Identify categorical columns. Explicitly include 'CLEANED BRAND' for one-hot encoding.
categorical_cols = X_raw.select_dtypes(include=['object']).columns.tolist()

# Apply one-hot encoding with drop_first=True as requested
X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True)

# Ensure 'TotalCount_log', 'Rating', and 'RatingCount' are explicitly kept if they are in X_raw and numeric
# These were not in columns_to_drop_from_X so they should be in X.
# No explicit re-addition needed if the drop list is correct and they exist in X_raw.

# Step 3: Split the data into train and test using train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data preparation complete!")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# Display the first few rows of X and y to confirm
print("\nX (features) head:")
display(X.head())
print("\ny (target) head:")
display(y.head())

For Regression Problems, some of the algorithms used are :<br>

**1) Linear Regression** <br>
**2) Ridge / Lasso Regression** <br>
**3) Decision Trees** <br>
**4) Random Forest** <br>

Linear Regression can be implemented using:

1) Sklearn: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
2) Statsmodels: https://www.statsmodels.org/stable/regression.html

### **Fitting a linear model**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Initialize the Linear Regression model
lin_reg = LinearRegression()

# Fit the model to the training data
lin_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred_test = lin_reg.predict(X_test)

# Evaluate the model
r2 = r2_score(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)

print(f"Linear Regression Model Performance:")
print(f"R-squared: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def get_model_score(model, flag = True):
    '''
    model : regressor to predict values of X

    '''
    # Defining an empty list to store train and test results
    score_list = []

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    # If flag is True, convert predictions and actuals back to original scale
    if flag:
        pred_train_transformed = np.expm1(pred_train) # Inverse of log1p
        pred_test_transformed = np.expm1(pred_test) # Inverse of log1p
        y_train_transformed = np.expm1(y_train) # Inverse of log1p for actuals
        y_test_transformed = np.expm1(y_test) # Inverse of log1p for actuals
    else:
        pred_train_transformed = pred_train
        pred_test_transformed = pred_test
        y_train_transformed = y_train
        y_test_transformed = y_test

    train_r2 = r2_score(y_train_transformed, pred_train_transformed)
    test_r2 = r2_score(y_test_transformed, pred_test_transformed)
    train_rmse = np.sqrt(mean_squared_error(y_train_transformed, pred_train_transformed))
    test_rmse = np.sqrt(mean_squared_error(y_test_transformed, pred_test_transformed))

    # Adding all scores in the list
    score_list.extend((train_r2, test_r2, train_rmse, test_rmse))

    # If the flag is set to True then only the following print statements will be dispayed, the default value is True
    if flag == True:

        print("R-square on training set : ", train_r2)

        print("R-square on test set : ", test_r2)

        print("RMSE on training set : ", train_rmse)

        print("RMSE on test set : ", test_rmse)

    # Returning the list with train and test scores
    return score_list

# Get score of the model
LR_score = get_model_score(lin_reg)
LR_score

In [ ]:
# Important variables of Linear Regression

# Get coefficients and feature names from the scikit-learn model
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lin_reg.coef_
})

# Sort by the absolute value of coefficients to find the most impactful features
coefficients['Absolute_Coefficient'] = np.abs(coefficients['Coefficient'])
coefficients = coefficients.sort_values(by='Absolute_Coefficient', ascending=False)

print("Top Important Features from sklearn Linear Regression (by absolute coefficient value):")
display(coefficients.head(100))

# Build a Linear Regression model using statsmodels
import statsmodels.api as sm

# Convert boolean columns to integer (0 or 1) for statsmodels compatibility
X_train_numeric = X_train.copy()
for col in X_train_numeric.select_dtypes(include='bool').columns:
    X_train_numeric[col] = X_train_numeric[col].astype(int)

# Add a constant to the independent variables for statsmodels
X_train_sm = sm.add_constant(X_train_numeric, prepend=False)

# Convert boolean columns in X_test to integer (0 or 1) before adding constant
X_test_numeric = X_test.copy()
for col in X_test_numeric.select_dtypes(include='bool').columns:
    X_test_numeric[col] = X_test_numeric[col].astype(int)

X_test_sm = sm.add_constant(X_test_numeric, prepend=False)

# Fit the OLS model
sm_lin_reg = sm.OLS(y_train, X_train_sm)
sm_lin_reg_result = sm_lin_reg.fit()

print("\nStatsmodels Linear Regression Model Summary:")
print(sm_lin_reg_result.summary())

In [ ]:
import numpy as np

# Get the p-values from the statsmodels OLS result
p_values = sm_lin_reg_result.pvalues

# Filter for statistically significant variables (p-value <= 0.05)
significant_features_with_dummies = p_values[p_values <= 0.05].index.tolist()

# Exclude the 'const' term if present
if 'const' in significant_features_with_dummies:
    significant_features_with_dummies.remove('const')

# Get the original (un-one-hot encoded) categorical and numerical variables
significant_overall_variables = set()

# Get original categorical column names from X_raw for mapping
original_categorical_cols = X_raw.select_dtypes(include=['object']).columns.tolist()

for feature in significant_features_with_dummies:
    # Check if the feature is a numerical column directly (e.g., 'Rating', 'RatingCount', 'TotalCount_log')
    if feature in X_raw.select_dtypes(include=np.number).columns.tolist() or feature.startswith('TotalCount_log'):
        significant_overall_variables.add(feature)
    else:
        # It's likely a one-hot encoded categorical feature
        # Iterate through original categorical columns to find a match
        found_original_cat = False
        for original_col in original_categorical_cols:
            # If the feature name starts with the original column name and an underscore
            # or is an exact match for a binary categorical variable that might not have an underscore
            if feature.startswith(original_col + '_') or feature == original_col:
                significant_overall_variables.add(original_col)
                found_original_cat = True
                break
        if not found_original_cat: # Add features that didn't match an original category (e.g. TotalCount_log)
            significant_overall_variables.add(feature)


start = '\033[1m'
end = '\033[0m'
print(start + 'Overall Significant Variables from LINEAR REGRESSION (p <= 0.05):' + end)
for var in sorted(list(significant_overall_variables)):
    print(f'- {var}')


Based on the `statsmodels` linear regression summary from the previous output, we can identify overall significant variables by looking at their **P>|t| (p-value)**. A common threshold for statistical significance is 0.05. Variables with a p-value less than 0.05 are generally considered statistically significant.

From the summary, some variables with significant p-values include:
- `Rating` (p-value = 0.017)
- `User_Feminine` (p-value = 0.002)
- `User_Man` (p-value = 0.000)
- `Container_Resealable Pack` (p-value = 0.022)

**However, it's crucial to remember the note at the bottom of the `statsmodels` summary:**

> `[2] The smallest eigenvalue is 7.04e-33. This might indicate that there are strong multicollinearity problems or that the design matrix is singular.`

This indicates **severe multicollinearity** among the features. When multicollinearity is present, the individual coefficients and their p-values can be unstable and difficult to interpret reliably. While these variables show statistical significance in this model, their individual importance might be overstated or misleading due to their strong correlation with other features. This aligns with the unresolved task identified earlier: to address severe multicollinearity for a more robust understanding of variable importance.

**Build Ridge / Lasso Regression similar to Linear Regression:**<br>

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html

In [ ]:
import numpy as np

# Get the p-values from the statsmodels OLS result
p_values = sm_lin_reg_result.pvalues

# Filter for statistically significant variables (p-value <= 0.05)
significant_features_with_dummies = p_values[p_values <= 0.05].index.tolist()

# Exclude the 'const' term if present
if 'const' in significant_features_with_dummies:
    significant_features_with_dummies.remove('const')

# Get the original (un-one-hot-encoded) categorical and numerical variables
significant_overall_variables = set()

# Get original categorical column names from X_raw for mapping
original_categorical_cols = X_raw.select_dtypes(include=['object']).columns.tolist()

for feature in significant_features_with_dummies:
    # Check if the feature is a numerical column directly (e.g., 'Rating', 'RatingCount', 'TotalCount_log')
    if feature in X_raw.select_dtypes(include=np.number).columns.tolist() or feature.startswith('TotalCount_log'):
        significant_overall_variables.add(feature)
    else:
        # It's likely a one-hot encoded categorical feature
        # Iterate through original categorical columns to find a match
        found_original_cat = False
        for original_col in original_categorical_cols:
            # If the feature name starts with the original column name and an underscore
            # or is an exact match for a binary categorical variable that might not have an underscore
            if feature.startswith(original_col + '_') or feature == original_col:
                significant_overall_variables.add(original_col)
                found_original_cat = True
                break
        if not found_original_cat: # Add features that didn't match an original category (e.g. TotalCount_log)
            significant_overall_variables.add(feature)


start = '\033[1m'
end = '\033[0m'
print(start + 'Overall Significant Variables from LINEAR REGRESSION (p <= 0.05):' + end)
for var in sorted(list(significant_overall_variables)):
    print(f'- {var}')

Overall Significant Variables from LINEAR REGRESSION (p <= 0.05):
- Attribute
- Benefits
- CLEANED BRAND
- Container
- Features
- Flushable
- Free_From
- Rating
- Scent_ingredient
- Skin_Type
- User

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Initialize the Ridge Regression model
# Alpha is the regularization strength; needs tuning, but start with a common value
ridge_reg = Ridge(alpha=1.0)

# Fit the model to the training data
ridge_reg.fit(X_train_numeric, y_train)

# Make predictions on the test set
y_pred_ridge = ridge_reg.predict(X_test_numeric)

# Evaluate the model
r2_ridge = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mse_ridge)

print(f"Ridge Regression Model Performance:")
print(f"R-squared: {r2_ridge:.4f}")
print(f"Mean Absolute Error (MAE): {mae_ridge:.4f}")
print(f"Mean Squared Error (MSE): {mse_ridge:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ridge:.4f}")

# Get coefficients and feature names from the Ridge model
ridge_coefficients = pd.DataFrame({
    'Feature': X_train_numeric.columns,
    'Coefficient': ridge_reg.coef_
})

# Sort by the absolute value of coefficients to find the most impactful features
ridge_coefficients['Absolute_Coefficient'] = np.abs(ridge_coefficients['Coefficient'])
ridge_coefficients = ridge_coefficients.sort_values(by='Absolute_Coefficient', ascending=False)

print("\nTop 10 Most Important Features from Ridge Regression (by absolute coefficient value):")
display(ridge_coefficients.head(10))

In [ ]:
# Get score of the ridge regression model
ridge_score = get_model_score(ridge_reg)
ridge_score

**Observations from results:
1. Using Ridge regression model, it can explain 33.3% of the variance in the training set, 22.8% of the variance in the testing set; Ridge regression model's R-square on test set performs better than linear regression(14.2%).
2. RMSE on the training set is 0.718 on the training set, and 0.476 on the test set; the RMSE score in the testing set performs better than linear regression model(0.502). The low RMSE values indicate that the model's predictions are better than linear regression model.


**

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Create a Ridge regression model
ridge = Ridge(alpha=1.0)

# Fit Ridge regression model
ridge.fit(X_train, y_train)

# Make predictions on the test set
y_pred_ridge_tuned = ridge.predict(X_test)

# Evaluate the model
r2_ridge_tuned = r2_score(y_test, y_pred_ridge_tuned)
mae_ridge_tuned = mean_absolute_error(y_test, y_pred_ridge_tuned)
mse_ridge_tuned = mean_squared_error(y_test, y_pred_ridge_tuned)
rmse_ridge_tuned = np.sqrt(mse_ridge_tuned)

print(f"Ridge Regression Model Performance (after fixing):")
print(f"R-squared: {r2_ridge_tuned:.4f}")
print(f"Mean Absolute Error (MAE): {mae_ridge_tuned:.4f}")
print(f"Mean Squared Error (MSE): {mse_ridge_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ridge_tuned:.4f}")

# Get coefficients and feature names from the Ridge model
ridge_coefficients_tuned = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': ridge.coef_
})

# Sort by the absolute value of coefficients to find the most impactful features
ridge_coefficients_tuned['Absolute_Coefficient'] = np.abs(ridge_coefficients_tuned['Coefficient'])
ridge_coefficients_tuned = ridge_coefficients_tuned.sort_values(by='Absolute_Coefficient', ascending=False)

print("\nTop 10 Most Important Features from Ridge Regression (by absolute coefficient value, after fixing):")
display(ridge_coefficients_tuned.head(10))

# --- Code added to filter and download significant p-values ---

import pandas as pd
from google.colab import files

# Get the p-values from the statsmodels OLS result
p_values = sm_lin_reg_result.pvalues

# Create a DataFrame from OLS p-values for merging
ols_pvalues_df = pd.DataFrame(p_values).reset_index()
ols_pvalues_df.columns = ['Feature', 'P-Value']

# Merge Ridge coefficients with OLS p-values
combined_features_df = pd.merge(ridge_coefficients_tuned, ols_pvalues_df, on='Feature', how='left')

# Sort by Absolute_Coefficient (descending) to show most impactful features first
combined_features_df = combined_features_df.sort_values(by='Absolute_Coefficient', ascending=False).reset_index(drop=True)

print("\nCombined Feature Importance (Ridge Coefficients) and Statistical Significance (OLS P-Values) - Top 20:")
display(combined_features_df[['Feature', 'Coefficient', 'P-Value']].head(20))

# Save the complete combined DataFrame to a CSV file
csv_filename_combined = 'combined_feature_importance_and_pvalues.csv'
combined_features_df[['Feature', 'Coefficient', 'P-Value']].to_csv(csv_filename_combined, index=False)

files.download(csv_filename_combined)
print(f"\n'{csv_filename_combined}' has been generated and is ready for download.")

# Filter for statistically significant variables (p-value < 0.05) - existing logic
significant_p_values = p_values[p_values < 0.05]

# Sort in descending order
significant_p_values_sorted = significant_p_values.sort_values(ascending=False)

# Create a DataFrame for display and download
significant_p_values_df = pd.DataFrame({
    'Feature': significant_p_values_sorted.index,
    'P-Value': significant_p_values_sorted.values
})

print("\nStatistically Significant P-Values (p < 0.05), sorted descending (from OLS model alone):")
display(significant_p_values_df)

# Save to CSV and provide download link (for OLS p-values only - existing logic)
csv_filename_ols_pvalues = 'significant_p_values_linear_regression.csv'
significant_p_values_df.to_csv(csv_filename_ols_pvalues, index=False)

files.download(csv_filename_ols_pvalues)
print(f"\n'{csv_filename_ols_pvalues}' has been generated and is ready for download.")

Ridge Regression Model Performance:


R-square on training set :  0.333423953926819
R-square on test set :  0.2277032498201157
RMSE on training set :  0.718430782139913
RMSE on test set :  0.4764925736832653
[0.333423953926819,
 0.2277032498201157,
 np.float64(0.718430782139913),
 np.float64(0.4764925736832653)]

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files # Import for downloading files

# Ensure 'data', 'baby_wipe_data', and 'other_wipe_data' are up-to-date
# This block re-initializes them to ensure consistency in case prior cells were skipped or modified.
# Assuming the Excel file path is correct and accessible.
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    # Handle the error, perhaps by exiting or loading dummy data
    raise

data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct

baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

user_segments = {
    "All Users": data,
    "Baby Users": baby_wipe_data,
    "Non-Baby Users": other_wipe_data
}

# List to store dataframes for consolidation
all_segments_top_brands = []

for segment_name, segment_df in user_segments.items():
    if segment_df.empty:
        print(f"No data available for {segment_name}.")
        continue

    # Calculate total Rating_Count Weight per brand within the segment
    brand_rating_weight = segment_df.groupby('CLEANED BRAND')['Rating_Count Weight'].sum()

    # Determine the number of top brands (top 20%) to select
    num_unique_brands = len(brand_rating_weight)
    top_n_brands = max(1, int(num_unique_brands * 0.20)) # Ensure at least 1 brand if data exists

    # Get the names of the top N brands by Rating_Count Weight
    top_brands_names = brand_rating_weight.nlargest(top_n_brands).index.tolist()

    if not top_brands_names:
        print(f"No top brands found for {segment_name} based on Rating_Count Weight.")
        continue

    # Filter the original DataFrame for these top brands
    top_brands_data = segment_df[segment_df['CLEANED BRAND'].isin(top_brands_names)].copy()

    # Calculate the average 'Calculated Price/Count or Price/Bundle' for these top brands
    avg_price_per_count_by_brand = top_brands_data.groupby('CLEANED BRAND')['Calculated Price/Count or Price/Bundle'].mean().reset_index()

    # Sort by average price per count in descending order
    avg_price_per_count_by_brand = avg_price_per_count_by_brand.sort_values(
        by='Calculated Price/Count or Price/Bundle', ascending=False
    ).reset_index(drop=True)

    # Add a column for the user segment before appending
    avg_price_per_count_by_brand['User_Segment'] = segment_name
    all_segments_top_brands.append(avg_price_per_count_by_brand)

    print(f"\n--- Top {top_n_brands} Brands for {segment_name} (ranked by Avg. Calculated Price/Count or Price/Bundle) ---")
    display(avg_price_per_count_by_brand)

print("\nAnalysis complete: Top brands by average price per count have been listed for each user segment.")

# Concatenate all segment dataframes into a single one
consolidated_top_brands_df = pd.concat(all_segments_top_brands, ignore_index=True)

# Save the consolidated DataFrame to a CSV file
csv_filename = 'top_20_percent_brands_by_price_per_count.csv'
consolidated_top_brands_df.to_csv(csv_filename, index=False)

# Provide a download link
files.download(csv_filename)
print(f"\n'{csv_filename}' has been generated and is ready for download.")

You're asking about the important factors from all the columns, derived from the Ridge regression model. Based on the model, these are the top 10 most impactful features (or factors) by absolute coefficient value:

1.  **CLEANED BRAND_BRAVA**
2.  **CLEANED BRAND_TIEJIANDAN**
3.  **CLEANED BRAND_SHOWER POUCH**
4.  **CLEANED BRAND_SPIFF CLOTH**
5.  **CLEANED BRAND_S SHOWER POUCH**
6.  **CLEANED BRAND_BODUM**
7.  **CLEANED BRAND_MR.CLEAN**
8.  **CLEANED BRAND_WATERWIPES**
9.  **CLEANED BRAND_JOHNSON'S**
10. **CLEANED BRAND_DR.BROWN'S**

These features, primarily specific 'CLEANED BRAND' indicators, have the strongest influence on the predicted `Calculated Price/Count or Price/Bundle_log` according to the Ridge Regression. Their larger absolute coefficient values suggest a greater impact on the target variable, with the regularization helping to manage multicollinearity.

In [ ]:
import pandas as pd
from google.colab import files

# Load the previously saved combined feature importance and p-values
# Assuming 'combined_features_df' is still available from the previous run. If not, load from CSV.
if 'combined_features_df' not in locals():
    try:
        combined_features_df = pd.read_csv('combined_feature_importance_and_pvalues.csv')
    except FileNotFoundError:
        print("Error: 'combined_feature_importance_and_pvalues.csv' not found. Please ensure it was generated and is accessible.")
        # Fallback if the file is not found
        # (You would normally regenerate or ask the user to rerun previous steps)
        # For this example, we'll create a dummy or raise an error.
        raise

# Sort the DataFrame by 'P-Value' in ascending order to show most significant first
combined_features_sorted_by_pvalue = combined_features_df.sort_values(by='P-Value', ascending=True).reset_index(drop=True)

print("\nCombined Feature Importance and Statistical Significance (sorted by P-Value) - Top 20:")
display(combined_features_sorted_by_pvalue[['Feature', 'Coefficient', 'P-Value']].head(20))

# Save the sorted DataFrame to a new CSV file
csv_filename_sorted_pvalue = 'combined_feature_importance_and_pvalues_sorted_by_pvalue.csv'
combined_features_sorted_by_pvalue[['Feature', 'Coefficient', 'P-Value']].to_csv(csv_filename_sorted_pvalue, index=False)

files.download(csv_filename_sorted_pvalue)
print(f"\n'{csv_filename_sorted_pvalue}' has been generated and is ready for download.")

### **Decision Tree**

https://scikit-learn.org/stable/auto_examples/tree/plot_tree_regression.html

In [ ]:
# Import Decision tree for Regression from sklearn
from sklearn.tree import DecisionTreeRegressor

In [ ]:
# Create a decision tree regression model, use random_state = 1
dtree = DecisionTreeRegressor(random_state=1)

In [ ]:
# Fit decision tree regression model
dtree.fit(X_train, y_train)

In [ ]:
# Get score of the model
Dtree_model = get_model_score(dtree)
Dtree_model

R²  0.9908042429397105 is so high on training and negative -0.1233626229978948 on test because your model is badly overfitting the training data, so its predictions on unseen data are worse than just predicting the mean every time.
How to fix it
For a decision tree regressor in scikit‑learn, try:

Regularize the tree: Set max_depth, min_samples_split, min_samples_leaf, and/or max_leaf_nodes to limit complexity and reduce variance.

Use cross‑validation: Instead of a single train/test split, use cross_val_score with cv=5 or similar to get a more stable estimate and to tune hyperparameters.

Try an ensemble with regularization: Random forests or gradient boosting with constrained depth often generalize better than a single deep tree.

Check data issues: Make sure the feature distributions in train vs test are similar, remove obvious leakage, and scale/transform highly skewed variables if needed.

In [ ]:
#<!-- The following code was removed because it redefined the `get_model_score` function to return a string, causing a `ValueError` in subsequent cells. The correct `get_model_score` function, returning numerical values, is defined elsewhere in the notebook. -->
#<!--
# Get score of the model
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def get_model_score(model):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return f"R-squared: {r2:.4f}, MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}"

Dtree_model_score = get_model_score(dtree)
print(f"Decision Tree Model Performance: {Dtree_model_score}")
#-->

Print the importance of features in the tree building. The importance of a feature is computed as the (normalized) total reduction of the criterion brought by that feature. It is also known as the Gini importance.

In [ ]:
print(pd.DataFrame(dtree.feature_importances_, columns = ["Imp"], index = X_train.columns).sort_values(by = 'Imp', ascending = False))

Decision Tree is not a good model for wipe pricing prediction.

### **Random Forest**

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

In [ ]:
# Import Randomforest for Regression from sklearn
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Create a Randomforest regression model
Rand= RandomForestRegressor()

In [ ]:
# Fit Randomforest regression model
Rand.fit(X_train, y_train)

In [ ]:
# Get score of the model
Rand_model = get_model_score(Rand)
Rand_model

**Observations and insights:
R-squared: 0.3737, MAE: 0.1008, MSE: 0.0357, RMSE: 0.1890
**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from google.colab import files

# --- Ensure 'data' and derived dataframes are fully processed ---
# Reload data and apply necessary transformations to ensure consistency
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    raise # Stop execution if data file is not found

# Apply initial preprocessing steps (standardize brand, calculate rating metrics, log-transform counts and prices)
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct
data['TotalCount_log'] = np.log(data['TotalCount'])
data['CurrentPrice_log'] = np.log1p(data['CurrentPrice'])
data['Calculated Price/Count or Price/Bundle_log'] = np.log1p(data['Calculated Price/Count or Price/Bundle'])

# Create segmented dataframes from the fully processed 'data'
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define the base columns to drop from feature set (X) for consistency across all segments
columns_to_drop_from_X_base = [
    'ASIN', 'ProductTitle',
    'Count', 'Pack', 'TotalCount',
    'CurrentPrice', 'Calculated Price/Count or Price/Bundle',
    "Gissel's L52W $", "Gissel's YAG $", "Gissel's Units",
    'Rating_x_RatingCount', 'Rating_Count Weight',
    'Rating_x_RatingCount_log', # Already dropped, but ensures robustness if added in future
    'Rating_Count_Weight_log',  # Already dropped, but ensures robustness if added in future
    'CurrentPrice_log',
    'Calculated Price/Count or Price/Bundle_log' # This is our target variable 'y'
]

def prepare_X_y_for_segment(df_segment):
    """
    Prepares features (X) and target (y) for a given DataFrame segment,
    including dropping specified columns, handling NaNs, and one-hot encoding.
    """
    y_segment = df_segment['Calculated Price/Count or Price/Bundle_log']

    X_raw_segment = df_segment.drop(columns=columns_to_drop_from_X_base, errors='ignore')

    # Handle NaN values in 'XL' and 'Flushable' before one-hot encoding
    X_raw_segment['XL'] = X_raw_segment['XL'].fillna('Non-XL')
    X_raw_segment['Flushable'] = X_raw_segment['Flushable'].fillna('Non-Flushable')

    categorical_cols_segment = X_raw_segment.select_dtypes(include=['object']).columns.tolist()
    X_segment = pd.get_dummies(X_raw_segment, columns=categorical_cols_segment, drop_first=True)

    # Ensure TotalCount_log is included if it was not dropped and is numeric
    if 'TotalCount_log' not in X_segment.columns and 'TotalCount_log' in df_segment.columns:
        X_segment['TotalCount_log'] = df_segment['TotalCount_log']

    # Ensure Rating_log is present - it is `Rating` from df_model
    if 'Rating' not in X_segment.columns and 'Rating' in df_segment.columns:
        X_segment['Rating'] = df_segment['Rating']

    # Ensure RatingCount is present
    if 'RatingCount' not in X_segment.columns and 'RatingCount' in df_segment.columns:
        X_segment['RatingCount'] = df_segment['RatingCount']

    return X_segment, y_segment

# Prepare the 'All Users' data first to establish a reference set of columns for X
X_all, y_all = prepare_X_y_for_segment(data)

# Store the columns of the 'All Users' X for consistent alignment across segments
global_X_columns = X_all.columns

def analyze_and_save_feature_importances(df_segment, segment_name):
    """
    Processes a user segment, trains a Random Forest model, and reports feature importances.
    """
    print(f"\n--- Analyzing Feature Importances for {segment_name} User Group ---")

    if df_segment.empty:
        print(f"No data available for {segment_name}. Skipping.")
        return

    X_segment_raw, y_segment = prepare_X_y_for_segment(df_segment)

    # Align columns of X_segment with the global X columns
    missing_cols = set(global_X_columns) - set(X_segment_raw.columns)
    for c in missing_cols:
        X_segment_raw[c] = 0 # Add missing columns with 0

    extra_cols = set(X_segment_raw.columns) - set(global_X_columns)
    X_segment = X_segment_raw.drop(columns=list(extra_cols))

    X_segment = X_segment[global_X_columns] # Ensure order of columns is the same

    if X_segment.empty or len(X_segment.columns) == 0:
        print(f"No valid features left for {segment_name} after alignment. Skipping.")
        return

    # If the segment_df has very few rows, train_test_split might fail if test_size is too large
    # or if there are too few samples after splitting. Adjust for very small datasets if needed.
    if len(X_segment) < 2:
        print(f"Not enough samples in {segment_name} to perform train_test_split. Skipping model training.")
        return

    # Split data for the current segment
    X_train_segment, X_test_segment, y_train_segment, y_test_segment = train_test_split(X_segment, y_segment, test_size=0.2, random_state=42)

    # Re-initialize and fit Random Forest model
    rf_model_segment = RandomForestRegressor(random_state=42)
    rf_model_segment.fit(X_train_segment, y_train_segment)

    # Get feature importances
    feature_importances_segment = rf_model_segment.feature_importances_

    # Create a DataFrame to display feature names with their importance scores
    feature_importance_df_segment = pd.DataFrame({
        'Feature': X_train_segment.columns,
        'Importance': feature_importances_segment
    }).sort_values(by='Importance', ascending=False)

    print(f"\nTop 10 Feature Importances for {segment_name} User Group:")
    display(feature_importance_df_segment.head(10))

    # Save to CSV and provide download link
    csv_filename_segment = f'random_forest_feature_importances_{segment_name.replace(" ", "_").lower()}.csv'
    feature_importance_df_segment.to_csv(csv_filename_segment, index=False)
    files.download(csv_filename_segment)
    print(f"'{csv_filename_segment}' has been generated and is ready for download.")

# --- Process each user group ---
user_groups = {
    "All Users": data,
    "Baby Users": baby_wipe_data,
    "Non-Baby Users": other_wipe_data
}

for name, df in user_groups.items():
    analyze_and_save_feature_importances(df, name)


### **Hyperparameter Tuning: Decision Tree**

In [ ]:
from sklearn.metrics import make_scorer, r2_score
# Choose the type of estimator
dtree_tuned = DecisionTreeRegressor(random_state = 1)

# Grid of parameters to choose from
# Check documentation for all the parametrs that the model takes and play with those
parameters = {
        'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2', None]
}

# Type of scoring used to compare parameter combinations
scorer = make_scorer(r2_score)

# Run the grid search
grid_obj = GridSearchCV(dtree_tuned, parameters, scoring=scorer, cv=5)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the model to the best combination of parameters
dtree_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data
dtree_tuned.fit(X_train, y_train)

In [ ]:
# Get score of the dtree_tuned
# Get the R² score on the training set
train_score = dtree_tuned.score(X_train, y_train)
print("Training R² Score:", train_score)

# Get the R² score on the test set
test_score = dtree_tuned.score(X_test, y_test)
print("Test R² Score:", test_score)

Observations: After fine tuning, the training r2 score increased to 0.75; the test r2 score increased to 0.29.

**Feature Importance**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# --- Start: Ensure data is loaded and preprocessed for standalone execution ---
# This block ensures 'data', 'baby_wipe_data', and 'other_wipe_data' are properly initialized
# assuming the excel file is in the specified Google Drive path.
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    # Exit or handle the error gracefully
    raise

# Apply initial preprocessing steps (standardize brand, calculate rating metrics, log-transform counts and prices)
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct
data['TotalCount_log'] = np.log(data['TotalCount'])
data['CurrentPrice_log'] = np.log1p(data['CurrentPrice'])
data['Calculated Price/Count or Price/Bundle_log'] = np.log1p(data['Calculated Price/Count or Price/Bundle'])

# Create segmented dataframes from the fully processed 'data'
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define the base columns to drop from feature set (X) for consistency across all segments
columns_to_drop_from_X_base = [
    'ASIN', 'ProductTitle',
    'Count', 'Pack', 'TotalCount',
    'CurrentPrice', 'Calculated Price/Count or Price/Bundle',
    "Gissel's L52W $", "Gissel's YAG $", "Gissel's Units",
    'Rating_x_RatingCount', 'Rating_Count Weight',
    'Rating_x_RatingCount_log', # Not explicitly in X_raw initially, but good to include if it were
    'Rating_Count_Weight_log',  # Not explicitly in X_raw initially, but good to include if it were
    'CurrentPrice_log',
    'Calculated Price/Count or Price/Bundle_log' # This is our target variable 'y'
]

def prepare_X_y_for_segment(df_segment):
    """
    Prepares features (X) and target (y) for a given DataFrame segment,
    including dropping specified columns, handling NaNs, and one-hot encoding.
    """
    y_segment = df_segment['Calculated Price/Count or Price/Bundle_log']

    X_raw_segment = df_segment.drop(columns=columns_to_drop_from_X_base, errors='ignore')

    # Handle NaN values in 'XL' and 'Flushable' before one-hot encoding
    X_raw_segment['XL'] = X_raw_segment['XL'].fillna('Non-XL')
    X_raw_segment['Flushable'] = X_raw_segment['Flushable'].fillna('Non-Flushable')

    categorical_cols_segment = X_raw_segment.select_dtypes(include=['object']).columns.tolist()
    X_segment_encoded = pd.get_dummies(X_raw_segment, columns=categorical_cols_segment, drop_first=True)

    # Ensure 'TotalCount_log' is included if it was not dropped and is numeric
    if 'TotalCount_log' not in X_segment_encoded.columns and 'TotalCount_log' in df_segment.columns:
        X_segment_encoded['TotalCount_log'] = df_segment['TotalCount_log']

    # Ensure 'Rating' is present
    if 'Rating' not in X_segment_encoded.columns and 'Rating' in df_segment.columns:
        X_segment_encoded['Rating'] = df_segment['Rating']

    # Ensure 'RatingCount' is present
    if 'RatingCount' not in X_segment_encoded.columns and 'RatingCount' in df_segment.columns:
        X_segment_encoded['RatingCount'] = df_segment['RatingCount']

    return X_segment_encoded, y_segment

# Prepare the 'All Users' data first to establish a reference set of columns for X
X_all_processed, y_all_processed = prepare_X_y_for_segment(data)
global_X_columns = X_all_processed.columns
# --- End: Ensure data is loaded and preprocessed for standalone execution ---


# Define the optimal hyperparameters obtained from GridSearchCV
# (from previous `dtree_tuned` output: max_features='sqrt', min_samples_split=10, random_state=1)
optimal_params = {
    'max_features': 'sqrt',
    'min_samples_split': 10,
    'random_state': 1,
    'min_samples_leaf': 1 # Default from GridSearchCV output unless specified
}

user_groups_data = {
    "Baby Users": baby_wipe_data,
    "Non-Baby Users": other_wipe_data
}

for group_name, df_group in user_groups_data.items():
    print(f"\n--- Feature Importances for {group_name} ---")

    if df_group.empty:
        print(f"No data available for {group_name}. Skipping feature importance calculation.")
        continue

    X_group, y_group = prepare_X_y_for_segment(df_group)

    # Align columns of X_group with the global X columns
    missing_cols = set(global_X_columns) - set(X_group.columns)
    for c in missing_cols:
        X_group[c] = 0 # Add missing columns with 0

    extra_cols = set(X_group.columns) - set(global_X_columns)
    X_group = X_group.drop(columns=list(extra_cols))

    X_group = X_group[global_X_columns] # Ensure order of columns is the same

    # Skip if not enough samples to split
    if len(X_group) < 2:
        print(f"Not enough samples in {group_name} to perform train_test_split. Skipping model training.")
        continue

    # Split data for the current group - only need X_train and y_train for fitting
    X_train_group, _, y_train_group, _ = train_test_split(X_group, y_group, test_size=0.2, random_state=42)

    # Initialize and fit Decision Tree Regressor with optimal parameters
    dtree_tuned_group = DecisionTreeRegressor(**optimal_params)
    dtree_tuned_group.fit(X_train_group, y_train_group)

    # Get feature importances
    feature_importances_group = dtree_tuned_group.feature_importances_

    # Create a DataFrame to display feature names with their importance scores
    feature_importance_df_group = pd.DataFrame({
        'Feature': X_train_group.columns,
        'Importance': feature_importances_group
    }).sort_values(by='Importance', ascending=False)

    # Display the sorted feature importances
    print(f"Top 30 Feature Importances for Tuned Decision Tree ({group_name}):")
    print(feature_importance_df_group.head(30).to_string()) # Use to_string() for full display in notebook output

### **Hyperparameter Tuning: Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import make_scorer, r2_score
import pandas as pd
import numpy as np

# --- Start: Ensure data is loaded and preprocessed for standalone execution ---
# This block ensures 'data' and derived dataframes are properly initialized
# assuming the excel file is in the specified Google Drive path.
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    raise # Exit if data file is not found

# Apply initial preprocessing steps (standardize brand, calculate rating metrics, log-transform counts and prices)
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct
data['TotalCount_log'] = np.log(data['TotalCount'])
data['CurrentPrice_log'] = np.log1p(data['CurrentPrice'])
data['Calculated Price/Count or Price/Bundle_log'] = np.log1p(data['Calculated Price/Count or Price/Bundle'])

# Create segmented dataframes from the fully processed 'data'
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define the base columns to drop from feature set (X) for consistency across all segments
columns_to_drop_from_X_base = [
    'ASIN', 'ProductTitle',
    'Count', 'Pack', 'TotalCount',
    'CurrentPrice', 'Calculated Price/Count or Price/Bundle',
    "Gissel's L52W $", "Gissel's YAG $", "Gissel's Units",
    'Rating_x_RatingCount', 'Rating_Count Weight',
    'Rating_x_RatingCount_log', # Not explicitly in X_raw initially, but good to include if it were
    'Rating_Count_Weight_log',  # Not explicitly in X_raw initially, but good to include if it were
    'CurrentPrice_log',
    'Calculated Price/Count or Price/Bundle_log' # This is our target variable 'y'
]

def prepare_X_y_for_segment(df_segment):
    """
    Prepares features (X) and target (y) for a given DataFrame segment,
    including dropping specified columns, handling NaNs, and one-hot encoding.
    """
    y_segment = df_segment['Calculated Price/Count or Price/Bundle_log']

    X_raw_segment = df_segment.drop(columns=columns_to_drop_from_X_base, errors='ignore')

    # Handle NaN values in 'XL' and 'Flushable' before one-hot encoding
    X_raw_segment['XL'] = X_raw_segment['XL'].fillna('Non-XL')
    X_raw_segment['Flushable'] = X_raw_segment['Flushable'].fillna('Non-Flushable')

    categorical_cols_segment = X_raw_segment.select_dtypes(include=['object']).columns.tolist()
    X_segment_encoded = pd.get_dummies(X_raw_segment, columns=categorical_cols_segment, drop_first=True)

    # Ensure 'TotalCount_log' is included if it was not dropped and is numeric
    if 'TotalCount_log' not in X_segment_encoded.columns and 'TotalCount_log' in df_segment.columns:
        X_segment_encoded['TotalCount_log'] = df_segment['TotalCount_log']

    # Ensure 'Rating' is present
    if 'Rating' not in X_segment_encoded.columns and 'Rating' in df_segment.columns:
        X_segment_encoded['Rating'] = df_segment['Rating']

    # Ensure 'RatingCount' is present
    if 'RatingCount' not in X_segment_encoded.columns and 'RatingCount' in df_segment.columns:
        X_segment_encoded['RatingCount'] = df_segment['RatingCount']

    return X_segment_encoded, y_segment

# Prepare the 'All Users' data first to establish a reference set of columns for X
X_all_processed, y_all_processed = prepare_X_y_for_segment(data)
global_X_columns = X_all_processed.columns
# --- End: Ensure data is loaded and preprocessed for standalone execution ---

user_segments_data = {
    "Baby Users": baby_wipe_data,
    "Non-Baby Users": other_wipe_data
}

# Define the parameters for Grid to choose from
parameters = {
    'n_estimators': [50, 100, 200],  # Number of trees
    'max_depth': [None, 10, 20],  # Maximum depth of trees
    'min_samples_split': [2, 5, 10],  # Minimum samples to split a node
    'min_samples_leaf': [1, 2, 4],  # Minimum samples in a leaf
    'max_features': ['auto', 'sqrt']  # Number of features considered for split
}
# Type of scoring used to compare parameter combinations
scorer = make_scorer(r2_score)

for segment_name, segment_df in user_segments_data.items():
    print(f"\n--- Hyperparameter Tuning for Random Forest Regressor ({segment_name}) ---")

    if segment_df.empty:
        print(f"No data available for {segment_name}. Skipping tuning.")
        continue

    X_segment, y_segment = prepare_X_y_for_segment(segment_df)

    # Align columns of X_segment with the global X columns
    missing_cols = set(global_X_columns) - set(X_segment.columns)
    for c in missing_cols:
        X_segment[c] = 0 # Add missing columns with 0

    extra_cols = set(X_segment.columns) - set(global_X_columns)
    X_segment = X_segment.drop(columns=list(extra_cols))

    X_segment = X_segment[global_X_columns] # Ensure order of columns is the same

    if X_segment.empty or len(X_segment.columns) == 0:
        print(f"No valid features left for {segment_name} after alignment. Skipping tuning.")
        continue

    if len(X_segment) < 10: # Ensure enough samples for CV and split (e.g., min 10 samples)
        print(f"Not enough samples in {segment_name} to perform GridSearchCV. Skipping tuning.")
        continue

    # Split data for the current segment
    X_train_segment, X_test_segment, y_train_segment, y_test_segment = train_test_split(X_segment, y_segment, test_size=0.2, random_state=42)

    # Choose the type of Regressor
    regressor = RandomForestRegressor(random_state=1)

    # Run the grid search
    grid_obj = GridSearchCV(regressor, parameters, scoring=scorer, cv=5, n_jobs=-1, verbose=1)
    grid_obj.fit(X_train_segment, y_train_segment)

    # Set the model to the best combination of parameters
    best_regressor = grid_obj.best_estimator_
    # Fit the best algorithm to the data
    best_regressor.fit(X_train_segment, y_train_segment)

    # Print the best parameters
    print(f"Best Parameters for {segment_name}:", grid_obj.best_params_)

    # Get the R² score
    train_score = best_regressor.score(X_train_segment, y_train_segment)
    test_score = best_regressor.score(X_test_segment, y_test_segment)

    print(f"Training R² Score for {segment_name}:", train_score)
    print(f"Test R² Score for {segment_name}:", test_score)


--- Hyperparameter Tuning for Random Forest Regressor (Baby Users) ---
Fitting 5 folds for each of 162 candidates, totalling 810 fits
Best Parameters for Baby Users: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Training R² Score for Baby Users: 0.9348171330417856
Test R² Score for Baby Users: 0.5487145073722746

--- Hyperparameter Tuning for Random Forest Regressor (Non-Baby Users) ---
Fitting 5 folds for each of 162 candidates, totalling 810 fits
Best Parameters for Non-Baby Users: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Training R² Score for Non-Baby Users: 0.9469445470904339
Test R² Score for Non-Baby Users: 0.6703950226605853

In [ ]:
train_score = best_regressor.score(X_train_segment, y_train_segment)
print("Training R² Score for non-baby dataset:", train_score)

# Get the R² score on the test set
test_score = best_regressor.score(X_test_segment, y_test_segment)
print("Test R² Score:", test_score)

Training R² Score: 0.9469445470904339
Test R² Score: 0.6703950226605853

**Feature Importance**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from google.colab import files # Import files for downloading

# --- Start: Ensure data is loaded and preprocessed for standalone execution ---
# This block ensures 'data' and derived dataframes are properly initialized
# assuming the excel file is in the specified Google Drive path.
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    raise # Exit if data file is not found

# Apply initial preprocessing steps (standardize brand, calculate rating metrics, log-transform counts and prices)
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct
data['TotalCount_log'] = np.log1p(data['TotalCount'])
data['CurrentPrice_log'] = np.log1p(data['CurrentPrice'])
data['Calculated Price/Count or Price/Bundle_log'] = np.log1p(data['Calculated Price/Count or Price/Bundle'])

# Create segmented dataframes from the fully processed 'data'
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

# Define the base columns to drop from feature set (X) for consistency across all segments
columns_to_drop_from_X_base = [
    'ASIN', 'ProductTitle',
    'Count', 'Pack', 'TotalCount',
    'CurrentPrice', 'Calculated Price/Count or Price/Bundle',
    "Gissel's L52W $", "Gissel's YAG $", "Gissel's Units",
    'Rating_x_RatingCount', 'Rating_Count Weight',
    'Rating_x_RatingCount_log',
    'Rating_Count_Weight_log',
    'CurrentPrice_log',
    'Calculated Price/Count or Price/Bundle_log' # This is our target variable 'y'
]

def prepare_X_y_for_segment(df_segment, include_brand=True):
    """
    Prepares features (X) and target (y) for a given DataFrame segment,
    including dropping specified columns, handling NaNs, and one-hot encoding.
    """
    y_segment = df_segment['Calculated Price/Count or Price/Bundle_log']

    X_raw_segment = df_segment.drop(columns=columns_to_drop_from_X_base, errors='ignore')

    # Handle NaN values in 'XL' and 'Flushable' before one-hot encoding
    X_raw_segment['XL'] = X_raw_segment['XL'].fillna('Non-XL')
    X_raw_segment['Flushable'] = X_raw_segment['Flushable'].fillna('Non-Flushable')

    categorical_cols_segment = X_raw_segment.select_dtypes(include=['object']).columns.tolist()

    if not include_brand and 'CLEANED BRAND' in categorical_cols_segment:
        categorical_cols_segment.remove('CLEANED BRAND')
        X_raw_segment = X_raw_segment.drop(columns=['CLEANED BRAND'], errors='ignore')

    X_segment_encoded = pd.get_dummies(X_raw_segment, columns=categorical_cols_segment, drop_first=True)

    # Ensure 'TotalCount_log' is included if it was not dropped and is numeric
    if 'TotalCount_log' not in X_segment_encoded.columns and 'TotalCount_log' in df_segment.columns:
        X_segment_encoded['TotalCount_log'] = df_segment['TotalCount_log']

    # Ensure 'Rating' is present
    if 'Rating' not in X_segment_encoded.columns and 'Rating' in df_segment.columns:
        X_segment_encoded['Rating'] = df_segment['Rating']

    # Ensure 'RatingCount' is present
    if 'RatingCount' not in X_segment_encoded.columns and 'RatingCount' in df_segment.columns:
        X_segment_encoded['RatingCount'] = df_segment['RatingCount']

    return X_segment_encoded, y_segment

# Define the optimal hyperparameters obtained from GridSearchCV for Random Forest
# (from previous `3kQbBuS6WBuA` output for Non-Baby Users: `{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}`)
optimal_rf_params = {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50, 'random_state': 42}

def analyze_and_save_feature_importances(df_group, group_name, include_brand_factor):
    """
    Processes a user segment, trains a Random Forest model, and reports feature importances.
    Generates feature importances based on the include_brand_factor flag.
    """
    brand_suffix = "with_brand" if include_brand_factor else "without_brand"
    print(f"\n--- Feature Importances for {group_name} ({brand_suffix.replace('_', ' ').title()}) ---")

    if df_group.empty:
        print(f"No data available for {group_name}. Skipping feature importance calculation.")
        return

    # Prepare X and y for the current segment and brand inclusion setting
    X_group_processed, y_group = prepare_X_y_for_segment(df_group, include_brand=include_brand_factor)

    # Establish a reference set of columns for alignment for the current brand inclusion scenario.
    # This ensures consistency even if a segment lacks certain features present in others.
    X_all_scenario, _ = prepare_X_y_for_segment(data.copy(), include_brand=include_brand_factor)
    global_X_columns_current_scenario = X_all_scenario.columns

    # Align columns of X_group with the global X columns for the current scenario
    missing_cols = set(global_X_columns_current_scenario) - set(X_group_processed.columns)
    for c in missing_cols:
        X_group_processed[c] = 0 # Add missing columns with 0

    extra_cols = set(X_group_processed.columns) - set(global_X_columns_current_scenario)
    X_group_processed = X_group_processed.drop(columns=list(extra_cols))

    X_group = X_group_processed[global_X_columns_current_scenario] # Ensure order of columns is the same

    # Skip if not enough samples to split (e.g., if a segment becomes too small after filtering)
    if len(X_group) < 2:
        print(f"Not enough samples in {group_name} ({brand_suffix}) to perform train_test_split. Skipping model training.")
        return

    # Split data for the current group - only need X_train and y_train for fitting
    X_train_group, _, y_train_group, _ = train_test_split(X_group, y_group, test_size=0.2, random_state=42)

    # Initialize and fit Random Forest Regressor with optimal parameters
    best_regressor = RandomForestRegressor(**optimal_rf_params)
    best_regressor.fit(X_train_group, y_train_group)

    # Get feature importances
    feature_importances_group = best_regressor.feature_importances_

    # Create a DataFrame to display feature names with their importance scores
    feature_importance_df_group = pd.DataFrame({
        'Feature': X_train_group.columns,
        'Importance': feature_importances_group
    }).sort_values(by='Importance', ascending=False)

    # Display the sorted feature importances
    print(f"Top 50 Feature Importances of tuned Random Forest for {group_name} ({brand_suffix.replace('_', ' ').title()}):")
    print(feature_importance_df_group[:50].to_string()) # Use to_string() for full display in notebook output

    # Save to CSV and provide download link
    csv_filename_segment = f'random_forest_feature_importances_{group_name.replace(" ", "_").lower()}_{brand_suffix.lower()}.csv'
    feature_importance_df_group.to_csv(csv_filename_segment, index=False)
    files.download(csv_filename_segment)
    print(f"'{csv_filename_segment}' has been generated and is ready for download.")

# --- Process each user group for both scenarios (with and without brand factor) ---
user_groups_data = {
    "Baby Users": baby_wipe_data,
    "Non-Baby Users": other_wipe_data
}

for group_name, df_group in user_groups_data.items():
    # Scenario 1: Include CLEANED BRAND factor
    analyze_and_save_feature_importances(df_group, group_name, include_brand_factor=True)
    # Scenario 2: Exclude CLEANED BRAND factor
    analyze_and_save_feature_importances(df_group, group_name, include_brand_factor=False)

In [ ]:
print(feature_importance_df_group[:26])

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Define the KNN model
knn = KNeighborsRegressor()

# Fit the model
knn.fit(X_train, y_train)

# Make predictions
y_train_pred = knn.predict(X_train)
y_test_pred = knn.predict(X_test)

# Evaluate the model
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("KNN Training R² Score:", train_r2)
print("KNN Test R² Score:", test_r2)

Using KNN model, the test R2 score didn't increase.
KNN Training R² Score: 0.6463926552937909
KNN Test R² Score: 0.3799027422295014

**1. Comparison of various techniques and their relative performance based on chosen Metric (Measure of success)**:
- How do different techniques perform? Which one is performing relatively better? Is there scope to improve the performance further?
Based on the comparison of linear regression,Ridge / Lasso Regression, decision trees, random forest and K-Nearest Neighbors, the fine-tuned random forecast perfomed the best with the highest Training R² Score: 0.9469445470904339 and Test R² Score: 0.6703950226605853, which means random forest model can explain about 67% of the variance with the minimum error rate. There are other scope to improve the performance, such as simplifying the features and reduce the variances in the features to avoid overfitting.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

# Re-import models to ensure they are available
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV # Needed for dtree_tuned and best_regressor

def get_model_score(model, flag = True):
    '''
    model : regressor to predict values of X

    '''
    # Defining an empty list to store train and test results
    score_list = []

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    # If flag is True, convert predictions and actuals back to original scale
    if flag:
        pred_train_transformed = np.expm1(pred_train) # Inverse of log1p
        pred_test_transformed = np.expm1(pred_test) # Inverse of log1p
        y_train_transformed = np.expm1(y_train) # Inverse of log1p for actuals
        y_test_transformed = np.expm1(y_test) # Inverse of log1p for actuals
    else:
        pred_train_transformed = pred_train
        pred_test_transformed = pred_test
        y_train_transformed = y_train
        y_test_transformed = y_test

    train_r2 = r2_score(y_train_transformed, pred_train_transformed)
    test_r2 = r2_score(y_test_transformed, pred_test_transformed)
    train_rmse = np.sqrt(mean_squared_error(y_train_transformed, pred_train_transformed))
    test_rmse = np.sqrt(mean_squared_error(y_test_transformed, pred_test_transformed))

    # Adding all scores in the list
    score_list.extend((train_r2, test_r2, train_rmse, test_rmse))

    # If the flag is set to True then only the following print statements will be dispayed, the default value is True
    if flag == True:

        print(f"R-square on training set for {type(model).__name__}: {train_r2}")

        print(f"R-square on test set for {type(model).__name__}: {test_r2}")

        print(f"RMSE on training set for {type(model).__name__}: {train_rmse}")

        print(f"RMSE on test set for {type(model).__name__}: {test_rmse}")

    # Returning the list with train and test scores
    return score_list

# --- Re-train all models to ensure feature consistency ---
print("Re-training models for consistent feature sets...")

# 1. Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# 2. Ridge Regression
ridge = Ridge(alpha=1.0) # Using alpha=1.0 as in previous cells
ridge.fit(X_train, y_train)

# 3. Decision Tree (tuned - using previously found optimal parameters)
# Assuming optimal_params for dtree_tuned are {'max_depth': None, 'min_samples_split': 10, 'random_state': 1}
# Based on previous output in cell `Ye_rR-vkQ9Nm` and `JwVvXNlPSPvZ`
optimal_dtree_params = {'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 1} # Adjusted to be more general from previous results
# If a specific optimal parameter set for dtree_tuned was found, use it directly.
# If `dtree_tuned` is expected to be the global variable from `Ye_rR-vkQ9Nm`, ensure it's defined.
# For safety, re-run tuning or define it directly if the tuning cell wasn't run recently.
# Given the context, `dtree_tuned` refers to the best_estimator from the GridSearchCV in `Ye_rR-vkQ9Nm`.
# Let's ensure it's recreated with its best params for consistency.
# From `veXDypfnNLf4`, best parameters for `dtree_tuned` were implicit, assuming it's `grid_obj.best_estimator_`
# Let's use a reasonable set if not explicitly captured. Previous best was `max_depth=None, min_samples_split=10, max_features='sqrt'`
# The best parameters for dtree_tuned were `{'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt'}`
# Let's directly create dtree_tuned with these properties.
dtree_tuned = DecisionTreeRegressor(max_depth=None, min_samples_split=10, min_samples_leaf=1, max_features='sqrt', random_state=1)
dtree_tuned.fit(X_train, y_train)

# 4. K-Nearest Neighbors
knn = KNeighborsRegressor() # No tuning applied in previous steps
knn.fit(X_train, y_train)

# 5. Random Forest (tuned - using previously found optimal parameters for Non-Baby Users)
# The `best_regressor` variable was set as the result of GridSearchCV for the Non-Baby Users segment.
# We need to recreate it with its best parameters.
# From `3kQbBuS6WBuA` output for Non-Baby Users: `{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}`
best_regressor = RandomForestRegressor(max_depth=None, max_features='sqrt', min_samples_leaf=1, min_samples_split=2, n_estimators=50, random_state=42) # Added random_state for reproducibility
best_regressor.fit(X_train, y_train)

print("Models re-trained successfully. Proceeding with comparison.")

# Defining list of models you have trained, including fine-tuned ones
models = [lin_reg, dtree_tuned, ridge, knn, best_regressor]

# Defining empty lists to add train and test results
r2_train = []
r2_test = []
rmse_train = []
rmse_test = []

# Looping through all the models to get the rmse and r2 scores
# Using flag=True in get_model_score to transform predictions back to the original scale for evaluation
for model in models:
    # Accuracy score (on original price scale)
    j = get_model_score(model, True)

    r2_train.append(j[0])
    r2_test.append(j[1])
    rmse_train.append(j[2])
    rmse_test.append(j[3])

# Create a new comparison frame
comparison_frame = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Ridge Regression', 'K-Nearest Neighbors', 'Random Forest'],
    'Train_r2': r2_train,
    'Test_r2': r2_test,
    'Train_RMSE': rmse_train,
    'Test_RMSE': rmse_test
})

# Display updated results
print("\nModel Performance Comparison (on original price scale):")
print(comparison_frame)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# Re-import models to ensure they are available
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV # Needed for dtree_tuned and best_regressor

# --- Start: Ensure data is loaded and preprocessed for standalone execution ---
# This block ensures 'data' and derived dataframes are properly initialized
# assuming the excel file is in the specified Google Drive path.
try:
    data = pd.read_excel("/content/drive/MyDrive/Personal Care Project/Web Scraping_Wipes/Cleaned-Selected-Amazon-Wipes-Data-full-coded 06032026.xlsx")
except FileNotFoundError:
    print("Error: The data file was not found. Please ensure it is uploaded to the specified Google Drive path and the drive is mounted.")
    raise # Exit if data file is not found

# Apply initial preprocessing steps (standardize brand, calculate rating metrics, log-transform counts and prices)
data['CLEANED BRAND'] = data['CLEANED BRAND'].str.strip().str.upper()
data['Rating_x_RatingCount'] = data['Rating'] * data['RatingCount']
total_rating_count_sumproduct = data['Rating_x_RatingCount'].sum()
data['Rating_Count Weight'] = data['Rating_x_RatingCount'] / total_rating_count_sumproduct
data['TotalCount_log'] = np.log1p(data['TotalCount'])
data['CurrentPrice_log'] = np.log1p(data['CurrentPrice'])
data['Calculated Price/Count or Price/Bundle_log'] = np.log1p(data['Calculated Price/Count or Price/Bundle'])

# Define the base columns to drop from feature set (X) for consistency across all segments
columns_to_drop_from_X_base = [
    'ASIN', 'ProductTitle',
    'Count', 'Pack', 'TotalCount',
    'CurrentPrice', 'Calculated Price/Count or Price/Bundle',
    "Gissel's L52W $", "Gissel's YAG $", "Gissel's Units",
    'Rating_x_RatingCount', 'Rating_Count Weight',
    'Rating_x_RatingCount_log', # Not explicitly in X_raw initially, but good to include if it were
    'Rating_Count_Weight_log',  # Not explicitly in X_raw initially, but good to include if it were
    'CurrentPrice_log',
    'Calculated Price/Count or Price/Bundle_log' # This is our target variable 'y'
]

def prepare_X_y_for_segment(df_segment):
    """
    Prepares features (X) and target (y) for a given DataFrame segment,
    including dropping specified columns, handling NaNs, and one-hot encoding.
    """
    y_segment = df_segment['Calculated Price/Count or Price/Bundle_log']

    X_raw_segment = df_segment.drop(columns=columns_to_drop_from_X_base, errors='ignore')

    # Handle NaN values in 'XL' and 'Flushable' before one-hot encoding
    X_raw_segment['XL'] = X_raw_segment['XL'].fillna('Non-XL')
    X_raw_segment['Flushable'] = X_raw_segment['Flushable'].fillna('Non-Flushable')

    categorical_cols_segment = X_raw_segment.select_dtypes(include=['object']).columns.tolist()
    X_segment_encoded = pd.get_dummies(X_raw_segment, columns=categorical_cols_segment, drop_first=True)

    # Ensure 'TotalCount_log' is included if it was not dropped and is numeric
    if 'TotalCount_log' not in X_segment_encoded.columns and 'TotalCount_log' in df_segment.columns:
        X_segment_encoded['TotalCount_log'] = df_segment['TotalCount_log']

    # Ensure 'Rating' is present
    if 'Rating' not in X_segment_encoded.columns and 'Rating' in df_segment.columns:
        X_segment_encoded['Rating'] = df_segment['Rating']

    # Ensure 'RatingCount' is present
    if 'RatingCount' not in X_segment_encoded.columns and 'RatingCount' in df_segment.columns:
        X_segment_encoded['RatingCount'] = df_segment['RatingCount']

    return X_segment_encoded, y_segment

# Prepare the 'All Users' data first to establish a reference set of columns for X
X_all_processed, y_all_processed = prepare_X_y_for_segment(data)
global_X_columns = X_all_processed.columns
# --- End: Ensure data is loaded and preprocessed for standalone execution ---

def get_model_score(model, X_train_seg, X_test_seg, y_train_seg, y_test_seg, flag = True):
    '''
    model : regressor to predict values of X
    X_train_seg, X_test_seg, y_train_seg, y_test_seg: Segmented train/test data

    '''
    # Defining an empty list to store train and test results
    score_list = []

    pred_train = model.predict(X_train_seg)
    pred_test = model.predict(X_test_seg)

    # If flag is True, convert predictions and actuals back to original scale
    if flag:
        pred_train_transformed = np.expm1(pred_train) # Inverse of log1p
        pred_test_transformed = np.expm1(pred_test) # Inverse of log1p
        y_train_transformed = np.expm1(y_train_seg) # Inverse of log1p for actuals
        y_test_transformed = np.expm1(y_test_seg) # Inverse of log1p for actuals
    else:
        pred_train_transformed = pred_train
        pred_test_transformed = pred_test
        y_train_transformed = y_train_seg
        y_test_transformed = y_test_seg

    train_r2 = r2_score(y_train_transformed, pred_train_transformed)
    test_r2 = r2_score(y_test_transformed, pred_test_transformed)
    train_rmse = np.sqrt(mean_squared_error(y_train_transformed, pred_train_transformed))
    test_rmse = np.sqrt(mean_squared_error(y_test_transformed, pred_test_transformed))

    # Adding all scores in the list
    score_list.extend((train_r2, test_r2, train_rmse, test_rmse))

    # If the flag is set to True then only the following print statements will be dispayed, the default value is True
    if flag == True:

        print(f"R-square on training set for {type(model).__name__}: {train_r2}")

        print(f"R-square on test set for {type(model).__name__}: {test_r2}")

        print(f"RMSE on training set for {type(model).__name__}: {train_rmse}")

        print(f"RMSE on test set for {type(model).__name__}: {test_rmse}")

    # Returning the list with train and test scores
    return score_list


# Create segmented dataframes from the fully processed 'data'
baby_wipe_data = data[data['User'] == 'Baby'].copy()
other_wipe_data = data[data['User'] != 'Baby'].copy()

user_segments_data = {
    "All Users": data,
    "Baby Users": baby_wipe_data,
    "Non-Baby Users": other_wipe_data
}

# Optimal parameters for Decision Tree (from cell `Ye_rR-vkQ9Nm` and `JwVvXNlPSPvZ` for the overall dataset)
optimal_dtree_params = {'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 1}

# Optimal parameters for Random Forest (from cell `3kQbBuS6WBuA` output for Non-Baby Users segment)
optimal_rf_params = {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50, 'random_state': 42}

for segment_name, segment_df_raw in user_segments_data.items():
    print(f"\n--- Training and Evaluating Models for {segment_name} ---")

    if segment_df_raw.empty:
        print(f"No data available for {segment_name}. Skipping model training.")
        continue

    # Prepare X and y for the current segment
    X_segment_processed, y_segment_processed = prepare_X_y_for_segment(segment_df_raw)

    # Align columns of X_segment with the global X columns
    missing_cols = set(global_X_columns) - set(X_segment_processed.columns)
    for c in missing_cols:
        X_segment_processed[c] = 0 # Add missing columns with 0

    extra_cols = set(X_segment_processed.columns) - set(global_X_columns)
    X_segment_processed = X_segment_processed.drop(columns=list(extra_cols))

    X_segment_processed = X_segment_processed[global_X_columns] # Ensure order of columns is the same

    # Check if enough samples after preprocessing and alignment for train_test_split
    if len(X_segment_processed) < 2: # Need at least 2 samples to split
        print(f"Not enough samples in {segment_name} to perform train_test_split. Skipping model training.")
        continue

    # Split data for the current segment
    X_train_seg, X_test_seg, y_train_seg, y_test_seg = train_test_split(X_segment_processed, y_segment_processed, test_size=0.2, random_state=42)

    # --- Initialize and Train Models for the Current Segment ---
    models = []

    # 1. Linear Regression
    lin_reg_seg = LinearRegression()
    lin_reg_seg.fit(X_train_seg, y_train_seg)
    models.append(('Linear Regression', lin_reg_seg))

    # 2. Ridge Regression
    ridge_seg = Ridge(alpha=1.0) # Using alpha=1.0 as in previous cells
    ridge_seg.fit(X_train_seg, y_train_seg)
    models.append(('Ridge Regression', ridge_seg))

    # 3. Tuned Decision Tree
    dtree_tuned_seg = DecisionTreeRegressor(**optimal_dtree_params)
    dtree_tuned_seg.fit(X_train_seg, y_train_seg)
    models.append(('Tuned Decision Tree', dtree_tuned_seg))

    # 4. K-Nearest Neighbors
    knn_seg = KNeighborsRegressor() # No tuning applied in previous steps
    knn_seg.fit(X_train_seg, y_train_seg)
    models.append(('K-Nearest Neighbors', knn_seg))

    # 5. Tuned Random Forest
    rf_tuned_seg = RandomForestRegressor(**optimal_rf_params)
    rf_tuned_seg.fit(X_train_seg, y_train_seg)
    models.append(('Tuned Random Forest', rf_tuned_seg))

    # Defining empty lists to add train and test results for the current segment
    r2_train_seg = []
    r2_test_seg = []
    rmse_train_seg = []
    rmse_test_seg = []

    # Looping through all the models to get the rmse and r2 scores
    for model_name, model in models:
        # Accuracy score (on original price scale)
        j = get_model_score(model, X_train_seg, X_test_seg, y_train_seg, y_test_seg, True)

        r2_train_seg.append(j[0])
        r2_test_seg.append(j[1])
        rmse_train_seg.append(j[2])
        rmse_test_seg.append(j[3])

    # Create a new comparison frame for the current segment
    comparison_frame_seg = pd.DataFrame({
        'Model': [m[0] for m in models],
        'Train_r2': r2_train_seg,
        'Test_r2': r2_test_seg,
        'Train_RMSE': rmse_train_seg,
        'Test_RMSE': rmse_test_seg
    })

    # Display updated results
    print(f"\nModel Performance Comparison for {segment_name} (on original price scale):")
    print(comparison_frame_seg)




Model Performance Comparison for All Users (on original price scale):
                 Model  Train_r2   Test_r2  Train_RMSE  Test_RMSE
0    Linear Regression  0.453793  0.278212    0.650338   0.460648
1     Ridge Regression  0.403493  0.360161    0.679623   0.433710
2  Tuned Decision Tree  0.753371  0.493140    0.437001   0.386018
3  K-Nearest Neighbors  0.353334  0.228419    0.707620   0.476272
4  Tuned Random Forest  0.764792  0.449468    0.426762   0.402305

Model Performance Comparison for Baby Users (on original price scale):
                 Model  Train_r2   Test_r2  Train_RMSE  Test_RMSE
0    Linear Regression  0.857717  0.232589    0.059679   0.133850
1     Ridge Regression  0.789033  0.406169    0.072670   0.117743
2  Tuned Decision Tree  0.702217  0.533303    0.086337   0.104381
3  K-Nearest Neighbors  0.399114  0.151363    0.122643   0.140756
4  Tuned Random Forest  0.904464  0.469566    0.048903   0.111281




Model Performance Comparison for Non-Baby Users (on original price scale):
                 Model  Train_r2   Test_r2  Train_RMSE  Test_RMSE
0    Linear Regression  0.602970  0.225065    0.442200   1.241110
1     Ridge Regression  0.521807  0.207377    0.485299   1.255195
2  Tuned Decision Tree  0.540563  0.296906    0.475686   1.182182
3  K-Nearest Neighbors  0.437354  0.239249    0.526411   1.229699
4  Tuned Random Forest  0.785817  0.197064    0.324788   1.263334